In [ ]:
# ============ CELDA 39 · PARTE 2 (1 de 2): TIEMPO, COMPETENCIA Y REDONDEO ============
# (1) descomposicion del cambio en oferente unico: competencia real vs recomposicion de modalidades
# (2) concentracion por año  (3) estacionalidad y efecto diciembre
# (4) numeros redondos  (5) montos modales: deteccion de topes legales y anclas
import os
import pandas as pd, duckdb



con.sql("CREATE OR REPLACE VIEW pm  AS SELECT * FROM read_parquet('plan_A/planA_procesos_modelo.parquet')")
con.sql("CREATE OR REPLACE VIEW ind AS SELECT * FROM read_parquet('plan_A/planA_indicadores.parquet')")
# --- todos los nombres de cada RUC, y el canonico por fecha real ----------
nombres = con.execute("""
WITH n AS (
  SELECT entidad_ruc, entidad_nombre,
         max(fecha_inicio_tender) AS ult, count(*) AS proc
  FROM pm GROUP BY 1, 2
)
SELECT entidad_ruc, entidad_nombre, ult, proc,
       row_number() OVER (PARTITION BY entidad_ruc ORDER BY ult DESC, proc DESC) AS rn
FROM n
""").df()

# ---------- (1) DESCOMPOSICION DEL OFERENTE UNICO ----------
t('(1) EL OFERENTE UNICO BAJA: cuanto es competencia y cuanto es recomposicion')
D = con.sql("""SELECT familia, anio, count(*) AS n,
     count(*) FILTER (WHERE ind_oferente_unico) AS unicos
   FROM ind WHERE anio BETWEEN 2015 AND 2024 GROUP BY 1,2""").df()
pn = D.pivot(index='familia', columns='anio', values='n').fillna(0)
pu = D.assign(r=D.unicos / D.n).pivot(index='familia', columns='anio', values='r')
t0, t1 = 2015, 2024
s0, s1 = pn[t0] / pn[t0].sum(), pn[t1] / pn[t1].sum()
u0, u1 = pu[t0].fillna(0), pu[t1].fillna(0)
U0, U1 = float((s0 * u0).sum()), float((s1 * u1).sum())
comp = float(((s1 - s0) * u0).sum()); dentro = float((s0 * (u1 - u0)).sum())
inter = float(((s1 - s0) * (u1 - u0)).sum())
print(f'oferente unico en {t0}: {100*U0:.2f}%   en {t1}: {100*U1:.2f}%   cambio: {100*(U1-U0):+.2f} pp\n')
print(f'  por recomposicion de modalidades ...... {100*comp:+.2f} pp  ({100*comp/(U1-U0):5.1f}% del cambio)')
print(f'  por cambio dentro de cada modalidad ... {100*dentro:+.2f} pp  ({100*dentro/(U1-U0):5.1f}% del cambio)')
print(f'  interaccion ........................... {100*inter:+.2f} pp  ({100*inter/(U1-U0):5.1f}% del cambio)')
print(f'  suma de control ....................... {100*(comp+dentro+inter):+.2f} pp')
SS = pd.DataFrame({'peso_2015': (100*s0).round(2), 'peso_2024': (100*s1).round(2),
                   'unico_2015_pct': (100*u0).round(2), 'unico_2024_pct': (100*u1).round(2),
                   'aporte_composicion_pp': (100*(s1-s0)*u0).round(3),
                   'aporte_interno_pp': (100*s0*(u1-u0)).round(3)}).reset_index()
print('\n' + g(SS.sort_values('aporte_composicion_pp'), 'descomposicion_oferente_unico').to_string(index=False))

# ---------- (2) CONCENTRACION POR AÑO ----------
t('(2) CONCENTRACION POR AÑO (solo entidades con 5 o mas procesos en el año)')
print(g(con.sql("""WITH s AS (SELECT anio_proceso AS anio, entidad_ruc, proveedor_ruc_final,
         sum(monto_final) AS m FROM pm GROUP BY 1,2,3),
     e AS (SELECT anio, entidad_ruc, sum(m) AS total, count(*) AS prov FROM s GROUP BY 1,2),
     p AS (SELECT anio, entidad_ruc, count(*) AS procesos FROM pm GROUP BY 1,2),
     h AS (SELECT s.anio, s.entidad_ruc, sum(power(s.m/nullif(e.total,0),2)) AS hhi, e.prov, e.total
           FROM s JOIN e USING (anio, entidad_ruc) GROUP BY 1,2,4,5)
   SELECT h.anio, count(*) AS entidades,
     round(median(h.hhi),4) AS hhi_mediano,
     round(sum(h.hhi*h.total)/sum(h.total),4) AS hhi_ponderado_por_monto,
     round(median(h.prov),0) AS mediana_proveedores_por_entidad,
     count(*) FILTER (WHERE h.hhi > 0.9999) AS entidades_con_un_solo_proveedor
   FROM h JOIN p USING (anio, entidad_ruc) WHERE p.procesos >= 5
   GROUP BY 1 ORDER BY 1""").df(), 'concentracion_por_anio').to_string(index=False))
print('\nSi el HHI mediano baja mientras el ponderado por monto se mantiene, la desconcentracion')
print('ocurre en entidades chicas y no donde esta la plata.')

# ---------- (3) ESTACIONALIDAD ----------
t('(3) ESTACIONALIDAD: en que mes se contrata y como')
print(g(con.sql("""SELECT month(fecha_proceso) AS mes, count(*) AS procesos,
     round(100.0*count(*)/(SELECT count(*) FROM ind WHERE anio BETWEEN 2015 AND 2024),2) AS pct_procesos,
     round(sum(monto_final)/1e6,1) AS millones,
     round(100.0*sum(monto_final)/(SELECT sum(monto_final) FROM ind WHERE anio BETWEEN 2015 AND 2024),2) AS pct_monto,
     round(median(monto_final),0) AS mediana_usd,
     round(100.0*count(*) FILTER (WHERE ind_oferente_unico)/count(*),2) AS pct_oferente_unico,
     round(median(ind_n_oferentes),0) AS mediana_oferentes
   FROM ind WHERE anio BETWEEN 2015 AND 2024 GROUP BY 1 ORDER BY 1""").df(),
   'estacionalidad_mensual').to_string(index=False))
print('\n-- diciembre contra el resto del año --')
print(con.sql("""SELECT CASE WHEN month(fecha_proceso)=12 THEN 'diciembre' ELSE 'enero a noviembre' END AS periodo,
     count(*) AS procesos, round(sum(monto_final)/1e6,1) AS millones,
     round(median(monto_final),0) AS mediana_usd,
     round(100.0*count(*) FILTER (WHERE ind_oferente_unico)/count(*),2) AS pct_oferente_unico,
     round(100.0*count(*) FILTER (WHERE ind_fraccionamiento_marca)/count(*),2) AS pct_con_proceso_previo_30d,
     round(median(ind_plazo_ofertas_dias),0) AS mediana_plazo_ofertas
   FROM ind WHERE anio BETWEEN 2015 AND 2024 GROUP BY 1 ORDER BY 1""").df().to_string(index=False))
print('\n-- la ultima quincena del año --')
print(con.sql("""SELECT CASE WHEN month(fecha_proceso)=12 AND day(fecha_proceso)>=16 THEN 'del 16 al 31 de diciembre'
                             WHEN month(fecha_proceso)=12 THEN 'del 1 al 15 de diciembre'
                             ELSE 'resto del año' END AS tramo,
     count(*) AS procesos, round(100.0*count(*)/(SELECT count(*) FROM ind WHERE anio BETWEEN 2015 AND 2024),2) AS pct,
     round(sum(monto_final)/1e6,1) AS millones, round(median(monto_final),0) AS mediana_usd,
     round(100.0*count(*) FILTER (WHERE ind_oferente_unico)/count(*),2) AS pct_oferente_unico
   FROM ind WHERE anio BETWEEN 2015 AND 2024 GROUP BY 1 ORDER BY 1""").df().to_string(index=False))

# ---------- (4) NUMEROS REDONDOS ----------
t('(4) NUMEROS REDONDOS EN EL MONTO ADJUDICADO')
RED = """CAST(round(monto_final*100) AS BIGINT) AS cent"""
print(con.sql(f"""WITH b AS (SELECT monto_final, familia, ind_oferente_unico, anio, {RED} FROM ind)
   SELECT count(*) AS procesos,
     round(100.0*count(*) FILTER (WHERE cent % 100 = 0)/count(*),2)      AS pct_sin_centavos,
     round(100.0*count(*) FILTER (WHERE cent % 100000 = 0)/count(*),2)   AS pct_multiplo_1000,
     round(100.0*count(*) FILTER (WHERE cent % 1000000 = 0)/count(*),2)  AS pct_multiplo_10000,
     round(100.0*count(*) FILTER (WHERE cent % 10000000 = 0)/count(*),2) AS pct_multiplo_100000
   FROM b""").df().to_string(index=False))
print('\n-- por modalidad --')
print(g(con.sql(f"""WITH b AS (SELECT monto_final, familia, {RED} FROM ind)
   SELECT familia, count(*) AS procesos,
     round(100.0*count(*) FILTER (WHERE cent % 100 = 0)/count(*),2)     AS pct_sin_centavos,
     round(100.0*count(*) FILTER (WHERE cent % 100000 = 0)/count(*),2)  AS pct_multiplo_1000,
     round(100.0*count(*) FILTER (WHERE cent % 1000000 = 0)/count(*),2) AS pct_multiplo_10000
   FROM b GROUP BY 1 ORDER BY pct_multiplo_1000 DESC""").df(),
   'numeros_redondos_por_familia').to_string(index=False))
print('\n-- redondeo segun si hubo o no competencia (dentro de cada modalidad competitiva) --')
print(con.sql(f"""WITH b AS (SELECT familia, ind_oferente_unico, {RED} FROM ind
     WHERE familia NOT IN ('Contratacion directa','Regimen especial'))
   SELECT familia, ind_oferente_unico AS un_solo_oferente, count(*) AS procesos,
     round(100.0*count(*) FILTER (WHERE cent % 100000 = 0)/count(*),2)  AS pct_multiplo_1000,
     round(100.0*count(*) FILTER (WHERE cent % 1000000 = 0)/count(*),2) AS pct_multiplo_10000
   FROM b GROUP BY 1,2 ORDER BY familia, un_solo_oferente""").df().to_string(index=False))

# ---------- (5) MONTOS MODALES: TOPES Y ANCLAS ----------
t('(5) LOS 8 MONTOS MAS REPETIDOS DE CADA MODALIDAD (asi aparecen los topes legales)')
MO = con.sql("""WITH c AS (SELECT familia, monto_final, count(*) AS veces,
       row_number() OVER (PARTITION BY familia ORDER BY count(*) DESC) AS puesto
     FROM ind GROUP BY 1,2)
   SELECT familia, puesto, monto_final AS monto_usd, veces,
     round(100.0*veces/sum(veces) OVER (PARTITION BY familia),3) AS pct_de_la_familia
   FROM c WHERE puesto <= 8 ORDER BY familia, puesto""").df()
print(g(MO, 'montos_modales_por_familia').to_string(index=False))
print('\nUn monto que se repite miles de veces en una modalidad suele ser un tope legal o un precio')
print('de catalogo. Si el tope aparece como monto modal, hay acumulacion justo en el limite.')
print('\n-- cuanta masa hay pegada al techo de cada modalidad y año (2015-2024) --')
print(g(con.sql("""WITH q AS (SELECT familia, anio, quantile_cont(monto_final, 0.99) AS p99
       FROM ind WHERE anio BETWEEN 2015 AND 2024 GROUP BY 1,2)
   SELECT i.familia, count(*) AS procesos,
     round(100.0*count(*) FILTER (WHERE i.monto_final BETWEEN 0.98*q.p99 AND q.p99)/count(*),3) AS pct_en_el_2pct_bajo_el_p99,
     round(100.0*count(*) FILTER (WHERE i.monto_final BETWEEN 0.90*q.p99 AND q.p99)/count(*),3) AS pct_en_el_10pct_bajo_el_p99
   FROM ind i JOIN q USING (familia, anio) WHERE i.anio BETWEEN 2015 AND 2024
   GROUP BY 1 ORDER BY pct_en_el_2pct_bajo_el_p99 DESC""").df(),
   'masa_bajo_el_techo').to_string(index=False))
print('\nSi la distribucion fuera suave, el 2% inferior al p99 deberia contener cerca del 2% de los')
print('casos de ese tramo. Valores muy por encima indican acumulacion deliberada bajo el limite.')
print('\nPega aqui toda la salida.')

In [ ]:
# ============ CELDA 39b · PARTES (2) A (5) CORREGIDAS ============
# Continua la celda 39. Reutiliza las vistas pm e ind que ya quedaron creadas.
import os
import pandas as pd, duckdb
os.chdir(r'C:\SERCOP_FINAL')
EST = 'plan_A/documentacion/estadisticas'; os.makedirs(EST, exist_ok=True)
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 80); pd.set_option('display.max_rows', 200)
try: con
except NameError: con = duckdb.connect()
con.sql("SET preserve_insertion_order=false")
con.sql("CREATE OR REPLACE VIEW pm  AS SELECT * FROM read_parquet('plan_A/planA_procesos_modelo.parquet')")
con.sql("CREATE OR REPLACE VIEW ind AS SELECT * FROM read_parquet('plan_A/planA_indicadores.parquet')")
def t(x): print('\n' + '=' * 95); print(x); print('=' * 95)
def g(df, nombre):
    df.to_csv(f'{EST}/{nombre}.csv', index=False, encoding='utf-8-sig'); return df

# ---------- (2) CONCENTRACION POR AÑO ----------
t('(2) CONCENTRACION POR AÑO (solo entidades con 5 o mas procesos en el año)')
print(g(con.sql("""WITH s AS (SELECT anio_proceso AS anio, entidad_ruc, proveedor_ruc_final,
           sum(monto_final) AS m FROM pm GROUP BY 1,2,3),
     e AS (SELECT anio, entidad_ruc, sum(m) AS total, count(*) AS prov FROM s GROUP BY 1,2),
     p AS (SELECT anio_proceso AS anio, entidad_ruc, count(*) AS procesos FROM pm GROUP BY 1,2),
     h AS (SELECT s.anio, s.entidad_ruc, sum(power(s.m/nullif(e.total,0),2)) AS hhi,
                  any_value(e.prov) AS prov, any_value(e.total) AS total
           FROM s JOIN e USING (anio, entidad_ruc) GROUP BY 1,2)
   SELECT h.anio, count(*) AS entidades,
     round(median(h.hhi),4) AS hhi_mediano,
     round(sum(h.hhi*h.total)/sum(h.total),4) AS hhi_ponderado_por_monto,
     round(median(h.prov),0) AS mediana_proveedores_por_entidad,
     count(*) FILTER (WHERE h.hhi > 0.9999) AS entidades_con_un_solo_proveedor
   FROM h JOIN p USING (anio, entidad_ruc) WHERE p.procesos >= 5
   GROUP BY 1 ORDER BY 1""").df(), 'concentracion_por_anio').to_string(index=False))
print('\nSi el HHI mediano baja mientras el ponderado por monto se mantiene, la desconcentracion')
print('ocurre en entidades chicas y no donde esta la plata.')

# ---------- (3) ESTACIONALIDAD ----------
t('(3) ESTACIONALIDAD: en que mes se contrata y como')
print(g(con.sql("""SELECT month(fecha_proceso) AS mes, count(*) AS procesos,
     round(100.0*count(*)/(SELECT count(*) FROM ind WHERE anio BETWEEN 2015 AND 2024),2) AS pct_procesos,
     round(sum(monto_final)/1e6,1) AS millones,
     round(100.0*sum(monto_final)/(SELECT sum(monto_final) FROM ind WHERE anio BETWEEN 2015 AND 2024),2) AS pct_monto,
     round(median(monto_final),0) AS mediana_usd,
     round(100.0*count(*) FILTER (WHERE ind_oferente_unico)/count(*),2) AS pct_oferente_unico,
     round(median(ind_n_oferentes),0) AS mediana_oferentes
   FROM ind WHERE anio BETWEEN 2015 AND 2024 GROUP BY 1 ORDER BY 1""").df(),
   'estacionalidad_mensual').to_string(index=False))
print('\n-- diciembre contra el resto del año --')
print(con.sql("""SELECT CASE WHEN month(fecha_proceso)=12 THEN 'diciembre' ELSE 'enero a noviembre' END AS periodo,
     count(*) AS procesos, round(sum(monto_final)/1e6,1) AS millones,
     round(median(monto_final),0) AS mediana_usd,
     round(100.0*count(*) FILTER (WHERE ind_oferente_unico)/count(*),2) AS pct_oferente_unico,
     round(100.0*count(*) FILTER (WHERE ind_fraccionamiento_marca)/count(*),2) AS pct_con_proceso_previo_30d,
     round(median(ind_plazo_ofertas_dias),0) AS mediana_plazo_ofertas
   FROM ind WHERE anio BETWEEN 2015 AND 2024 GROUP BY 1 ORDER BY 1""").df().to_string(index=False))
print('\n-- la ultima quincena del año --')
print(con.sql("""SELECT CASE WHEN month(fecha_proceso)=12 AND day(fecha_proceso)>=16 THEN 'c) 16 al 31 de diciembre'
                             WHEN month(fecha_proceso)=12 THEN 'b) 1 al 15 de diciembre'
                             ELSE 'a) resto del año' END AS tramo,
     count(*) AS procesos,
     round(100.0*count(*)/(SELECT count(*) FROM ind WHERE anio BETWEEN 2015 AND 2024),2) AS pct,
     round(sum(monto_final)/1e6,1) AS millones, round(median(monto_final),0) AS mediana_usd,
     round(100.0*count(*) FILTER (WHERE ind_oferente_unico)/count(*),2) AS pct_oferente_unico
   FROM ind WHERE anio BETWEEN 2015 AND 2024 GROUP BY 1 ORDER BY 1""").df().to_string(index=False))

# ---------- (4) NUMEROS REDONDOS ----------
t('(4) NUMEROS REDONDOS EN EL MONTO ADJUDICADO')
print(con.sql("""WITH b AS (SELECT CAST(round(monto_final*100) AS BIGINT) AS cent FROM ind)
   SELECT count(*) AS procesos,
     round(100.0*count(*) FILTER (WHERE cent % 100 = 0)/count(*),2)      AS pct_sin_centavos,
     round(100.0*count(*) FILTER (WHERE cent % 100000 = 0)/count(*),2)   AS pct_multiplo_1000,
     round(100.0*count(*) FILTER (WHERE cent % 1000000 = 0)/count(*),2)  AS pct_multiplo_10000,
     round(100.0*count(*) FILTER (WHERE cent % 10000000 = 0)/count(*),2) AS pct_multiplo_100000
   FROM b""").df().to_string(index=False))
print('\n-- por modalidad --')
print(g(con.sql("""WITH b AS (SELECT familia, CAST(round(monto_final*100) AS BIGINT) AS cent FROM ind)
   SELECT familia, count(*) AS procesos,
     round(100.0*count(*) FILTER (WHERE cent % 100 = 0)/count(*),2)     AS pct_sin_centavos,
     round(100.0*count(*) FILTER (WHERE cent % 100000 = 0)/count(*),2)  AS pct_multiplo_1000,
     round(100.0*count(*) FILTER (WHERE cent % 1000000 = 0)/count(*),2) AS pct_multiplo_10000
   FROM b GROUP BY 1 ORDER BY pct_multiplo_1000 DESC""").df(),
   'numeros_redondos_por_familia').to_string(index=False))
print('\n-- redondeo segun si hubo o no competencia, dentro de cada modalidad competitiva --')
print(g(con.sql("""WITH b AS (SELECT familia, ind_oferente_unico,
       CAST(round(monto_final*100) AS BIGINT) AS cent FROM ind
     WHERE familia NOT IN ('Contratacion directa','Regimen especial'))
   SELECT familia, ind_oferente_unico AS un_solo_oferente, count(*) AS procesos,
     round(100.0*count(*) FILTER (WHERE cent % 100 = 0)/count(*),2)     AS pct_sin_centavos,
     round(100.0*count(*) FILTER (WHERE cent % 100000 = 0)/count(*),2)  AS pct_multiplo_1000,
     round(100.0*count(*) FILTER (WHERE cent % 1000000 = 0)/count(*),2) AS pct_multiplo_10000
   FROM b GROUP BY 1,2 ORDER BY familia, un_solo_oferente""").df(),
   'numeros_redondos_por_competencia').to_string(index=False))

# ---------- (5) MONTOS MODALES ----------
t('(5) LOS 8 MONTOS MAS REPETIDOS DE CADA MODALIDAD (asi aparecen los topes legales)')
print(g(con.sql("""WITH c AS (SELECT familia, monto_final, count(*) AS veces,
         row_number() OVER (PARTITION BY familia ORDER BY count(*) DESC) AS puesto,
         sum(count(*)) OVER (PARTITION BY familia) AS total_familia
       FROM ind GROUP BY 1,2)
   SELECT familia, puesto, monto_final AS monto_usd, veces, total_familia,
     round(100.0*veces/total_familia,3) AS pct_de_la_familia
   FROM c WHERE puesto <= 8 ORDER BY familia, puesto""").df(),
   'montos_modales_por_familia').to_string(index=False))
print('\nUn monto que se repite miles de veces en una modalidad suele ser un tope legal o un precio')
print('de catalogo. Si el tope aparece como monto modal, hay acumulacion justo en el limite.')
print('\n-- cuanta masa hay pegada al techo de cada modalidad y año (2015-2024) --')
print(g(con.sql("""WITH q AS (SELECT familia, anio, quantile_cont(monto_final, 0.99) AS p99
       FROM ind WHERE anio BETWEEN 2015 AND 2024 GROUP BY 1,2)
   SELECT i.familia, count(*) AS procesos,
     round(100.0*count(*) FILTER (WHERE i.monto_final BETWEEN 0.98*q.p99 AND q.p99)/count(*),3) AS pct_en_el_2pct_bajo_el_p99,
     round(100.0*count(*) FILTER (WHERE i.monto_final BETWEEN 0.90*q.p99 AND q.p99)/count(*),3) AS pct_en_el_10pct_bajo_el_p99
   FROM ind i JOIN q USING (familia, anio) WHERE i.anio BETWEEN 2015 AND 2024
   GROUP BY 1 ORDER BY pct_en_el_2pct_bajo_el_p99 DESC""").df(),
   'masa_bajo_el_techo').to_string(index=False))
print('\nSi la distribucion fuera suave, el 2% inferior al p99 deberia contener cerca del 2% de los')
print('casos de ese tramo. Valores muy por encima indican acumulacion bajo el limite.')
print('\nPega aqui toda la salida.')

In [ ]:
# ============ CELDA 40 · PARTE 2 (2a): PLIEGO, ESTACIONALIDAD DESCOMPUESTA Y GEOGRAFIA ============
# (1) el monto adjudicado contra el presupuesto del pliego: el mecanismo detras del redondeo
# (2) descomposicion de enero, febrero y diciembre: composicion de modalidades vs comportamiento
# (3) geografia: provincia de la entidad y del proveedor, y contratacion fuera de provincia
import os
import pandas as pd, duckdb

os.chdir(r'C:\SERCOP_FINAL')
EST = 'plan_A/documentacion/estadisticas'; os.makedirs(EST, exist_ok=True)
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 80); pd.set_option('display.max_rows', 200)
con = duckdb.connect(); con.sql("SET preserve_insertion_order=false")
def t(x): print('\n' + '=' * 95); print(x); print('=' * 95)
def g(df, nombre):
    df.to_csv(f'{EST}/{nombre}.csv', index=False, encoding='utf-8-sig'); return df

con.sql("CREATE OR REPLACE VIEW pm  AS SELECT * FROM read_parquet('plan_A/planA_procesos_modelo.parquet')")
con.sql("CREATE OR REPLACE VIEW ind AS SELECT * FROM read_parquet('plan_A/planA_indicadores.parquet')")
# mesa unica: lo necesario de las dos tablas, con nombres sin ambiguedad
con.sql("""CREATE OR REPLACE TABLE w AS
  SELECT ind.ocid, ind.familia, ind.anio, ind.fecha_proceso, ind.monto_final,
         ind.ind_oferente_unico AS unico, ind.ind_n_oferentes AS oferentes,
         pm.monto_tender, pm.provincia_entidad, pm.provincia_proveedor,
         pm.proveedor_provincia_distinta_entidad AS fuera_de_provincia,
         pm.entidad_ruc, pm.proveedor_ruc_final AS proveedor_ruc,
         CAST(round(ind.monto_final*100) AS BIGINT) AS cent
  FROM ind JOIN pm USING (ocid)""")
print(con.sql("""SELECT count(*) AS filas,
     round(100.0*count(monto_tender)/count(*),2)        AS pct_con_monto_pliego,
     round(100.0*count(provincia_entidad)/count(*),2)   AS pct_con_provincia_entidad,
     round(100.0*count(provincia_proveedor)/count(*),2) AS pct_con_provincia_proveedor,
     round(100.0*count(fuera_de_provincia)/count(*),2)  AS pct_con_marca_fuera_de_provincia
   FROM w""").df().to_string(index=False))

# ---------- (1) EL MONTO CONTRA EL PLIEGO ----------
t('(1a) RELACION ENTRE EL MONTO ADJUDICADO Y EL PRESUPUESTO DEL PLIEGO')
print(con.sql("""SELECT count(*) AS con_pliego,
     count(*) FILTER (WHERE abs(monto_final - monto_tender) < 0.005) AS igual_al_pliego,
     round(100.0*count(*) FILTER (WHERE abs(monto_final - monto_tender) < 0.005)/count(*),2) AS pct_igual,
     round(min(monto_final/monto_tender),4)                   AS razon_minima,
     round(quantile_cont(monto_final/monto_tender,0.05),4)    AS p05,
     round(quantile_cont(monto_final/monto_tender,0.25),4)    AS p25,
     round(median(monto_final/monto_tender),4)                AS razon_mediana,
     round(quantile_cont(monto_final/monto_tender,0.75),4)    AS p75,
     round(max(monto_final/monto_tender),2)                   AS razon_maxima,
     count(*) FILTER (WHERE monto_final > monto_tender*1.0001) AS adjudicado_sobre_el_pliego
   FROM w WHERE monto_tender IS NOT NULL AND monto_tender > 0""").df().to_string(index=False))

t('(1b) POR MODALIDAD Y SEGUN CONCURRENCIA  (la prueba que explica el redondeo)')
print(g(con.sql("""SELECT familia, unico AS un_solo_oferente, count(*) AS con_pliego,
     round(100.0*count(*) FILTER (WHERE abs(monto_final - monto_tender) < 0.005)/count(*),2) AS pct_igual_al_pliego,
     round(median(monto_final/monto_tender),4) AS razon_mediana,
     round(100.0*count(*) FILTER (WHERE monto_final/monto_tender BETWEEN 0.9490 AND 0.9510)/count(*),2) AS pct_en_el_0_95,
     round(100.0*count(*) FILTER (WHERE cent % 100 = 0)/count(*),2) AS pct_sin_centavos
   FROM w WHERE monto_tender IS NOT NULL AND monto_tender > 0 AND unico IS NOT NULL
   GROUP BY 1,2 ORDER BY familia, un_solo_oferente""").df(),
   'monto_contra_pliego').to_string(index=False))
print('\nSi pct_igual_al_pliego sigue el mismo patron que pct_sin_centavos, el redondeo lo explica')
print('la coincidencia con el pliego y no la modalidad.')

t('(1c) EL REDONDEO SEGUN SI EL MONTO COINCIDE O NO CON EL PLIEGO')
print(g(con.sql("""SELECT familia,
     (abs(monto_final - monto_tender) < 0.005) AS monto_igual_al_pliego, count(*) AS procesos,
     round(100.0*count(*) FILTER (WHERE cent % 100 = 0)/count(*),2)    AS pct_sin_centavos,
     round(100.0*count(*) FILTER (WHERE cent % 100000 = 0)/count(*),2) AS pct_multiplo_1000
   FROM w WHERE monto_tender IS NOT NULL AND monto_tender > 0
   GROUP BY 1,2 ORDER BY familia, monto_igual_al_pliego""").df(),
   'redondeo_segun_coincidencia_con_pliego').to_string(index=False))

t('(1d) LAS 15 RAZONES monto/pliego MAS REPETIDAS EN SUBASTA INVERSA CON UN SOLO OFERENTE')
print(con.sql("""SELECT round(monto_final/monto_tender, 4) AS razon, count(*) AS procesos,
     round(100.0*count(*)/sum(count(*)) OVER (),2) AS pct
   FROM w WHERE familia = 'Subasta inversa electronica' AND unico
     AND monto_tender IS NOT NULL AND monto_tender > 0
   GROUP BY 1 ORDER BY procesos DESC LIMIT 15""").df().to_string(index=False))
print('\nSi la normativa obliga a una rebaja minima en la negociacion de oferta unica, la razon')
print('deberia acumularse en el valor correspondiente a esa rebaja.')
print('\n-- y en subasta inversa CON competencia, para contrastar --')
print(con.sql("""SELECT round(monto_final/monto_tender, 4) AS razon, count(*) AS procesos,
     round(100.0*count(*)/sum(count(*)) OVER (),2) AS pct
   FROM w WHERE familia = 'Subasta inversa electronica' AND NOT unico
     AND monto_tender IS NOT NULL AND monto_tender > 0
   GROUP BY 1 ORDER BY procesos DESC LIMIT 10""").df().to_string(index=False))

# ---------- (2) ESTACIONALIDAD DESCOMPUESTA ----------
t('(2a) COMPOSICION POR MODALIDAD DE CADA MES  (% de los procesos del mes)')
CM = con.sql("""SELECT month(fecha_proceso) AS mes, familia, count(*) AS n
   FROM w WHERE anio BETWEEN 2015 AND 2024 GROUP BY 1,2""").df()
piv = CM.pivot(index='familia', columns='mes', values='n').fillna(0)
pct = (100 * piv / piv.sum()).round(2)
print(g(pct.reset_index(), 'composicion_modalidad_por_mes').to_string(index=False))

t('(2b) DESCOMPOSICION DE ENERO, FEBRERO Y DICIEMBRE CONTRA LA BASE DE MARZO A NOVIEMBRE')
D = con.sql("""SELECT familia, month(fecha_proceso) AS mes, count(*) AS n,
     count(*) FILTER (WHERE unico) AS u
   FROM w WHERE anio BETWEEN 2015 AND 2024 AND unico IS NOT NULL GROUP BY 1,2""").df()
B = D[D.mes.between(3, 11)].groupby('familia')[['n', 'u']].sum()
res = []
for m in (1, 2, 12):
    E = D[D.mes == m].set_index('familia')[['n', 'u']].reindex(B.index).fillna(0)
    sB, sE = B.n / B.n.sum(), E.n / E.n.sum()
    uB = (B.u / B.n).fillna(0)
    uE = (E.u / E.n.replace(0, pd.NA)).astype(float).fillna(0)
    UB, UE = float((sB * uB).sum()), float((sE * uE).sum())
    comp, dentro = float(((sE - sB) * uB).sum()), float((sB * (uE - uB)).sum())
    inter = float(((sE - sB) * (uE - uB)).sum())
    res.append({'mes': m, 'oferente_unico_base_mar_nov_pct': round(100*UB, 2),
                'oferente_unico_del_mes_pct': round(100*UE, 2),
                'diferencia_pp': round(100*(UE-UB), 2),
                'por_composicion_pp': round(100*comp, 2),
                'por_comportamiento_pp': round(100*dentro, 2),
                'interaccion_pp': round(100*inter, 2),
                'pct_del_cambio_por_composicion': round(100*comp/(UE-UB), 1) if UE != UB else None})
print(g(pd.DataFrame(res), 'descomposicion_estacionalidad').to_string(index=False))
print('\nSi la mayor parte de la diferencia de enero es composicion, el inicio del ejercicio no es')
print('menos competitivo: simplemente se contrata por modalidades que no compiten.')

# ---------- (3) GEOGRAFIA ----------
t('(3a) LAS 15 PROVINCIAS CON MAS CONTRATACION, POR LA ENTIDAD QUE CONTRATA')
print(g(con.sql("""SELECT provincia_entidad AS provincia, count(*) AS procesos,
     count(DISTINCT entidad_ruc) AS entidades, count(DISTINCT proveedor_ruc) AS proveedores,
     round(sum(monto_final)/1e6,1) AS millones,
     round(100.0*sum(monto_final)/(SELECT sum(monto_final) FROM w),2) AS pct_del_monto,
     round(median(monto_final),0) AS mediana_usd,
     round(100.0*count(*) FILTER (WHERE fuera_de_provincia)/count(*),2) AS pct_a_proveedor_de_otra_provincia,
     round(100.0*count(*) FILTER (WHERE unico)/count(*),2) AS pct_oferente_unico
   FROM w WHERE provincia_entidad IS NOT NULL
   GROUP BY 1 ORDER BY millones DESC LIMIT 15""").df(),
   'geografia_por_provincia_entidad').to_string(index=False))

t('(3b) LAS 15 PROVINCIAS DE DONDE SALEN LOS PROVEEDORES')
print(g(con.sql("""SELECT provincia_proveedor AS provincia, count(*) AS procesos,
     count(DISTINCT proveedor_ruc) AS proveedores,
     count(DISTINCT provincia_entidad) AS provincias_a_las_que_vende,
     round(sum(monto_final)/1e6,1) AS millones,
     round(100.0*sum(monto_final)/(SELECT sum(monto_final) FROM w),2) AS pct_del_monto,
     round(median(monto_final),0) AS mediana_usd
   FROM w WHERE provincia_proveedor IS NOT NULL
   GROUP BY 1 ORDER BY millones DESC LIMIT 15""").df(),
   'geografia_por_provincia_proveedor').to_string(index=False))

t('(3c) CONTRATACION FUERA DE PROVINCIA, POR MODALIDAD Y POR AÑO')
print(g(con.sql("""SELECT familia, count(*) AS procesos,
     round(100.0*count(*) FILTER (WHERE fuera_de_provincia)/count(*),2) AS pct_fuera_de_provincia,
     round(100.0*sum(CASE WHEN fuera_de_provincia THEN monto_final ELSE 0 END)/sum(monto_final),2) AS pct_del_monto_fuera,
     round(median(CASE WHEN fuera_de_provincia THEN monto_final END),0) AS mediana_fuera,
     round(median(CASE WHEN NOT fuera_de_provincia THEN monto_final END),0) AS mediana_local
   FROM w WHERE fuera_de_provincia IS NOT NULL
   GROUP BY 1 ORDER BY pct_fuera_de_provincia DESC""").df(),
   'fuera_de_provincia_por_familia').to_string(index=False))
print()
print(con.sql("""SELECT anio, count(*) AS procesos,
     round(100.0*count(*) FILTER (WHERE fuera_de_provincia)/count(*),2) AS pct_fuera_de_provincia,
     round(100.0*sum(CASE WHEN fuera_de_provincia THEN monto_final ELSE 0 END)/sum(monto_final),2) AS pct_del_monto_fuera
   FROM w WHERE fuera_de_provincia IS NOT NULL AND anio BETWEEN 2015 AND 2024
   GROUP BY 1 ORDER BY 1""").df().to_string(index=False))

t('(3d) RETENCION LOCAL: cuanta plata de cada provincia se queda en proveedores de la misma provincia')
print(g(con.sql("""SELECT provincia_entidad AS provincia,
     round(sum(monto_final)/1e6,1) AS millones_contratados,
     round(100.0*sum(CASE WHEN provincia_proveedor = provincia_entidad THEN monto_final ELSE 0 END)
           /sum(monto_final),2) AS pct_que_se_queda_en_la_provincia,
     round(100.0*sum(CASE WHEN provincia_proveedor IN ('PICHINCHA','GUAYAS') THEN monto_final ELSE 0 END)
           /sum(monto_final),2) AS pct_que_va_a_pichincha_o_guayas
   FROM w WHERE provincia_entidad IS NOT NULL AND provincia_proveedor IS NOT NULL
   GROUP BY 1 ORDER BY millones_contratados DESC LIMIT 20""").df(),
   'retencion_local_por_provincia').to_string(index=False))
print('\nOJO: si los nombres de provincia vienen con otra grafia (minusculas, tildes), dime como')
print('salen y ajusto el filtro de Pichincha y Guayas.')
print('\nPega aqui toda la salida.')

In [ ]:
# ============ CELDA 41 · PARTE 2 (2b): CRUCES DE BANDERAS, FRACCIONAMIENTO Y VALIDACION ============
# (4) cuanto se solapan las nueve variables del modelo entre si
# (5) fraccionamiento: entidades y pares con mayor incidencia
# (6) los 108 procesos de los 15 RUC marcados por el SRI contra el resto de la base
import os, itertools
import pandas as pd, duckdb

os.chdir(r'C:\SERCOP_FINAL')
EST = 'plan_A/documentacion/estadisticas'; os.makedirs(EST, exist_ok=True)
pd.set_option('display.width', 260); pd.set_option('display.max_columns', 80); pd.set_option('display.max_rows', 220)
con = duckdb.connect(); con.sql("SET preserve_insertion_order=false")
def t(x): print('\n' + '=' * 95); print(x); print('=' * 95)
def g(df, n):
    df.to_csv(f'{EST}/{n}.csv', index=False, encoding='utf-8-sig'); return df

con.sql("CREATE OR REPLACE VIEW ind AS SELECT * FROM read_parquet('plan_A/planA_indicadores.parquet')")
con.sql("CREATE OR REPLACE VIEW pm  AS SELECT * FROM read_parquet('plan_A/planA_procesos_modelo.parquet')")
con.sql("CREATE OR REPLACE VIEW val AS SELECT * FROM read_parquet('plan_A/planA_validacion_sri.parquet')")
con.sql("CREATE OR REPLACE VIEW dpr AS SELECT * FROM read_parquet('plan_A/planA_dim_proveedor.parquet')")
V = ['pct_precio_alto', 'pct_cuota_12m', 'pct_recurrencia_par', 'pct_monto_capital',
     'pct_fraccionamiento', 'pct_proveedor_nuevo', 'pct_poca_concurrencia', 'pct_plazo_corto',
     'pct_firma_rapida']
print('columnas del conjunto de validacion:', list(con.sql("DESCRIBE SELECT * FROM val").df().column_name))

# ---------- (4a) CORRELACION ENTRE LAS NUEVE VARIABLES ----------
t('(4a) CORRELACION ENTRE LAS NUEVE VARIABLES DEL MODELO')
pz = [f"""SELECT '{a}' AS v1, '{b}' AS v2, round(corr("{a}","{b}"),3) AS r,
      count(*) FILTER (WHERE "{a}" IS NOT NULL AND "{b}" IS NOT NULL) AS pares FROM ind"""
      for a, b in itertools.combinations(V, 2)]
C = con.sql('\nUNION ALL\n'.join(pz)).df()
M = pd.DataFrame(index=[v.replace('pct_', '') for v in V], columns=[v.replace('pct_', '') for v in V], dtype=float)
for _, r in C.iterrows():
    M.loc[r.v1.replace('pct_', ''), r.v2.replace('pct_', '')] = r.r
    M.loc[r.v2.replace('pct_', ''), r.v1.replace('pct_', '')] = r.r
for v in M.index: M.loc[v, v] = 1.0
print(M.round(3).to_string())
g(M.reset_index().rename(columns={'index': 'variable'}), 'correlacion_entre_variables')
print('\n-- las 8 correlaciones mas altas en valor absoluto --')
C['abs_r'] = C.r.abs()
print(C.sort_values('abs_r', ascending=False).head(8)[['v1','v2','r','pares']].to_string(index=False))

# ---------- (4b) CUANTAS BANDERAS SE PRENDEN A LA VEZ ----------
t('(4b) BANDERAS SIMULTANEAS  (bandera = la variable esta en el 5% superior de su grupo)')
suma_al = ' + '.join(f'CASE WHEN "{v}" >= 0.95 THEN 1 ELSE 0 END' for v in V)
suma_me = ' + '.join(f'CASE WHEN "{v}" IS NOT NULL THEN 1 ELSE 0 END' for v in V)
con.sql(f"""CREATE OR REPLACE TABLE al AS
  SELECT ocid, familia, anio, monto_final, ({suma_al}) AS n_alertas, ({suma_me}) AS n_medibles FROM ind""")
print(g(con.sql("""SELECT n_alertas, count(*) AS procesos,
     round(100.0*count(*)/sum(count(*)) OVER (),3) AS pct_procesos,
     round(sum(monto_final)/1e6,1) AS millones,
     round(100.0*sum(monto_final)/sum(sum(monto_final)) OVER (),2) AS pct_monto,
     round(median(monto_final),0) AS mediana_usd, round(median(n_medibles),0) AS mediana_medibles
   FROM al GROUP BY 1 ORDER BY 1""").df(), 'banderas_simultaneas').to_string(index=False))
print('\nSi las nueve variables fueran independientes, con 5% de probabilidad cada una, los procesos')
print('con 3 o mas banderas serian el 0,84% del total. Mas que eso indica que las señales se solapan.')
print(con.sql("""SELECT count(*) FILTER (WHERE n_alertas = 0) AS sin_ninguna,
     count(*) FILTER (WHERE n_alertas >= 1) AS con_una_o_mas,
     count(*) FILTER (WHERE n_alertas >= 2) AS con_dos_o_mas,
     count(*) FILTER (WHERE n_alertas >= 3) AS con_tres_o_mas,
     count(*) FILTER (WHERE n_alertas >= 4) AS con_cuatro_o_mas,
     round(100.0*count(*) FILTER (WHERE n_alertas >= 3)/count(*),3) AS pct_tres_o_mas,
     round(100.0*sum(CASE WHEN n_alertas >= 3 THEN monto_final ELSE 0 END)/sum(monto_final),2) AS pct_monto_tres_o_mas
   FROM al""").df().to_string(index=False))
print('\n-- cobertura: cuantas de las nueve variables son medibles en cada proceso --')
print(con.sql("""SELECT n_medibles, count(*) AS procesos,
     round(100.0*count(*)/sum(count(*)) OVER (),2) AS pct
   FROM al GROUP BY 1 ORDER BY 1""").df().to_string(index=False))
print('\n-- banderas simultaneas por modalidad --')
print(g(con.sql("""SELECT familia, count(*) AS procesos, round(avg(n_alertas),3) AS banderas_promedio,
     round(100.0*count(*) FILTER (WHERE n_alertas >= 3)/count(*),3) AS pct_tres_o_mas,
     round(avg(n_medibles),2) AS medibles_promedio
   FROM al GROUP BY 1 ORDER BY pct_tres_o_mas DESC""").df(),
   'banderas_por_familia').to_string(index=False))

t('(4c) QUE PARES DE BANDERAS SE PRENDEN JUNTOS MAS DE LO ESPERABLE')
pz = []
for a, b in itertools.combinations(V, 2):
    pz.append(f"""SELECT '{a.replace('pct_','')}' AS bandera_1, '{b.replace('pct_','')}' AS bandera_2,
      count(*) AS pares,
      round(100.0*count(*) FILTER (WHERE "{a}">=0.95)/count(*),2) AS pct_1,
      round(100.0*count(*) FILTER (WHERE "{b}">=0.95)/count(*),2) AS pct_2,
      round(100.0*count(*) FILTER (WHERE "{a}">=0.95 AND "{b}">=0.95)/count(*),3) AS pct_ambas,
      round((count(*) FILTER (WHERE "{a}">=0.95 AND "{b}">=0.95)*1.0/count(*))
            / nullif((count(*) FILTER (WHERE "{a}">=0.95)*1.0/count(*))
                   * (count(*) FILTER (WHERE "{b}">=0.95)*1.0/count(*)),0),2) AS cuantas_veces_lo_esperado
      FROM ind WHERE "{a}" IS NOT NULL AND "{b}" IS NOT NULL""")
L = con.sql('\nUNION ALL\n'.join(pz)).df().sort_values('cuantas_veces_lo_esperado', ascending=False)
print(g(L, 'pares_de_banderas').head(12).to_string(index=False))
print('\n-- y los pares que casi nunca coinciden --')
print(L.tail(6).to_string(index=False))
print('\nUn valor de 1 significa independencia. Muy por encima de 1: las dos banderas miden')
print('practicamente lo mismo y no aportan señales separadas al modelo.')

# ---------- (5) FRACCIONAMIENTO ----------
t('(5a) DISTRIBUCION DEL FRACCIONAMIENTO (procesos previos del mismo par y CPC5 en 30 dias)')
print(g(con.sql("""SELECT ind_fraccionamiento_n_30d AS procesos_previos_30d, count(*) AS procesos,
     round(100.0*count(*)/sum(count(*)) OVER (),3) AS pct,
     round(sum(monto_final)/1e6,1) AS millones,
     round(median(monto_final),0) AS mediana_usd,
     round(median(ind_fraccionamiento_monto_30d),0) AS mediana_monto_acumulado_previo
   FROM ind GROUP BY 1 ORDER BY 1""").df(), 'distribucion_fraccionamiento').to_string(index=False))

t('(5b) LAS 20 ENTIDADES CON MAS FRACCIONAMIENTO (minimo 200 procesos)')
print(g(con.sql("""SELECT ind.entidad_ruc, any_value(pm.entidad_nombre) AS entidad,
     count(*) AS procesos,
     count(*) FILTER (WHERE ind.ind_fraccionamiento_marca) AS con_proceso_previo_30d,
     round(100.0*count(*) FILTER (WHERE ind.ind_fraccionamiento_marca)/count(*),2) AS pct,
     round(sum(ind.monto_final)/1e6,1) AS millones,
     round(median(ind.monto_final),0) AS mediana_usd
   FROM ind JOIN pm USING (ocid) GROUP BY 1 HAVING count(*) >= 200
   ORDER BY pct DESC LIMIT 20""").df(), 'entidades_con_mas_fraccionamiento').to_string(index=False))
print(f"\nreferencia: el promedio de la base es "
      f"{con.sql('SELECT round(100.0*count(*) FILTER (WHERE ind_fraccionamiento_marca)/count(*),2) FROM ind').fetchone()[0]}%")

t('(5c) LOS 15 PARES ENTIDAD-PROVEEDOR CON MAS PROCESOS SUCESIVOS EN 30 DIAS')
print(g(con.sql("""SELECT ind.entidad_ruc, ind.proveedor_ruc,
     any_value(pm.entidad_nombre) AS entidad, any_value(dpr.sri_RAZON_SOCIAL) AS proveedor,
     count(*) AS procesos_del_par,
     count(*) FILTER (WHERE ind.ind_fraccionamiento_marca) AS con_previo_30d,
     max(ind.ind_fraccionamiento_n_30d) AS maximo_previos_en_30d,
     round(sum(ind.monto_final)/1e6,2) AS millones,
     round(max(ind.ind_fraccionamiento_monto_30d),0) AS maximo_monto_acumulado_30d
   FROM ind JOIN pm USING (ocid) LEFT JOIN dpr ON dpr.ruc = ind.proveedor_ruc
   GROUP BY 1,2 ORDER BY con_previo_30d DESC LIMIT 15""").df(),
   'pares_con_mas_fraccionamiento').to_string(index=False))

# ---------- (6) VALIDACION EXTERNA ----------
t('(6a) LOS 15 RUC MARCADOS POR EL SRI: SUS PROCESOS CONTRA EL RESTO DE LA BASE')
kval = next((c for c in con.sql("DESCRIBE SELECT * FROM val").df().column_name if c.lower() == 'ruc'), 'ruc')
con.sql(f"""CREATE OR REPLACE TABLE cmp AS
  SELECT ind.*, al.n_alertas, al.n_medibles,
         (ind.proveedor_ruc IN (SELECT "{kval}" FROM val)) AS marcado_sri
  FROM ind JOIN al USING (ocid)""")
print(con.sql("""SELECT marcado_sri, count(*) AS procesos, count(DISTINCT proveedor_ruc) AS proveedores,
     count(DISTINCT entidad_ruc) AS entidades, round(sum(monto_final)/1e6,2) AS millones,
     round(median(monto_final),0) AS mediana_usd,
     round(avg(n_alertas),3) AS banderas_promedio,
     round(100.0*count(*) FILTER (WHERE n_alertas >= 1)/count(*),2) AS pct_con_una_o_mas,
     round(100.0*count(*) FILTER (WHERE n_alertas >= 3)/count(*),2) AS pct_con_tres_o_mas,
     round(100.0*count(*) FILTER (WHERE ind_oferente_unico)/count(*),2) AS pct_oferente_unico
   FROM cmp GROUP BY 1 ORDER BY 1""").df().to_string(index=False))
print('\nADVERTENCIA: 15 contribuyentes y 108 procesos no permiten inferencia estadistica. Esta')
print('comparacion es validacion externa cualitativa, no una prueba de hipotesis.')

t('(6b) MEDIANA DE CADA VARIABLE DEL MODELO: marcados contra el resto')
pz = [f"""SELECT '{v.replace('pct_','')}' AS variable,
      round(median(CASE WHEN marcado_sri THEN "{v}" END),4) AS mediana_marcados,
      round(median(CASE WHEN NOT marcado_sri THEN "{v}" END),4) AS mediana_resto,
      count("{v}") FILTER (WHERE marcado_sri) AS medibles_marcados
      FROM cmp""" for v in V]
print(g(con.sql('\nUNION ALL\n'.join(pz)).df(), 'validacion_sri_contra_resto').to_string(index=False))

t('(6c) LOS 15 RUC, UNO POR UNO')
print(g(con.sql(f"""SELECT v."{kval}" AS ruc, any_value(d.sri_RAZON_SOCIAL) AS proveedor,
     count(c.ocid) AS procesos, round(coalesce(sum(c.monto_final),0)/1e6,3) AS millones,
     round(avg(c.n_alertas),2) AS banderas_promedio,
     max(c.n_alertas) AS maximo_banderas,
     min(c.anio) AS primer_anio, max(c.anio) AS ultimo_anio
   FROM val v LEFT JOIN cmp c ON c.proveedor_ruc = v."{kval}"
     LEFT JOIN dpr d ON d.ruc = v."{kval}"
   GROUP BY 1 ORDER BY procesos DESC""").df(), 'validacion_sri_por_ruc').to_string(index=False))
print('\nPega aqui toda la salida.')

In [ ]:
# ============ CELDA 42 · FIGURAS PARA LA TESIS ============
# Crea plan_A/graficos/ con 12 PNG a 300 dpi listos para Word y un catalogo con la
# descripcion de cada figura. Cada figura va en su propio bloque: si una falla, las demas siguen.
import os, math, warnings, itertools
import pandas as pd, numpy as np, duckdb
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
warnings.filterwarnings('ignore')

os.chdir(r'C:\SERCOP_FINAL')
GRA = 'plan_A/graficos'; EST = 'plan_A/documentacion/estadisticas'
os.makedirs(GRA, exist_ok=True)
con = duckdb.connect(); con.sql("SET preserve_insertion_order=false")
con.sql("CREATE OR REPLACE VIEW pm  AS SELECT * FROM read_parquet('plan_A/planA_procesos_modelo.parquet')")
con.sql("CREATE OR REPLACE VIEW ind AS SELECT * FROM read_parquet('plan_A/planA_indicadores.parquet')")
con.sql("CREATE OR REPLACE VIEW val AS SELECT * FROM read_parquet('plan_A/planA_validacion_sri.parquet')")

# ---------- paleta validada y estilo comun ----------
AZUL, NARANJA, AGUA = '#2a78d6', '#eb6834', '#1baf7a'
SURF, TINTA, TINTA2, MUDO, REJILLA, EJE = '#fcfcfb', '#0b0b0b', '#52514e', '#898781', '#e1e0d9', '#c3c2b7'
plt.rcParams.update({'font.family': ['Segoe UI', 'DejaVu Sans', 'sans-serif'],
                     'font.size': 9, 'axes.titlesize': 11, 'figure.dpi': 110})
CATALOGO = []

def lienzo(alto=4.2, ancho=7.2, filas=1):
    fig, ax = plt.subplots(filas, 1, figsize=(ancho, alto), facecolor=SURF)
    fig.patch.set_facecolor(SURF)
    for a in (ax if filas > 1 else [ax]):
        a.set_facecolor(SURF)
        a.spines['top'].set_visible(False); a.spines['right'].set_visible(False)
        a.spines['left'].set_color(EJE); a.spines['bottom'].set_color(EJE)
        a.spines['left'].set_linewidth(0.8); a.spines['bottom'].set_linewidth(0.8)
        a.tick_params(colors=MUDO, labelsize=8, length=3)
        a.set_axisbelow(True)
    return fig, ax

def rotular(ax, titulo, subtitulo=None):
    ax.set_title(titulo, color=TINTA, loc='left', pad=14 if subtitulo else 8, fontweight='semibold')
    if subtitulo:
        ax.text(0, 1.015, subtitulo, transform=ax.transAxes, color=TINTA2, fontsize=8.5, va='bottom')

def guardar(fig, archivo, titulo, muestra, lectura, fuente, bloque):
    r = f'{GRA}/{archivo}'
    fig.savefig(r, dpi=300, bbox_inches='tight', facecolor=SURF)
    plt.close(fig)
    CATALOGO.append({'archivo': archivo, 'titulo': titulo, 'que_muestra': muestra,
                     'como_leerla': lectura, 'datos_de_origen': fuente, 'bloque': bloque})
    print(f'  OK  {archivo}')

def falla(archivo, e):
    print(f'  FALLA {archivo}: {type(e).__name__}: {e}')

print('=' * 95); print('GENERANDO FIGURAS'); print('=' * 95)

# ---------- F01 · distribucion del monto ----------
try:
    m = con.sql("SELECT monto_final FROM pm WHERE monto_final > 0").df().monto_final.values
    lg = np.log10(m); med = float(np.median(m))
    fig, ax = lienzo()
    ax.hist(lg, bins=90, color=AZUL, edgecolor=SURF, linewidth=0.3)
    ax.axvline(math.log10(med), color=TINTA, linewidth=1.2)
    ax.annotate(f'mediana\n{med:,.0f} USD', xy=(math.log10(med), ax.get_ylim()[1]*0.88),
                xytext=(8, 0), textcoords='offset points', color=TINTA, fontsize=8.5, va='top')
    ax.set_xticks(range(0, 10))
    ax.set_xticklabels(['1', '10', '100', '1 mil', '10 mil', '100 mil', '1 M', '10 M', '100 M', '1 000 M'])
    ax.set_xlabel('monto adjudicado, escala logaritmica (USD)', color=TINTA2, fontsize=8.5)
    ax.set_ylabel('procesos', color=TINTA2, fontsize=8.5)
    ax.grid(axis='y', color=REJILLA, linewidth=0.6)
    rotular(ax, 'La distribucion del monto adjudicado abarca ocho ordenes de magnitud',
            '446.677 procesos modelables, 2015-2026. El eje horizontal es logaritmico.')
    guardar(fig, 'F01_distribucion_monto.png',
            'Distribucion del monto adjudicado en escala logaritmica',
            'El histograma del logaritmo del monto de los 446.677 procesos de la base de modelado, '
            'con la mediana marcada.',
            'La forma aproximadamente simetrica en escala logaritmica, frente a la fuerte asimetria '
            'en escala natural, es lo que justifica trabajar con el logaritmo del monto y reportar '
            'medianas en lugar de promedios. El promedio equivale a 4,3 veces la mediana.',
            'planA_procesos_modelo.parquet', 'Bloque 16')
except Exception as e: falla('F01', e)

# ---------- F02 · serie anual, dos paneles ----------
try:
    A = con.sql("""SELECT anio_proceso AS anio, count(*) AS procesos,
         round(sum(monto_final)/1e6,1) AS millones FROM pm GROUP BY 1 ORDER BY 1""").df()
    inc = A.anio >= 2025
    fig, ax = lienzo(alto=6.2, filas=2)
    for a, col, lab, fmt in ((ax[0], 'procesos', 'procesos', '{:,.0f}'),
                             (ax[1], 'millones', 'millones de USD', '{:,.0f}')):
        a.bar(A.anio[~inc], A[col][~inc], color=AZUL, width=0.7)
        a.bar(A.anio[inc], A[col][inc], color=SURF, width=0.7, edgecolor=AZUL,
              linewidth=0.9, hatch='////')
        a.set_ylabel(lab, color=TINTA2, fontsize=8.5)
        a.grid(axis='y', color=REJILLA, linewidth=0.6)
        a.set_xticks(A.anio)
    ax[1].set_xlabel('anio de inicio del procedimiento', color=TINTA2, fontsize=8.5)
    ax[0].annotate('2025 y 2026 incompletos\n(retraso de publicacion)', xy=(2025.5, A.procesos.max()*0.72),
                   ha='center', color=TINTA2, fontsize=8)
    rotular(ax[0], 'La caida del volumen esta en 2020, no en 2021',
            'Numero de procesos (arriba) y valor contratado (abajo). Dos paneles, no dos ejes.')
    fig.tight_layout()
    guardar(fig, 'F02_serie_anual.png', 'Evolucion anual del volumen y del valor contratado',
            'Procesos por anio y millones de dolares por anio, usando el anio de inicio del periodo '
            'de ofertas. Los anios 2025 y 2026 llevan trama porque estan incompletos.',
            'La contraccion real se produce en 2020 (27.908 procesos, un 27,6% menos que 2019) con '
            'recuperacion en 2021 y maximo de valor en 2022. La serie construida sobre la fecha de '
            'publicacion del release mostraba en cambio un minimo falso en 2021.',
            'documentacion/estadisticas/serie_anual_canonica.csv', 'Bloque 19')
except Exception as e: falla('F02', e)

# ---------- F03 · retraso de publicacion ----------
try:
    R = con.sql("""SELECT anio_proceso AS anio,
         round(median(dias_retraso_publicacion),0) AS mediana,
         count(*) FILTER (WHERE dias_retraso_publicacion > 365) AS mas_de_un_anio
       FROM pm GROUP BY 1 ORDER BY 1""").df()
    fig, ax = lienzo()
    ax.bar(R.anio, R.mediana, color=AZUL, width=0.7)
    ax.axvline(2019.5, color=NARANJA, linewidth=1.4)
    ax.annotate('quiebre del regimen\nde publicacion', xy=(2019.6, R.mediana.max()*0.86),
                color=NARANJA, fontsize=8.5)
    for x, y in zip(R.anio, R.mediana):
        if y > 0: ax.annotate(f'{y:.0f}', xy=(x, y), xytext=(0, 3), textcoords='offset points',
                              ha='center', color=TINTA2, fontsize=8)
    ax.set_xticks(R.anio); ax.set_ylabel('dias', color=TINTA2, fontsize=8.5)
    ax.set_xlabel('anio de inicio del procedimiento', color=TINTA2, fontsize=8.5)
    ax.grid(axis='y', color=REJILLA, linewidth=0.6)
    rotular(ax, 'Hasta 2019 el portal publicaba el mismo dia; desde 2020 hay meses de retraso',
            'Mediana de dias entre el inicio del procedimiento y la publicacion del release.')
    guardar(fig, 'F03_retraso_publicacion.png',
            'Desfase entre el inicio del procedimiento y su publicacion en datos abiertos',
            'Mediana de dias de retraso de publicacion por anio del procedimiento.',
            'El valor es cero hasta 2019 y salta a 76 dias en 2021 y 147 en 2022. Es un cambio en la '
            'practica de publicacion del portal, no en la contratacion, y deforma cualquier serie '
            'temporal construida sobre la fecha de release a partir de 2020. Tambien implica que los '
            'ultimos meses de cualquier descarga estan incompletos.',
            'planA_procesos_modelo.parquet, columna dias_retraso_publicacion', 'Bloque 36')
except Exception as e: falla('F03', e)

# ---------- F04 · descomposicion del oferente unico ----------
try:
    D = con.sql("""SELECT familia, anio, count(*) AS n, count(*) FILTER (WHERE ind_oferente_unico) AS u
       FROM ind WHERE anio IN (2015, 2024) GROUP BY 1,2""").df()
    pn = D.pivot(index='familia', columns='anio', values='n').fillna(0)
    pu = D.assign(r=D.u / D.n).pivot(index='familia', columns='anio', values='r')
    s0, s1 = pn[2015] / pn[2015].sum(), pn[2024] / pn[2024].sum()
    u0, u1 = pu[2015].fillna(0), pu[2024].fillna(0)
    comp = 100 * float(((s1 - s0) * u0).sum()); dentro = 100 * float((s0 * (u1 - u0)).sum())
    inter = 100 * float(((s1 - s0) * (u1 - u0)).sum()); tot = comp + dentro + inter
    et = ['recomposicion\nde modalidades', 'mejora dentro de\ncada modalidad', 'interaccion', 'cambio total']
    vl = [comp, dentro, inter, tot]
    fig, ax = lienzo(alto=3.6)
    ax.barh(range(4), vl, color=[AZUL, AZUL, AZUL, TINTA2], height=0.6)
    for i, v in enumerate(vl):
        ax.annotate(f'{v:+.2f} pp', xy=(v, i), xytext=(-6 if v < 0 else 6, 0),
                    textcoords='offset points', va='center', ha='right' if v < 0 else 'left',
                    color=TINTA, fontsize=9)
    ax.set_yticks(range(4)); ax.set_yticklabels(et, fontsize=8.5, color=TINTA2)
    ax.invert_yaxis(); ax.axvline(0, color=EJE, linewidth=0.8)
    ax.set_xlabel('puntos porcentuales', color=TINTA2, fontsize=8.5)
    ax.grid(axis='x', color=REJILLA, linewidth=0.6); ax.set_xlim(min(vl) * 1.35, 2)
    rotular(ax, 'La mitad de la mejora en concurrencia es recomposicion, no competencia',
            'Descomposicion de cambio-participacion de la tasa de oferente unico, 2015 a 2024.')
    guardar(fig, 'F04_descomposicion_oferente_unico.png',
            'Descomposicion del cambio en la tasa de oferente unico',
            'Los tres componentes en que se descompone la caida de 18,07 puntos de la tasa de '
            'oferente unico entre 2015 y 2024.',
            'El componente de recomposicion mide cuanto cayo la tasa agregada por el solo hecho de '
            'que las modalidades sin concurrencia perdieron peso; el componente interno mide la '
            'mejora efectiva dentro de cada modalidad. Se reparten casi por mitades, de modo que '
            'ninguna de las dos lecturas simples es correcta.',
            'documentacion/estadisticas/descomposicion_oferente_unico.csv', 'Bloque 37')
except Exception as e: falla('F04', e)

# ---------- F05 · oferente unico por modalidad, 2015 contra 2024 ----------
try:
    T = pd.DataFrame({'f': pu.index, 'a2015': (100*u0).values, 'a2024': (100*u1).values,
                      'peso': (100*s1).values}).sort_values('a2015')
    fig, ax = lienzo(alto=4.4)
    y = np.arange(len(T))
    ax.hlines(y, T.a2015, T.a2024, color=EJE, linewidth=1.6, zorder=1)
    ax.scatter(T.a2015, y, s=58, color=AZUL, zorder=3, label='2015')
    ax.scatter(T.a2024, y, s=58, color=NARANJA, marker='D', zorder=3, label='2024')
    for i, r in enumerate(T.itertuples()):
        ax.annotate(f'{r.a2024:.0f}%', xy=(r.a2024, i), xytext=(9 if r.a2024 >= r.a2015 else -9, 0),
                    textcoords='offset points', va='center',
                    ha='left' if r.a2024 >= r.a2015 else 'right', color=TINTA, fontsize=8)
    ax.set_yticks(y); ax.set_yticklabels(T.f, fontsize=8.5, color=TINTA2)
    ax.set_xlabel('procesos con un unico oferente (%)', color=TINTA2, fontsize=8.5)
    ax.grid(axis='x', color=REJILLA, linewidth=0.6); ax.set_xlim(-4, 112)
    ax.legend(frameon=False, loc='lower right', fontsize=8.5, labelcolor=TINTA2)
    rotular(ax, 'La mejora real esta en subasta inversa y menor cuantia',
            'Tasa de oferente unico por modalidad. Circulo 2015, rombo 2024.')
    guardar(fig, 'F05_oferente_unico_por_modalidad.png',
            'Tasa de oferente unico por modalidad, 2015 contra 2024',
            'La tasa de procesos con un solo oferente en cada modalidad, en los dos extremos del '
            'periodo completo.',
            'Contratacion directa y regimen especial permanecen en torno al cien por ciento: no '
            'compiten por diseno legal. La mejora efectiva se concentra en menor cuantia, que cae '
            'del 56,43% al 32,49%, y en subasta inversa electronica, del 28,82% al 19,61%. La '
            'figura justifica comparar cada indicador dentro de su modalidad y no contra el universo.',
            'documentacion/estadisticas/descomposicion_oferente_unico.csv', 'Bloque 37')
except Exception as e: falla('F05', e)

# ---------- F06 · estacionalidad ----------
try:
    M = con.sql("""SELECT month(fecha_proceso) AS mes, count(*) AS n, sum(monto_final) AS m
       FROM ind WHERE anio BETWEEN 2015 AND 2024 GROUP BY 1 ORDER BY 1""").df()
    M['pct_monto'] = 100 * M.m / M.m.sum()
    MES = ['ene','feb','mar','abr','may','jun','jul','ago','sep','oct','nov','dic']
    fig, ax = lienzo()
    col = [NARANJA if x == 12 else AZUL for x in M.mes]
    ax.bar(M.mes, M.pct_monto, color=col, width=0.7)
    ax.axhline(100/12, color=TINTA2, linewidth=1.0)
    ax.annotate('reparto uniforme (8,33%)', xy=(1.0, 100/12), xytext=(0, 5),
                textcoords='offset points', color=TINTA2, fontsize=8)
    ax.annotate(f'{M.pct_monto.iloc[11]:.2f}%', xy=(12, M.pct_monto.iloc[11]), xytext=(0, 4),
                textcoords='offset points', ha='center', color=NARANJA, fontsize=9)
    ax.set_xticks(M.mes); ax.set_xticklabels(MES)
    ax.set_ylabel('valor contratado del periodo (%)', color=TINTA2, fontsize=8.5)
    ax.grid(axis='y', color=REJILLA, linewidth=0.6)
    rotular(ax, 'Diciembre concentra el 13,96% del valor contratado del anio',
            'Reparto mensual del monto adjudicado, 2015-2024, contra el reparto uniforme.')
    guardar(fig, 'F06_estacionalidad.png', 'Reparto mensual del valor contratado',
            'Porcentaje del valor contratado del periodo 2015-2024 que corresponde a cada mes, con '
            'la linea del reparto uniforme de 8,33% como referencia.',
            'Enero apenas reune el 1,84% del valor y diciembre el 13,96%, con una mediana de monto '
            'un 33% superior a la del resto del anio. El patron es consistente con concentracion de '
            'ejecucion presupuestaria al cierre del ejercicio. La concurrencia, en cambio, no se '
            'deteriora en diciembre: la diferencia en la tasa de oferente unico es de 0,44 puntos.',
            'documentacion/estadisticas/estacionalidad_mensual.csv', 'Bloque 39')
except Exception as e: falla('F06', e)

# ---------- F07 · Benford ----------
try:
    B = con.sql("""SELECT CAST(floor(monto_final/pow(10, floor(log10(monto_final)))) AS INTEGER) AS d,
         count(*) AS n FROM pm WHERE monto_final > 0 GROUP BY 1 ORDER BY 1""").df()
    B = B[B.d.between(1, 9)]
    tot = B.n.sum()
    B['obs'] = 100 * B.n / tot
    B['esp'] = [100 * math.log10(1 + 1/d) for d in B.d]
    mad = float((B.obs/100 - B.esp/100).abs().mean())
    fig, ax = lienzo()
    ax.bar(B.d, B.obs, color=AZUL, width=0.62, label='observado')
    ax.plot(B.d, B.esp, color=NARANJA, linewidth=1.6, marker='D', markersize=7,
            markeredgecolor=SURF, markeredgewidth=1.2, linestyle='--', label='esperado por Benford')
    ax.set_xticks(B.d); ax.set_xlabel('primer digito significativo del monto', color=TINTA2, fontsize=8.5)
    ax.set_ylabel('procesos (%)', color=TINTA2, fontsize=8.5)
    ax.grid(axis='y', color=REJILLA, linewidth=0.6)
    ax.legend(frameon=False, fontsize=8.5, labelcolor=TINTA2)
    ax.annotate(f'MAD de Nigrini = {mad:.5f}  (conformidad aceptable)', xy=(0.98, 0.72),
                xycoords='axes fraction', ha='right', color=TINTA2, fontsize=8.5)
    rotular(ax, 'El primer digito se ajusta razonablemente a la ley de Benford',
            f'{tot:,} procesos. Barras: frecuencia observada. Rombos: frecuencia teorica.')
    guardar(fig, 'F07_benford.png', 'Conformidad con la ley de Benford del primer digito',
            'Frecuencia observada del primer digito significativo del monto adjudicado frente a la '
            'frecuencia teorica de la ley de Benford.',
            'La desviacion mayor es del digito uno, con 2,42 puntos por encima de lo esperado. El '
            'chi-cuadrado rechaza la conformidad, pero con 446.677 observaciones rechaza casi '
            'cualquier desviacion, por lo que se reporta la desviacion absoluta media de Nigrini, '
            'independiente del tamano de muestra, que situa al conjunto en conformidad aceptable. '
            'El contraste no es aplicable en cotizacion ni lista corta, cuyos montos recorren apenas '
            'un orden de magnitud por efecto de los umbrales legales.',
            'documentacion/estadisticas/benford_primer_digito.csv', 'Bloque 35')
except Exception as e: falla('F07', e)

# ---------- F08 · matriz de correlacion ----------
try:
    V = ['pct_precio_alto','pct_cuota_12m','pct_recurrencia_par','pct_monto_capital',
         'pct_fraccionamiento','pct_proveedor_nuevo','pct_poca_concurrencia','pct_plazo_corto',
         'pct_firma_rapida']
    pz = [f"""SELECT '{a}' AS v1, '{b}' AS v2, corr("{a}","{b}") AS r FROM ind"""
          for a, b in itertools.combinations(V, 2)]
    C = con.sql('\nUNION ALL\n'.join(pz)).df()
    et = [v.replace('pct_', '').replace('_', ' ') for v in V]
    Mz = pd.DataFrame(np.eye(len(V)), index=V, columns=V)
    for _, r in C.iterrows():
        Mz.loc[r.v1, r.v2] = r.r; Mz.loc[r.v2, r.v1] = r.r
    cmap = LinearSegmentedColormap.from_list('div', [AZUL, '#f0efec', '#e34948'])
    fig, ax = lienzo(alto=5.6, ancho=6.6)
    im = ax.imshow(Mz.values, cmap=cmap, vmin=-1, vmax=1)
    ax.set_xticks(range(len(V))); ax.set_xticklabels(et, rotation=45, ha='right', fontsize=8)
    ax.set_yticks(range(len(V))); ax.set_yticklabels(et, fontsize=8)
    for i in range(len(V)):
        for j in range(len(V)):
            v = Mz.values[i, j]
            ax.text(j, i, f'{v:.2f}'.replace('-0.00', '0.00'), ha='center', va='center',
                    fontsize=7.2, color='#ffffff' if abs(v) > 0.6 else TINTA)
    for s in ax.spines.values(): s.set_visible(False)
    ax.tick_params(length=0)
    cb = fig.colorbar(im, ax=ax, shrink=0.72, ticks=[-1, -0.5, 0, 0.5, 1])
    cb.outline.set_visible(False); cb.ax.tick_params(colors=MUDO, labelsize=8)
    cb.set_label('correlacion de Pearson', color=TINTA2, fontsize=8.5)
    rotular(ax, 'Dos de las nueve variables del modelo son casi la misma',
            'Correlacion entre los nueve rangos percentiles que alimentan el modelo.')
    guardar(fig, 'F08_matriz_correlacion.png',
            'Correlacion entre las nueve variables del modelo',
            'Matriz de correlacion de Pearson entre los nueve rangos percentiles, con el valor '
            'impreso en cada celda.',
            'La correlacion de 0,97 entre la cuota del proveedor en la entidad y la recurrencia del '
            'par es en buena medida un artefacto: ambas variables tienen mas de la mitad de sus '
            'valores en cero exacto, correspondientes a los procesos sin historial previo del par, y '
            'la correlacion de Pearson queda dominada por esa coincidencia de ceros. La medida '
            'pertinente de redundancia es el solapamiento en la cola superior, que resulta de 3,93 '
            'veces lo esperable por azar y no de veinte. El azul indica relacion inversa, el rojo '
            'directa y el gris ausencia de relacion.',
            'documentacion/estadisticas/correlacion_entre_variables.csv', 'Bloque 42')
except Exception as e: falla('F08', e)

# ---------- F09 · curva de concentracion ----------
try:
    K = con.sql("""WITH s AS (SELECT proveedor_ruc_final AS ruc, sum(monto_final) AS m
         FROM pm WHERE proveedor_ruc_final IS NOT NULL GROUP BY 1),
       r AS (SELECT m, row_number() OVER (ORDER BY m DESC) AS p, count(*) OVER () AS n,
                    sum(m) OVER () AS t, sum(m) OVER (ORDER BY m DESC ROWS UNBOUNDED PRECEDING) AS a
             FROM s)
     SELECT p*1.0/n AS fx, a/t AS fy FROM r ORDER BY p""").df()
    fig, ax = lienzo()
    ax.plot([0, 1], [0, 1], color=EJE, linewidth=1.0, linestyle='-')
    ax.annotate('reparto igualitario', xy=(0.62, 0.62), xytext=(6, -12),
                textcoords='offset points', color=MUDO, fontsize=8, rotation=32)
    ax.plot(100*K.fx, 100*K.fy, color=AZUL, linewidth=2.0)
    for pct in (1, 5, 10):
        yy = 100 * float(K.fy[K.fx <= pct/100].max())
        ax.scatter([pct], [yy], s=46, color=AZUL, zorder=4, edgecolor=SURF, linewidth=1.2)
        ax.annotate(f'{pct}% de proveedores\nconcentra el {yy:.1f}%', xy=(pct, yy),
                    xytext=(10, -4), textcoords='offset points', color=TINTA, fontsize=8.5)
    ax.set_xlim(0, 100); ax.set_ylim(0, 101)
    ax.set_xlabel('proveedores ordenados de mayor a menor (%)', color=TINTA2, fontsize=8.5)
    ax.set_ylabel('valor contratado acumulado (%)', color=TINTA2, fontsize=8.5)
    ax.grid(color=REJILLA, linewidth=0.6)
    rotular(ax, 'El 1% de los proveedores concentra el 44% del valor contratado',
            '62.613 proveedores con al menos una adjudicacion en la base de modelado.')
    guardar(fig, 'F09_curva_concentracion.png', 'Curva de concentracion de proveedores',
            'Porcentaje acumulado del valor contratado en funcion del porcentaje acumulado de '
            'proveedores, ordenados de mayor a menor adjudicacion.',
            'La diagonal representaria un reparto perfectamente igualitario. La distancia entre la '
            'curva y la diagonal mide la desigualdad del reparto. Los 626 proveedores del primer '
            'centil concentran el 44,25% del valor y el primer decil el 79,71%, mientras la mitad '
            'inferior del padron participa en menos del 2%. Seis de los veinte mayores proveedores '
            'son empresas publicas, de modo que parte del valor corresponde a contratacion entre '
            'entidades del propio Estado.',
            'documentacion/estadisticas/curva_concentracion_proveedores.csv', 'Bloque 32')
except Exception as e: falla('F09', e)

# ---------- F10 · banderas simultaneas ----------
try:
    V9 = ['pct_precio_alto','pct_cuota_12m','pct_recurrencia_par','pct_monto_capital',
          'pct_fraccionamiento','pct_proveedor_nuevo','pct_poca_concurrencia','pct_plazo_corto',
          'pct_firma_rapida']
    sa = ' + '.join(f'CASE WHEN "{v}" >= 0.95 THEN 1 ELSE 0 END' for v in V9)
    F = con.sql(f"""SELECT ({sa}) AS k, count(*) AS n, sum(monto_final) AS m FROM ind GROUP BY 1 ORDER BY 1""").df()
    F['pn'] = 100 * F.n / F.n.sum(); F['pm_'] = 100 * F.m / F.m.sum()
    fig, ax = lienzo(alto=3.8, filas=2)
    for a, c, lab in ((ax[0], 'pn', 'procesos (%)'), (ax[1], 'pm_', 'valor contratado (%)')):
        a.bar(F.k, F[c], color=AZUL, width=0.68)
        for x, y in zip(F.k, F[c]):
            a.annotate(f'{y:.2f}', xy=(x, y), xytext=(0, 3), textcoords='offset points',
                       ha='center', color=TINTA2, fontsize=7.6)
        a.set_ylabel(lab, color=TINTA2, fontsize=8.5)
        a.grid(axis='y', color=REJILLA, linewidth=0.6); a.set_xticks(F.k)
    ax[1].set_xlabel('numero de banderas encendidas en el mismo proceso', color=TINTA2, fontsize=8.5)
    rotular(ax[0], 'El 77,9% de los procesos no enciende ninguna bandera',
            'Arriba, reparto de procesos. Abajo, reparto del valor contratado.')
    fig.tight_layout()
    guardar(fig, 'F10_banderas_simultaneas.png',
            'Procesos y valor contratado segun el numero de banderas encendidas',
            'Distribucion de los procesos y del valor contratado segun cuantas de las nueve '
            'variables del modelo situan al proceso en el 5% superior de su grupo comparable.',
            'La comparacion entre los dos paneles es el resultado relevante: los procesos con tres o '
            'mas banderas son el 0,65% del total pero concentran el 2,8% del valor, y su mediana de '
            'monto es casi cuatro veces la de los procesos sin ninguna bandera. Ningun proceso '
            'enciende mas de cinco. Hay que advertir que una de las nueve variables no puede '
            'encenderse nunca con el umbral de 0,95 por efecto de los empates, de modo que las '
            'variables operativas son ocho.',
            'documentacion/estadisticas/banderas_simultaneas.csv', 'Bloque 42')
except Exception as e: falla('F10', e)

# ---------- F11 · retencion local ----------
try:
    P = con.sql("""SELECT provincia_entidad AS prov, sum(monto_final) AS m,
         100.0*sum(CASE WHEN provincia_proveedor = provincia_entidad THEN monto_final ELSE 0 END)
           /sum(monto_final) AS local
       FROM pm WHERE provincia_entidad IS NOT NULL AND provincia_proveedor IS NOT NULL
       GROUP BY 1 ORDER BY m DESC LIMIT 15""").df().sort_values('local')
    nac = con.sql("""SELECT 100.0*sum(CASE WHEN provincia_proveedor = provincia_entidad
         THEN monto_final ELSE 0 END)/sum(monto_final) FROM pm
       WHERE provincia_entidad IS NOT NULL AND provincia_proveedor IS NOT NULL""").fetchone()[0]
    fig, ax = lienzo(alto=4.8)
    y = np.arange(len(P))
    ax.barh(y, P.local, color=AZUL, height=0.65)
    ax.axvline(nac, color=NARANJA, linewidth=1.3)
    ax.annotate(f'promedio nacional {nac:.1f}%', xy=(nac, len(P)-0.4), xytext=(5, 0),
                textcoords='offset points', color=NARANJA, fontsize=8.5)
    for i, v in enumerate(P.local):
        ax.annotate(f'{v:.1f}%', xy=(v, i), xytext=(4, 0), textcoords='offset points',
                    va='center', color=TINTA, fontsize=8)
    ax.set_yticks(y); ax.set_yticklabels([s.title() for s in P.prov], fontsize=8.5, color=TINTA2)
    ax.set_xlabel('valor contratado que queda en proveedores de la misma provincia (%)',
                  color=TINTA2, fontsize=8.5)
    ax.set_xlim(0, 82); ax.grid(axis='x', color=REJILLA, linewidth=0.6)
    rotular(ax, 'Pichincha y Guayas retienen el 68% y el 70%; el resto, entre el 37% y el 61%',
            'Las 15 provincias con mayor valor contratado por sus entidades.')
    guardar(fig, 'F11_retencion_local.png',
            'Retencion provincial del valor contratado',
            'Porcentaje del valor contratado por las entidades de cada provincia que se adjudica a '
            'proveedores domiciliados en la misma provincia.',
            'Las dos provincias con mayor actividad retienen cerca de siete decimas partes de su '
            'contratacion, mientras provincias como Santo Domingo de los Tsachilas retienen el '
            '36,88% y Loja el 43,00%. En el agregado, el 62,59% del valor contratado del pais se '
            'adjudica a proveedores de Pichincha y Guayas, y la proporcion de procesos adjudicados '
            'fuera de la provincia de la entidad crecio del 38,25% en 2015 al 42,96% en 2024.',
            'documentacion/estadisticas/retencion_local_por_provincia.csv', 'Bloque 43')
except Exception as e: falla('F11', e)

# ---------- F12 · validacion con el SRI ----------
try:
    kv = next((c for c in con.sql("DESCRIBE SELECT * FROM val").df().column_name if c.lower() == 'ruc'), 'ruc')
    pz = [f"""SELECT '{v.replace('pct_','').replace('_',' ')}' AS variable,
          median(CASE WHEN marc THEN "{v}" END) AS marcados,
          median(CASE WHEN NOT marc THEN "{v}" END) AS resto FROM
          (SELECT *, proveedor_ruc IN (SELECT "{kv}" FROM val) AS marc FROM ind)""" for v in V9]
    S = con.sql('\nUNION ALL\n'.join(pz)).df().dropna()
    S = S.sort_values('marcados')
    fig, ax = lienzo(alto=4.4)
    y = np.arange(len(S))
    ax.hlines(y, S.resto, S.marcados, color=EJE, linewidth=1.6, zorder=1)
    ax.scatter(S.resto, y, s=58, color=AZUL, zorder=3, label='resto de la base (446.569 procesos)')
    ax.scatter(S.marcados, y, s=58, color=NARANJA, marker='D', zorder=3,
               label='proveedores marcados por el SRI (108 procesos)')
    ax.axvline(0.5, color=TINTA2, linewidth=0.9)
    ax.annotate('mediana teorica\nde un rango uniforme', xy=(0.5, len(S)-0.5), xytext=(5, 0),
                textcoords='offset points', color=TINTA2, fontsize=8)
    ax.set_yticks(y); ax.set_yticklabels(S.variable, fontsize=8.5, color=TINTA2)
    ax.set_xlabel('mediana del rango percentil', color=TINTA2, fontsize=8.5)
    ax.set_xlim(-0.03, 1.0); ax.grid(axis='x', color=REJILLA, linewidth=0.6)
    ax.legend(frameon=False, loc='lower right', fontsize=8, labelcolor=TINTA2)
    rotular(ax, 'Los proveedores marcados por el SRI puntuan mas alto en todas las variables medibles',
            'Mediana de cada variable del modelo. Circulo: resto. Rombo: marcados.')
    guardar(fig, 'F12_validacion_sri.png',
            'Validacion externa: proveedores marcados por el SRI frente al resto de la base',
            'Mediana de cada variable del modelo para los 108 procesos de los quince contribuyentes '
            'con marca verificada en el Servicio de Rentas Internas, frente a los 446.569 restantes.',
            'El desplazamiento es consistente y en la misma direccion en las seis variables con '
            'dispersion suficiente: antiguedad del proveedor, monto sobre capital, nivel de precio, '
            'concurrencia, rapidez de firma y plazo de ofertas. El resultado relevante es que ese '
            'desplazamiento ocurre en la zona media de la distribucion y no en los extremos: los '
            'procesos de estos proveedores encienden menos banderas que el resto (15,74% frente a '
            '22,12% con al menos una), de modo que un detector basado en umbrales los pasaria por '
            'alto mientras un puntaje agregado continuo los captaria. Quince contribuyentes no '
            'permiten inferencia estadistica; se trata de corroboracion externa cualitativa.',
            'documentacion/estadisticas/validacion_sri_contra_resto.csv', 'Bloque 44')
except Exception as e: falla('F12', e)

# ---------- catalogo ----------
print('\n' + '=' * 95); print('CATALOGO'); print('=' * 95)
L = ['# Catalogo de figuras - Tesis SERCOP', '',
     f'Generado el {pd.Timestamp.today().strftime("%Y-%m-%d %H:%M")} desde los datos definitivos.',
     'Carpeta: `plan_A\\graficos\\`. Todas las figuras son PNG a 300 ppp, listas para insertar en',
     'Word a ancho de pagina. Los datos de cada figura estan en los CSV que se indican, de modo que',
     'cualquier cifra del grafico se puede verificar en tabla.', '',
     '## Convenciones graficas', '',
     '- Paleta validada para distincion con deficiencias de vision del color (ΔE 9,2 en el par mas',
     '  cercano). Donde hay dos series se distinguen tambien por forma (circulo y rombo) para que la',
     '  figura siga siendo legible impresa en blanco y negro.',
     '- Ninguna figura usa dos ejes verticales: cuando hay dos medidas se usan dos paneles.',
     '- Una sola tinta cuando hay una sola serie; escala de un solo tono para magnitud; azul-rojo con',
     '  gris al centro para la matriz de correlacion.',
     '- Trama diagonal: anio incompleto por retraso de publicacion.',
     '- Las cifras con cola larga se reportan con mediana, nunca con promedio.', '']
for c in CATALOGO:
    L += [f'## {c["archivo"]}', '', f'**{c["titulo"]}**', '',
          f'*Que muestra.* {c["que_muestra"]}', '',
          f'*Como leerla.* {c["como_leerla"]}', '',
          f'*Datos de origen.* `{c["datos_de_origen"]}`  ·  *Bloque de la tesis.* {c["bloque"]}', '']
open(f'{GRA}/CATALOGO_DE_GRAFICOS.md', 'w', encoding='utf-8').write('\n'.join(L))
pd.DataFrame(CATALOGO).to_csv(f'{GRA}/catalogo.csv', index=False, encoding='utf-8-sig')
print(f'  {len(CATALOGO)} figuras generadas en {GRA}')
print(f'  catalogo: CATALOGO_DE_GRAFICOS.md ({len(L)} lineas) y catalogo.csv')
for f in sorted(os.listdir(GRA)): print('   ', f)
print('\nPega aqui la salida. Y mandame dos o tres PNG para que los revise con mis propios ojos.')

In [ ]:
# ============ CELDA 43 · FIGURAS CORREGIDAS Y LAS ESTADISTICAS QUE FALTABAN ============
import os, math, itertools, collections, warnings
import pandas as pd, numpy as np, duckdb
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

os.chdir(r'C:\SERCOP_FINAL')
GRA, EST = 'plan_A/graficos', 'plan_A/documentacion/estadisticas'
os.makedirs(GRA, exist_ok=True); os.makedirs(EST, exist_ok=True)
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 80); pd.set_option('display.max_rows', 200)
con = duckdb.connect(); con.sql("SET preserve_insertion_order=false")
con.sql("CREATE OR REPLACE VIEW pm  AS SELECT * FROM read_parquet('plan_A/planA_procesos_modelo.parquet')")
con.sql("CREATE OR REPLACE VIEW ind AS SELECT * FROM read_parquet('plan_A/planA_indicadores.parquet')")
def t(x): print('\n' + '=' * 95); print(x); print('=' * 95)
def g(df, n):
    df.to_csv(f'{EST}/{n}.csv', index=False, encoding='utf-8-sig'); return df

AZUL, NARANJA = '#2a78d6', '#eb6834'
SURF, TINTA, TINTA2, MUDO, REJILLA, EJE = '#fcfcfb', '#0b0b0b', '#52514e', '#898781', '#e1e0d9', '#c3c2b7'
plt.rcParams.update({'font.family': ['Segoe UI', 'DejaVu Sans', 'sans-serif'], 'font.size': 9})
def lienzo(alto=4.2, ancho=7.2, filas=1):
    fig, ax = plt.subplots(filas, 1, figsize=(ancho, alto), facecolor=SURF)
    fig.patch.set_facecolor(SURF)
    for a in (ax if filas > 1 else [ax]):
        a.set_facecolor(SURF)
        a.spines['top'].set_visible(False); a.spines['right'].set_visible(False)
        a.spines['left'].set_color(EJE); a.spines['bottom'].set_color(EJE)
        a.spines['left'].set_linewidth(0.8); a.spines['bottom'].set_linewidth(0.8)
        a.tick_params(colors=MUDO, labelsize=8, length=3); a.set_axisbelow(True)
    return fig, ax
def rotular(ax, titulo, sub=None):
    ax.set_title(titulo, color=TINTA, loc='left', pad=14 if sub else 8, fontweight='semibold')
    if sub: ax.text(0, 1.015, sub, transform=ax.transAxes, color=TINTA2, fontsize=8.5, va='bottom')

V9 = ['pct_precio_alto','pct_cuota_12m','pct_recurrencia_par','pct_monto_capital',
      'pct_fraccionamiento','pct_proveedor_nuevo','pct_poca_concurrencia','pct_plazo_corto',
      'pct_firma_rapida']

# ================= (A) LAS TRES FIGURAS CORREGIDAS =================
t('(A) FIGURAS CORREGIDAS')
# --- F10: faltaba aire arriba, la etiqueta se encimaba con el subtitulo ---
sa = ' + '.join(f'CASE WHEN "{v}" >= 0.95 THEN 1 ELSE 0 END' for v in V9)
F = con.sql(f"SELECT ({sa}) AS k, count(*) AS n, sum(monto_final) AS m FROM ind GROUP BY 1 ORDER BY 1").df()
F['pn'] = 100*F.n/F.n.sum(); F['pm_'] = 100*F.m/F.m.sum()
fig, ax = lienzo(alto=4.1, filas=2)
for a, c, lab in ((ax[0], 'pn', 'procesos (%)'), (ax[1], 'pm_', 'valor contratado (%)')):
    a.bar(F.k, F[c], color=AZUL, width=0.68)
    for x, y in zip(F.k, F[c]):
        a.annotate(f'{y:.2f}', xy=(x, y), xytext=(0, 3), textcoords='offset points',
                   ha='center', color=TINTA2, fontsize=7.6)
    a.set_ylabel(lab, color=TINTA2, fontsize=8.5)
    a.grid(axis='y', color=REJILLA, linewidth=0.6); a.set_xticks(F.k)
    a.set_ylim(0, F[c].max()*1.22)          # <-- el arreglo
ax[1].set_xlabel('numero de banderas encendidas en el mismo proceso', color=TINTA2, fontsize=8.5)
rotular(ax[0], 'El 77,9% de los procesos no enciende ninguna bandera',
        'Arriba, reparto de procesos. Abajo, reparto del valor contratado.')
fig.tight_layout(); fig.savefig(f'{GRA}/F10_banderas_simultaneas.png', dpi=300, bbox_inches='tight', facecolor=SURF)
plt.close(fig); print('  OK  F10 (mas aire sobre las barras)')

# --- F11: el titulo prometia un rango que la figura no mostraba ---
P = con.sql("""SELECT provincia_entidad AS prov, sum(monto_final) AS m,
     100.0*sum(CASE WHEN provincia_proveedor = provincia_entidad THEN monto_final ELSE 0 END)
       /sum(monto_final) AS local
   FROM pm WHERE provincia_entidad IS NOT NULL AND provincia_proveedor IS NOT NULL
   GROUP BY 1 ORDER BY m DESC LIMIT 15""").df().sort_values('local')
nac = con.sql("""SELECT 100.0*sum(CASE WHEN provincia_proveedor = provincia_entidad
     THEN monto_final ELSE 0 END)/sum(monto_final) FROM pm
   WHERE provincia_entidad IS NOT NULL AND provincia_proveedor IS NOT NULL""").fetchone()[0]
fig, ax = lienzo(alto=4.8)
y = np.arange(len(P))
ax.barh(y, P.local, color=AZUL, height=0.65)
ax.axvline(nac, color=NARANJA, linewidth=1.3)
ax.annotate(f'promedio nacional {nac:.1f}%', xy=(nac, len(P)-0.4), xytext=(5, 0),
            textcoords='offset points', color=NARANJA, fontsize=8.5)
for i, v in enumerate(P.local):
    ax.annotate(f'{v:.1f}%', xy=(v, i), xytext=(4, 0), textcoords='offset points',
                va='center', color=TINTA, fontsize=8)
ax.set_yticks(y); ax.set_yticklabels([s.title() for s in P.prov], fontsize=8.5, color=TINTA2)
ax.set_xlabel('valor contratado que queda en proveedores de la misma provincia (%)', color=TINTA2, fontsize=8.5)
ax.set_xlim(0, 82); ax.grid(axis='x', color=REJILLA, linewidth=0.6)
rotular(ax, f'Guayas y Pichincha retienen el {P.local.max():.0f}% y el 68%; Loja, el {P.local.min():.0f}%',
        'Las 15 provincias con mayor valor contratado por sus entidades.')
fig.savefig(f'{GRA}/F11_retencion_local.png', dpi=300, bbox_inches='tight', facecolor=SURF)
plt.close(fig); print('  OK  F11 (titulo ajustado a lo que la figura muestra)')

# --- F12: el titulo decia "todas" y tres variables no tienen diferencia que mostrar ---
kv = 'ruc'
con.sql("CREATE OR REPLACE VIEW val AS SELECT * FROM read_parquet('plan_A/planA_validacion_sri.parquet')")
pz = [f"""SELECT '{v.replace('pct_','').replace('_',' ')}' AS variable,
      median(CASE WHEN marc THEN "{v}" END) AS marcados,
      median(CASE WHEN NOT marc THEN "{v}" END) AS resto FROM
      (SELECT *, proveedor_ruc IN (SELECT "{kv}" FROM val) AS marc FROM ind)""" for v in V9]
S = con.sql('\nUNION ALL\n'.join(pz)).df().dropna()
S['dif'] = (S.marcados - S.resto).abs()
S_si = S[S.dif > 1e-9].sort_values('marcados')          # solo donde hay diferencia
S_no = S[S.dif <= 1e-9]
fig, ax = lienzo(alto=3.9)
y = np.arange(len(S_si))
ax.hlines(y, S_si.resto, S_si.marcados, color=EJE, linewidth=1.6, zorder=1)
ax.scatter(S_si.resto, y, s=58, color=AZUL, zorder=3, label='resto de la base (446.569 procesos)')
ax.scatter(S_si.marcados, y, s=58, color=NARANJA, marker='D', zorder=3,
           label='marcados por el SRI (108 procesos)')
ax.axvline(0.5, color=TINTA2, linewidth=0.9)
ax.annotate('mediana de un rango uniforme', xy=(0.5, len(S_si)-0.45), xytext=(5, 0),
            textcoords='offset points', color=TINTA2, fontsize=8)
ax.set_yticks(y); ax.set_yticklabels(S_si.variable, fontsize=8.5, color=TINTA2)
ax.set_xlabel('mediana del rango percentil', color=TINTA2, fontsize=8.5)
ax.set_xlim(0.20, 0.85); ax.grid(axis='x', color=REJILLA, linewidth=0.6)
ax.legend(frameon=False, loc='lower right', fontsize=8, labelcolor=TINTA2)
rotular(ax, f'Los marcados por el SRI puntuan mas alto en las {len(S_si)} variables con dispersion',
        'Mediana de cada variable. Las otras tres valen cero en ambos grupos y se omiten.')
fig.savefig(f'{GRA}/F12_validacion_sri.png', dpi=300, bbox_inches='tight', facecolor=SURF)
plt.close(fig)
print(f'  OK  F12 (se muestran {len(S_si)} variables; se omiten {len(S_no)} sin diferencia: '
      f'{", ".join(S_no.variable)})')

# ================= (B) QUE COMPRA EL ESTADO =================
t('(B) COMPOSICION DE LA CONTRATACION POR CLASIFICACION CPC')
SEC = {'0':'0 Agricultura, silvicultura y pesca','1':'1 Minerales, electricidad, gas y agua',
       '2':'2 Alimentos, bebidas, tabaco, textiles y cuero','3':'3 Otros bienes transportables',
       '4':'4 Productos metalicos, maquinaria y equipo','5':'5 Construcciones y servicios de construccion',
       '6':'6 Comercio, alojamiento, transporte y distribucion','7':'7 Servicios financieros, inmobiliarios y de alquiler',
       '8':'8 Servicios empresariales y de produccion','9':'9 Servicios comunitarios, sociales y personales'}
SEC_SQL = ' '.join(f"WHEN '{k}' THEN '{v}'" for k, v in SEC.items())
print(g(con.sql(f"""SELECT CASE substr(cpc5,1,1) {SEC_SQL} ELSE 'sin clasificar' END AS seccion_cpc,
     count(*) AS procesos, round(100.0*count(*)/sum(count(*)) OVER (),2) AS pct_procesos,
     round(sum(monto_final)/1e6,1) AS millones,
     round(100.0*sum(monto_final)/sum(sum(monto_final)) OVER (),2) AS pct_monto,
     round(median(monto_final),0) AS mediana_usd,
     round(100.0*count(*) FILTER (WHERE ind_oferente_unico)/count(*),2) AS pct_oferente_unico
   FROM ind GROUP BY 1 ORDER BY millones DESC""").df(), 'composicion_por_seccion_cpc').to_string(index=False))
print('\nOJO: los nombres de seccion son los del estandar CPC version 2. La base guarda solo el codigo,')
print('no la descripcion, asi que conviene confirmarlos contra el clasificador que publica el SERCOP.')
print('\n-- los 15 codigos CPC5 con mas valor contratado --')
print(g(con.sql("""SELECT cpc5, count(*) AS procesos, round(sum(monto_final)/1e6,1) AS millones,
     round(100.0*sum(monto_final)/(SELECT sum(monto_final) FROM ind),2) AS pct_monto,
     round(median(monto_final),0) AS mediana_usd, count(DISTINCT proveedor_ruc) AS proveedores,
     round(100.0*count(*) FILTER (WHERE ind_oferente_unico)/count(*),2) AS pct_oferente_unico
   FROM ind WHERE cpc5 IS NOT NULL GROUP BY 1 ORDER BY millones DESC LIMIT 15""").df(),
   'top15_cpc5').to_string(index=False))

# ================= (C) LAS MAYORES ENTIDADES CONTRATANTES =================
t('(C) LAS 20 ENTIDADES CON MAS VALOR CONTRATADO')
print(g(con.sql("""SELECT pm.entidad_ruc, any_value(pm.entidad_nombre) AS entidad,
     count(*) AS procesos, count(DISTINCT pm.proveedor_ruc_final) AS proveedores,
     round(sum(pm.monto_final)/1e6,1) AS millones,
     round(100.0*sum(pm.monto_final)/(SELECT sum(monto_final) FROM pm),2) AS pct_del_total,
     round(median(pm.monto_final),0) AS mediana_usd,
     round(100.0*count(*) FILTER (WHERE ind.ind_oferente_unico)/count(*),2) AS pct_oferente_unico
   FROM pm JOIN ind USING (ocid) GROUP BY 1 ORDER BY millones DESC LIMIT 20""").df(),
   'top20_entidades').to_string(index=False))

# ================= (D) TIPO INSTITUCIONAL DE LA ENTIDAD =================
t('(D) CLASIFICACION DE ENTIDADES POR TIPO INSTITUCIONAL (heuristica sobre el nombre)')
TIPO = """CASE
 WHEN n LIKE '%HOSPITAL%' OR n LIKE '%CENTRO DE SALUD%' OR n LIKE '%INSTITUTO DEL CANCER%' THEN 'Salud'
 WHEN n LIKE '%UNIVERSIDAD%' OR n LIKE '%POLITECNICA%' OR n LIKE '%ESCUELA SUPERIOR%' THEN 'Educacion superior'
 WHEN n LIKE '%MINISTERIO%' OR n LIKE '%SECRETARIA NACIONAL%' OR n LIKE '%VICEPRESIDENCIA%' THEN 'Gobierno central'
 WHEN n LIKE '%FUERZA AEREA%' OR n LIKE '%EJERCITO%' OR n LIKE '%ARMADA%' OR n LIKE '%BRIGADA%'
   OR n LIKE '%COMANDO%' OR n LIKE '%POLICIA%' OR n LIKE '%FAE %' OR n LIKE '%MILITAR%' THEN 'Fuerzas armadas y policia'
 WHEN n LIKE '%GOBIERNO AUTONOMO%' OR n LIKE '%GAD %' OR n LIKE '%MUNICIP%' OR n LIKE '%PREFECTURA%'
   OR n LIKE '%CONSEJO PROVINCIAL%' OR n LIKE '%GOBIERNO PROVINCIAL%' OR n LIKE '%JUNTA PARROQUIAL%' THEN 'Gobierno local'
 WHEN n LIKE '%EMPRESA PUBLICA%' OR n LIKE '% EP%' OR n LIKE '%CORPORACION NACIONAL%' THEN 'Empresa publica'
 WHEN n LIKE '%UNIDAD EDUCATIVA%' OR n LIKE '%COLEGIO%' OR n LIKE '%DISTRITO%EDUCA%' THEN 'Educacion'
 ELSE 'Otros' END"""
con.sql(f"""CREATE OR REPLACE TABLE tipo AS
  SELECT pm.ocid, pm.entidad_ruc, pm.monto_final, ind.ind_oferente_unico AS unico,
         ind.ind_fraccionamiento_marca AS frac,
         {TIPO.replace('n ', 'x.n ')} AS tipo
  FROM pm JOIN ind USING (ocid)
  JOIN (SELECT entidad_ruc AS r, upper(strip_accents(any_value(entidad_nombre))) AS n
        FROM pm GROUP BY 1) x ON x.r = pm.entidad_ruc""")
print(g(con.sql("""SELECT tipo, count(DISTINCT entidad_ruc) AS entidades, count(*) AS procesos,
     round(100.0*count(*)/sum(count(*)) OVER (),2) AS pct_procesos,
     round(sum(monto_final)/1e6,1) AS millones,
     round(100.0*sum(monto_final)/sum(sum(monto_final)) OVER (),2) AS pct_monto,
     round(median(monto_final),0) AS mediana_usd,
     round(100.0*count(*) FILTER (WHERE unico)/count(*),2) AS pct_oferente_unico,
     round(100.0*count(*) FILTER (WHERE frac)/count(*),2) AS pct_con_antecedente_30d
   FROM tipo GROUP BY 1 ORDER BY millones DESC""").df(),
   'composicion_por_tipo_de_entidad').to_string(index=False))
print('\nLa clasificacion es heuristica, construida sobre el nombre de la entidad, y debe declararse')
print('como tal. Sirve para evaluar los indicadores dentro de grupos homogeneos, no como dato oficial.')

# ================= (E) REFERENCIA EXACTA DE BANDERAS BAJO INDEPENDENCIA =================
t('(E) CUANTAS BANDERAS SIMULTANEAS CABRIA ESPERAR SI FUERAN INDEPENDIENTES')
sel = ', '.join(f'("{v}" >= 0.95) AS f{i}, ("{v}" IS NOT NULL) AS m{i}' for i, v in enumerate(V9))
B = con.sql(f"SELECT {sel} FROM ind").df()
ps = np.array([B[f'f{i}'].sum() / max(B[f'm{i}'].sum(), 1) for i in range(9)])
print('tasa de encendido observada de cada variable, sobre sus filas medibles:')
for v, p in zip(V9, ps): print(f'   {v:24s} {100*p:6.3f}%')
Mb = B[[f'm{i}' for i in range(9)]].values.astype(np.int8)
code = (Mb * (1 << np.arange(9))).sum(axis=1)
dist = np.zeros(10)
for c, n in collections.Counter(code.tolist()).items():
    d = np.array([1.0])
    for i in range(9):
        if (c >> i) & 1: d = np.convolve(d, [1 - ps[i], ps[i]])
    dist[:len(d)] += n * d
dist = dist / dist.sum()
obs = B[[f'f{i}' for i in range(9)]].values.sum(axis=1)
oc = np.bincount(obs, minlength=10) / len(obs)
R = pd.DataFrame({'banderas': range(10), 'observado_pct': (100*oc).round(4),
                  'esperado_si_independientes_pct': (100*dist).round(4)})
R['veces_lo_esperado'] = (R.observado_pct / R.esperado_si_independientes_pct.replace(0, np.nan)).round(2)
print('\n' + g(R, 'banderas_contra_independencia').to_string(index=False))
print(f'\ncon 3 o mas banderas: observado {100*oc[3:].sum():.3f}%  esperado {100*dist[3:].sum():.3f}%  '
      f'-> {oc[3:].sum()/dist[3:].sum():.2f} veces lo esperable')
print('El calculo es exacto, no simulado: para cada patron de variables medibles se obtiene la')
print('distribucion de Poisson-binomial con las tasas observadas, y se promedia por el peso del patron.')

# ================= (F) CORRELACION SIN LOS CEROS =================
t('(F) CORRELACION ENTRE VARIABLES, RESTRINGIDA A LAS FILAS DONDE AMBAS SON MAYORES QUE CERO')
pz = [f"""SELECT '{a.replace('pct_','')}' AS v1, '{b.replace('pct_','')}' AS v2,
      round(corr("{a}","{b}"),3) AS r_todas,
      count(*) FILTER (WHERE "{a}" > 0 AND "{b}" > 0) AS filas_sin_ceros,
      round(corr(CASE WHEN "{a}">0 AND "{b}">0 THEN "{a}" END,
                 CASE WHEN "{a}">0 AND "{b}">0 THEN "{b}" END),3) AS r_sin_ceros
      FROM ind""" for a, b in itertools.combinations(V9, 2)]
C = con.sql('\nUNION ALL\n'.join(pz)).df()
C['cambio'] = (C.r_todas - C.r_sin_ceros).round(3)
print(g(C.sort_values('r_todas', ascending=False).head(12), 'correlacion_sin_ceros').to_string(index=False))
print('\nSi la correlacion alta se desploma al quitar los ceros compartidos, la redundancia aparente')
print('venia de la coincidencia de ceros y no de que las variables midan lo mismo.')

# ================= (G) LOS ADJUDICADOS POR ENCIMA DEL PLIEGO =================
t('(G) LOS PROCESOS ADJUDICADOS POR ENCIMA DEL PRESUPUESTO DEL PLIEGO')
con.sql("""CREATE OR REPLACE TABLE sobre AS
  SELECT pm.ocid, ind.familia, ind.anio, pm.monto_tender, pm.monto_final,
         pm.monto_final/nullif(pm.monto_tender,0) AS razon, ind.ind_oferente_unico AS unico,
         pm.entidad_nombre
  FROM pm JOIN ind USING (ocid)
  WHERE pm.monto_tender IS NOT NULL AND pm.monto_tender > 0
    AND pm.monto_final > pm.monto_tender*1.0001""")
print(con.sql("""SELECT count(*) AS procesos, round(sum(monto_final-monto_tender)/1e6,2) AS millones_de_exceso,
     round(median(razon),4) AS razon_mediana, round(max(razon),3) AS razon_maxima,
     round(100.0*count(*) FILTER (WHERE unico)/count(*),2) AS pct_oferente_unico
   FROM sobre""").df().to_string(index=False))
print('\n-- por modalidad y por año --')
print(con.sql("SELECT familia, count(*) AS procesos FROM sobre GROUP BY 1 ORDER BY procesos DESC").df().to_string(index=False))
print(con.sql("SELECT anio, count(*) AS procesos FROM sobre GROUP BY 1 ORDER BY 1").df().to_string(index=False))
print('\n-- los 12 de mayor exceso, para revisar en el portal --')
print(g(con.sql("""SELECT ocid, familia, anio, monto_tender, monto_final, round(razon,3) AS razon,
     entidad_nombre FROM sobre ORDER BY monto_final-monto_tender DESC LIMIT 12""").df(),
   'adjudicados_sobre_el_pliego').to_string(index=False))

# ================= (H) LA RAZON 0,9500 EN COTIZACION =================
t('(H) LA RAZON ADJUDICADO SOBRE PLIEGO EN COTIZACION CON COMPETENCIA')
print(g(con.sql("""SELECT round(pm.monto_final/pm.monto_tender, 3) AS razon, count(*) AS procesos,
     round(100.0*count(*)/sum(count(*)) OVER (),2) AS pct
   FROM pm JOIN ind USING (ocid)
   WHERE ind.familia = 'Cotizacion' AND NOT ind.ind_oferente_unico
     AND pm.monto_tender IS NOT NULL AND pm.monto_tender > 0
   GROUP BY 1 ORDER BY procesos DESC LIMIT 15""").df(),
   'razon_cotizacion').to_string(index=False))
print('\n-- lo mismo en licitacion con competencia, donde la mediana sale 0,9072 --')
print(con.sql("""SELECT round(pm.monto_final/pm.monto_tender, 3) AS razon, count(*) AS procesos,
     round(100.0*count(*)/sum(count(*)) OVER (),2) AS pct
   FROM pm JOIN ind USING (ocid)
   WHERE ind.familia = 'Licitacion' AND NOT ind.ind_oferente_unico
     AND pm.monto_tender IS NOT NULL AND pm.monto_tender > 0
   GROUP BY 1 ORDER BY procesos DESC LIMIT 10""").df().to_string(index=False))
print('\nSi aparece una acumulacion en un valor concreto, hay una regla de rebaja minima operando.')
print('Si la distribucion es suave y la mediana cae en 0,9500 por casualidad, no hay nada que contar.')
print('\nPega aqui toda la salida.')

In [ ]:
# ============ CELDA 43b · ARREGLOS Y LO QUE FALTA (D a H) ============
import os, itertools, collections, warnings
import pandas as pd, numpy as np, duckdb
warnings.filterwarnings('ignore')
os.chdir(r'C:\SERCOP_FINAL')
EST = 'plan_A/documentacion/estadisticas'; os.makedirs(EST, exist_ok=True)
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 80); pd.set_option('display.max_rows', 200)
con = duckdb.connect(); con.sql("SET preserve_insertion_order=false")
con.sql("CREATE OR REPLACE VIEW pm  AS SELECT * FROM read_parquet('plan_A/planA_procesos_modelo.parquet')")
con.sql("CREATE OR REPLACE VIEW ind AS SELECT * FROM read_parquet('plan_A/planA_indicadores.parquet')")
def t(x): print('\n' + '=' * 95); print(x); print('=' * 95)
def g(df, n):
    df.to_csv(f'{EST}/{n}.csv', index=False, encoding='utf-8-sig'); return df
V9 = ['pct_precio_alto','pct_cuota_12m','pct_recurrencia_par','pct_monto_capital',
      'pct_fraccionamiento','pct_proveedor_nuevo','pct_poca_concurrencia','pct_plazo_corto',
      'pct_firma_rapida']

# ---------- (D) TIPO INSTITUCIONAL, corregido y con menos "Otros" ----------
t('(D) CLASIFICACION DE ENTIDADES POR TIPO INSTITUCIONAL (heuristica sobre el nombre)')
con.sql("""CREATE OR REPLACE TABLE tipo AS
  WITH e AS (SELECT entidad_ruc AS r, upper(strip_accents(any_value(entidad_nombre))) AS n
             FROM pm GROUP BY 1)
  SELECT pm.ocid, pm.entidad_ruc, pm.monto_final,
         ind.ind_oferente_unico AS unico, ind.ind_fraccionamiento_marca AS frac,
         CASE
           WHEN e.n LIKE '%HOSPITAL%' OR e.n LIKE '%CENTRO DE SALUD%'
             OR e.n LIKE '%DISTRITO%SALUD%' OR e.n LIKE '%INSTITUTO DEL CANCER%'
             OR e.n LIKE '%SOLCA%' THEN 'Salud'
           WHEN e.n LIKE '%UNIVERSIDAD%' OR e.n LIKE '%POLITECNICA%'
             OR e.n LIKE '%ESCUELA SUPERIOR%' THEN 'Educacion superior'
           WHEN e.n LIKE '%UNIDAD EDUCATIVA%' OR e.n LIKE '%DISTRITO%EDUCA%'
             OR e.n LIKE '%COORDINACION ZONAL%EDUCA%' OR e.n LIKE '%COLEGIO%' THEN 'Educacion'
           WHEN e.n LIKE '%FUERZA AEREA%' OR e.n LIKE '%EJERCITO%' OR e.n LIKE '%ARMADA%'
             OR e.n LIKE '%BRIGADA%' OR e.n LIKE '%COMANDO%' OR e.n LIKE '%POLICIA%'
             OR e.n LIKE '%FAE %' OR e.n LIKE '%MILITAR%' OR e.n LIKE '%BOMBEROS%' THEN 'Seguridad y defensa'
           WHEN e.n LIKE '%GOBIERNO AUTONOMO%' OR e.n LIKE '%GAD %' OR e.n LIKE '%MUNICIP%'
             OR e.n LIKE '%PREFECTURA%' OR e.n LIKE '%CONSEJO PROVINCIAL%'
             OR e.n LIKE '%GOBIERNO PROVINCIAL%' OR e.n LIKE '%JUNTA PARROQUIAL%' THEN 'Gobierno local'
           WHEN e.n LIKE '%EMPRESA PUBLICA%' OR e.n LIKE '%EMPRESA ELECTRICA%'
             OR e.n LIKE '% EP%' OR e.n LIKE '%CORPORACION NACIONAL%'
             OR e.n LIKE '%PETROECUADOR%' OR e.n LIKE '%CELEC%' THEN 'Empresa publica'
           WHEN e.n LIKE '%MINISTERIO%' OR e.n LIKE '%SECRETARIA%' OR e.n LIKE '%SERVICIO NACIONAL%'
             OR e.n LIKE '%INSTITUTO NACIONAL%' OR e.n LIKE '%AGENCIA%' OR e.n LIKE '%SUPERINTENDENCIA%'
             OR e.n LIKE '%CONSEJO NACIONAL%' OR e.n LIKE '%DIRECCION NACIONAL%'
             OR e.n LIKE '%FISCALIA%' OR e.n LIKE '%DEFENSORIA%' OR e.n LIKE '%JUDICATURA%'
             OR e.n LIKE '%REGISTRO CIVIL%' OR e.n LIKE '%VICEPRESIDENCIA%'
             OR e.n LIKE '%PRESIDENCIA%' OR e.n LIKE '%SEGURIDAD SOCIAL%' THEN 'Gobierno central'
           ELSE 'Otros' END AS tipo
  FROM pm JOIN ind USING (ocid) JOIN e ON e.r = pm.entidad_ruc""")
print(g(con.sql("""SELECT tipo, count(DISTINCT entidad_ruc) AS entidades, count(*) AS procesos,
     round(100.0*count(*)/sum(count(*)) OVER (),2) AS pct_procesos,
     round(sum(monto_final)/1e6,1) AS millones,
     round(100.0*sum(monto_final)/sum(sum(monto_final)) OVER (),2) AS pct_monto,
     round(median(monto_final),0) AS mediana_usd,
     round(100.0*count(*) FILTER (WHERE unico)/count(*),2) AS pct_oferente_unico,
     round(100.0*count(*) FILTER (WHERE frac)/count(*),2) AS pct_con_antecedente_30d
   FROM tipo GROUP BY 1 ORDER BY millones DESC""").df(),
   'composicion_por_tipo_de_entidad').to_string(index=False))
print('\n-- que quedo en "Otros": las 12 entidades mas grandes sin clasificar --')
print(con.sql("""SELECT any_value(pm.entidad_nombre) AS entidad, count(*) AS procesos,
     round(sum(t.monto_final)/1e6,1) AS millones
   FROM tipo t JOIN pm USING (ocid) WHERE t.tipo = 'Otros'
   GROUP BY t.entidad_ruc ORDER BY millones DESC LIMIT 12""").df().to_string(index=False))

# ---------- (E) REFERENCIA EXACTA DE BANDERAS BAJO INDEPENDENCIA ----------
t('(E) CUANTAS BANDERAS SIMULTANEAS CABRIA ESPERAR SI FUERAN INDEPENDIENTES')
sel = ', '.join(f'coalesce("{v}" >= 0.95, false) AS f{i}, ("{v}" IS NOT NULL) AS m{i}'
                for i, v in enumerate(V9))
B = con.sql(f"SELECT {sel} FROM ind").df()
Fb = B[[f'f{i}' for i in range(9)]].astype(bool).values
Mb = B[[f'm{i}' for i in range(9)]].astype(bool).values
ps = np.array([Fb[:, i].sum() / max(Mb[:, i].sum(), 1) for i in range(9)])
print('tasa de encendido de cada variable, sobre sus filas medibles:')
for v, p in zip(V9, ps): print(f'   {v:24s} {100*p:6.3f}%')
code = (Mb.astype(np.int64) * (1 << np.arange(9))).sum(axis=1)
dist = np.zeros(10)
for c, n in collections.Counter(code.tolist()).items():
    d = np.array([1.0])
    for i in range(9):
        if (c >> i) & 1: d = np.convolve(d, [1 - ps[i], ps[i]])
    dist[:len(d)] += n * d
dist = dist / dist.sum()
oc = np.bincount(Fb.sum(axis=1).astype(np.int64), minlength=10) / len(Fb)
R = pd.DataFrame({'banderas': range(10), 'observado_pct': (100*oc).round(4),
                  'esperado_si_independientes_pct': (100*dist).round(4)})
R['veces_lo_esperado'] = (R.observado_pct / R.esperado_si_independientes_pct.replace(0, np.nan)).round(2)
print('\n' + g(R, 'banderas_contra_independencia').to_string(index=False))
print(f'\ncon 3 o mas banderas: observado {100*oc[3:].sum():.3f}%   '
      f'esperado {100*dist[3:].sum():.3f}%   -> {oc[3:].sum()/dist[3:].sum():.2f} veces lo esperable')
print('Calculo exacto, no simulado: para cada patron de variables medibles se obtiene la distribucion')
print('de Poisson-binomial con las tasas observadas y se promedia por el peso del patron.')

# ---------- (F) CORRELACION SIN LOS CEROS ----------
t('(F) CORRELACION ENTRE VARIABLES, SOLO DONDE AMBAS SON MAYORES QUE CERO')
pz = [f"""SELECT '{a.replace('pct_','')}' AS v1, '{b.replace('pct_','')}' AS v2,
      round(corr("{a}","{b}"),3) AS r_todas,
      count(*) FILTER (WHERE "{a}" > 0 AND "{b}" > 0) AS filas_sin_ceros,
      round(corr(CASE WHEN "{a}">0 AND "{b}">0 THEN "{a}" END,
                 CASE WHEN "{a}">0 AND "{b}">0 THEN "{b}" END),3) AS r_sin_ceros
      FROM ind""" for a, b in itertools.combinations(V9, 2)]
C = con.sql('\nUNION ALL\n'.join(pz)).df()
C['cae'] = (C.r_todas - C.r_sin_ceros).round(3)
print(g(C.sort_values('r_todas', ascending=False).head(12), 'correlacion_sin_ceros').to_string(index=False))
print('\nSi la correlacion se desploma al quitar los ceros compartidos, la redundancia aparente venia')
print('de que ambas variables valen cero en los mismos procesos y no de que midan lo mismo.')

# ---------- (G) ADJUDICADOS POR ENCIMA DEL PLIEGO ----------
t('(G) LOS PROCESOS ADJUDICADOS POR ENCIMA DEL PRESUPUESTO DEL PLIEGO')
con.sql("""CREATE OR REPLACE TABLE sobre AS
  SELECT pm.ocid, ind.familia, ind.anio, pm.monto_tender, pm.monto_final,
         pm.monto_final/nullif(pm.monto_tender,0) AS razon,
         ind.ind_oferente_unico AS unico, pm.entidad_nombre
  FROM pm JOIN ind USING (ocid)
  WHERE pm.monto_tender IS NOT NULL AND pm.monto_tender > 0
    AND pm.monto_final > pm.monto_tender*1.0001""")
print(con.sql("""SELECT count(*) AS procesos,
     round(sum(monto_final-monto_tender)/1e6,2) AS millones_de_exceso,
     round(median(razon),4) AS razon_mediana, round(max(razon),3) AS razon_maxima,
     round(100.0*count(*) FILTER (WHERE unico)/count(*),2) AS pct_oferente_unico FROM sobre""").df().to_string(index=False))
print('\n-- por modalidad --')
print(con.sql("SELECT familia, count(*) AS procesos FROM sobre GROUP BY 1 ORDER BY procesos DESC").df().to_string(index=False))
print('\n-- por anio --')
print(con.sql("SELECT anio, count(*) AS procesos FROM sobre GROUP BY 1 ORDER BY 1").df().to_string(index=False))
print('\n-- los 12 de mayor exceso, para revisar en el portal --')
print(g(con.sql("""SELECT ocid, familia, anio, monto_tender, monto_final, round(razon,3) AS razon,
     entidad_nombre FROM sobre ORDER BY monto_final-monto_tender DESC LIMIT 12""").df(),
   'adjudicados_sobre_el_pliego').to_string(index=False))

# ---------- (H) LA RAZON 0,9500 ----------
t('(H) LA RAZON ADJUDICADO SOBRE PLIEGO EN COTIZACION CON COMPETENCIA')
print(g(con.sql("""SELECT round(pm.monto_final/pm.monto_tender, 3) AS razon, count(*) AS procesos,
     round(100.0*count(*)/sum(count(*)) OVER (),2) AS pct
   FROM pm JOIN ind USING (ocid)
   WHERE ind.familia = 'Cotizacion' AND NOT ind.ind_oferente_unico
     AND pm.monto_tender IS NOT NULL AND pm.monto_tender > 0
   GROUP BY 1 ORDER BY procesos DESC LIMIT 15""").df(), 'razon_cotizacion').to_string(index=False))
print('\n-- licitacion con competencia, donde la mediana sale 0,9072 --')
print(con.sql("""SELECT round(pm.monto_final/pm.monto_tender, 3) AS razon, count(*) AS procesos,
     round(100.0*count(*)/sum(count(*)) OVER (),2) AS pct
   FROM pm JOIN ind USING (ocid)
   WHERE ind.familia = 'Licitacion' AND NOT ind.ind_oferente_unico
     AND pm.monto_tender IS NOT NULL AND pm.monto_tender > 0
   GROUP BY 1 ORDER BY procesos DESC LIMIT 10""").df().to_string(index=False))
print('\nUna acumulacion en un valor concreto significa que hay una regla de rebaja minima operando.')
print('Una distribucion suave con la mediana en 0,9500 por casualidad no significa nada.')
print('\nPega aqui toda la salida.')

In [ ]:
# ============ CELDA 44 · REDISENO DEL CONJUNTO DE VARIABLES -> INDICADORES v6 ============
# (A) clasificacion de entidades terminada, con el residuo medido
# (B) conjunto nuevo: 4 binarias para lo discreto + 9 rangos, cada uno en su propio dominio
# (C) controles  (D) coincidencia contra independencia, separando la redundancia de construccion
# (E) escritura de v6, enmienda firmada y manifiesto
import os, json, hashlib, itertools, collections, warnings
import pandas as pd, numpy as np, duckdb
warnings.filterwarnings('ignore')
os.chdir(r'C:\SERCOP_FINAL')
DOC, HIST, EST = 'plan_A/documentacion', 'plan_A/_version_1_historico', 'plan_A/documentacion/estadisticas'
os.makedirs(HIST, exist_ok=True); os.makedirs(EST, exist_ok=True)
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 80); pd.set_option('display.max_rows', 200)
con = duckdb.connect(); con.sql("SET preserve_insertion_order=false")
con.sql("CREATE OR REPLACE VIEW pm  AS SELECT * FROM read_parquet('plan_A/planA_procesos_modelo.parquet')")
con.sql("CREATE OR REPLACE VIEW ind AS SELECT * FROM read_parquet('plan_A/planA_indicadores.parquet')")
def t(x): print('\n' + '=' * 95); print(x); print('=' * 95)
def g(df, n):
    df.to_csv(f'{EST}/{n}.csv', index=False, encoding='utf-8-sig'); return df
def sha16(p):
    h = hashlib.sha256()
    with open(p, 'rb') as fh:
        for b in iter(lambda: fh.read(1 << 20), b''): h.update(b)
    return h.hexdigest()[:16]

# ---------- (A) CLASIFICACION DE ENTIDADES, VERSION FINAL ----------
t('(A) TIPO INSTITUCIONAL: patrones ampliados con siglas y abreviaturas')
con.sql("""CREATE OR REPLACE TABLE tipo AS
  WITH e AS (SELECT entidad_ruc AS r, upper(strip_accents(any_value(entidad_nombre))) AS n
             FROM pm GROUP BY 1)
  SELECT pm.ocid, pm.entidad_ruc, pm.monto_final,
         ind.ind_oferente_unico AS unico, ind.ind_fraccionamiento_marca AS frac,
         CASE
           WHEN e.n LIKE '%HOSPITAL%' OR e.n LIKE '%CENTRO DE SALUD%' OR e.n LIKE '%DISTRITO%SALUD%'
             OR e.n LIKE '%INSTITUTO DEL CANCER%' OR e.n LIKE '%SOLCA%' OR e.n LIKE '%MATERNIDAD%'
             OR e.n LIKE '%DISPENSARIO%' THEN 'Salud'
           WHEN e.n LIKE '%UNIVERSIDAD%' OR e.n LIKE '%POLITECNICA%' OR e.n LIKE '%ESCUELA SUPERIOR%'
             OR e.n LIKE '%ESPOL%' OR e.n LIKE '%ULEAM%' OR e.n LIKE '%UTE%' OR e.n LIKE '%ESPE%'
             OR e.n LIKE '%YACHAY%' OR e.n LIKE '%IKIAM%' OR e.n LIKE '%SENESCYT%' THEN 'Educacion superior'
           WHEN e.n LIKE '%UNIDAD EDUCATIVA%' OR e.n LIKE '%DISTRITO%EDUCA%' OR e.n LIKE '%COLEGIO%'
             OR e.n LIKE '%COORDINACION ZONAL%EDUCA%' OR e.n LIKE '%INSTITUTO TECNOLOGICO%' THEN 'Educacion'
           WHEN e.n LIKE '%FUERZA%' OR e.n LIKE '%EJERCITO%' OR e.n LIKE '%ARMADA%' OR e.n LIKE '%NAVAL%'
             OR e.n LIKE '%BRIGADA%' OR e.n LIKE '%COMANDO%' OR e.n LIKE '%POLICIA%' OR e.n LIKE '%MILITAR%'
             OR e.n LIKE '%BOMBEROS%' OR e.n LIKE '%DIGMAT%' OR e.n LIKE '%ALA DE COMBATE%'
             OR e.n LIKE '%CUERPO DE INGENIEROS%' OR e.n LIKE '%TRANSITO%' THEN 'Seguridad y defensa'
           WHEN e.n LIKE '%GOBIERNO AUTONOMO%' OR e.n LIKE '%GAD %' OR e.n LIKE '%MUNICIP%'
             OR e.n LIKE '%PREFECTURA%' OR e.n LIKE '%CONSEJO PROVINCIAL%' OR e.n LIKE '%GOB. PROV%'
             OR e.n LIKE '%GOBIERNO PROVINCIAL%' OR e.n LIKE '%JUNTA PARROQUIAL%'
             OR e.n LIKE '%MANCOMUNIDAD%' OR e.n LIKE '%CONCEJO%' THEN 'Gobierno local'
           WHEN e.n LIKE '%EMPRESA PUBLICA%' OR e.n LIKE '%EMPRESA ELECTRICA%' OR e.n LIKE '% EP%'
             OR e.n LIKE '%CORPORACION NACIONAL%' OR e.n LIKE '%PETROECUADOR%' OR e.n LIKE '%CELEC%'
             OR e.n LIKE '%ELECTRO%' OR e.n LIKE '%HIDRO%' OR e.n LIKE '%BANECUADOR%'
             OR e.n LIKE '%BANCO %' OR e.n LIKE '%CORREOS%' THEN 'Empresa publica'
           WHEN e.n LIKE '%MINISTERIO%' OR e.n LIKE '%SECRETARIA%' OR e.n LIKE '%SERVICIO %'
             OR e.n LIKE '%INSTITUTO%' OR e.n LIKE '%AGENCIA%' OR e.n LIKE '%SUPERINTENDENCIA%'
             OR e.n LIKE '%CONSEJO NACIONAL%' OR e.n LIKE '%DIRECCION%' OR e.n LIKE '%FISCALIA%'
             OR e.n LIKE '%DEFENSORIA%' OR e.n LIKE '%JUDICATURA%' OR e.n LIKE '%REGISTRO CIVIL%'
             OR e.n LIKE '%PRESIDENCIA%' OR e.n LIKE '%SEGURIDAD SOCIAL%' OR e.n LIKE '%IESS%'
             OR e.n LIKE '%ISSFA%' OR e.n LIKE '%COMISION%' OR e.n LIKE '%CORTE%'
             OR e.n LIKE '%ASAMBLEA%' OR e.n LIKE '%CONTRALORIA%' THEN 'Gobierno central'
           ELSE 'Otros' END AS tipo
  FROM pm JOIN ind USING (ocid) JOIN e ON e.r = pm.entidad_ruc""")
T = g(con.sql("""SELECT tipo, count(DISTINCT entidad_ruc) AS entidades, count(*) AS procesos,
     round(100.0*count(*)/sum(count(*)) OVER (),2) AS pct_procesos,
     round(sum(monto_final)/1e6,1) AS millones,
     round(100.0*sum(monto_final)/sum(sum(monto_final)) OVER (),2) AS pct_monto,
     round(median(monto_final),0) AS mediana_usd,
     round(100.0*count(*) FILTER (WHERE unico)/count(*),2) AS pct_oferente_unico,
     round(100.0*count(*) FILTER (WHERE frac)/count(*),2) AS pct_con_antecedente_30d
   FROM tipo GROUP BY 1 ORDER BY millones DESC""").df(), 'composicion_por_tipo_de_entidad')
print(T.to_string(index=False))
r = T[T.tipo == 'Otros']
print(f"\nresiduo sin clasificar: {int(r.entidades.iloc[0])} entidades, "
      f"{float(r.pct_procesos.iloc[0])}% de los procesos, {float(r.pct_monto.iloc[0])}% del monto")
print('\n-- las 15 entidades mas grandes que siguen en "Otros": revisalas y dime si alguna va en un tipo --')
print(con.sql("""SELECT any_value(pm.entidad_nombre) AS entidad, count(*) AS procesos,
     round(sum(t.monto_final)/1e6,1) AS millones
   FROM tipo t JOIN pm USING (ocid) WHERE t.tipo = 'Otros'
   GROUP BY t.entidad_ruc ORDER BY millones DESC LIMIT 15""").df().to_string(index=False))

# ---------- (B) EL CONJUNTO NUEVO ----------
t('(B) CONJUNTO DE VARIABLES REDISENADO')
BIN = ['ind_oferente_unico', 'ind_fraccionamiento_marca',
       'ind_primera_vez_con_la_entidad', 'ind_proveedor_debutante']
# (nombre del rango, indicador, direccion, dominio en el que el rango tiene sentido)
RAN = [('pct_precio_alto',        'ind_desv_precio_unit_z',        'ASC',  None),
       ('pct_monto_capital',      'ind_monto_sobre_capital',       'ASC',  None),
       ('pct_proveedor_nuevo',    'ind_antiguedad_dias',           'DESC', None),
       ('pct_plazo_corto',        'ind_plazo_ofertas_dias',        'DESC', None),
       ('pct_firma_rapida',       'ind_plazo_adj_firma_dias',      'DESC', None),
       ('pct_concurrencia_pos',   'ind_n_oferentes',               'DESC', 'ind_n_oferentes > 1'),
       ('pct_fraccionamiento_pos','ind_fraccionamiento_monto_30d', 'ASC',  'ind_fraccionamiento_monto_30d > 0'),
       ('pct_cuota_pos',          'ind_cuota_prov_en_entidad_12m', 'ASC',  'ind_cuota_prov_en_entidad_12m > 0'),
       ('pct_recurrencia_pos',    'ind_recurrencia_par_12m',       'ASC',  'ind_recurrencia_par_12m > 0')]
VIEJAS = ['pct_poca_concurrencia', 'pct_fraccionamiento', 'pct_cuota_12m', 'pct_recurrencia_par']
print('binarias (el fenomeno discreto, tal como ya existen):')
for b in BIN: print('   ', b)
print('\nrangos (cada uno calculado SOLO dentro de su dominio):')
for n, s, d, dom in RAN: print(f'   {n:26s} <- {s:32s} {d:4s}  dominio: {dom or "todo registro con dato"}')
print('\nse retiran los cuatro rangos calculados sobre dominios con bloques de empates:', VIEJAS)

for i, (pct, src, d, dom) in enumerate(RAN):
    filtro = f'AND ({dom})' if dom else ''
    con.sql(f"""CREATE OR REPLACE TABLE w{i} AS
      WITH c AS (SELECT ocid, familia, periodo, "{src}" AS x,
                   count(*) OVER (PARTITION BY familia, periodo) AS n_fp,
                   count(*) OVER (PARTITION BY familia)          AS n_f
                 FROM ind WHERE "{src}" IS NOT NULL {filtro}),
           e AS (SELECT ocid, x, CASE WHEN n_fp >= 30 THEN familia || '|' || periodo
                                      WHEN n_f  >= 30 THEN familia ELSE 'GLOBAL' END AS grupo
                 FROM c)
      SELECT ocid, percent_rank() OVER (PARTITION BY grupo ORDER BY x {d}) AS "{pct}" FROM e""")

C = list(con.sql("DESCRIBE SELECT * FROM ind").df().column_name)
quedan = [c for c in C if c not in set(VIEJAS)]
sel = ', '.join(f'ind."{c}"' for c in quedan) + ', ' + \
      ', '.join(f'w{i}."{n}"' for i, (n, _, _, _) in enumerate(RAN))
joins = ' '.join(f'LEFT JOIN w{i} USING (ocid)' for i in range(len(RAN)))
con.sql(f"CREATE OR REPLACE TABLE v6 AS SELECT {sel} FROM ind {joins}")
print(f'\nv6: {con.sql("SELECT count(*) FROM v6").fetchone()[0]:,} filas, '
      f'{len(con.sql("DESCRIBE SELECT * FROM v6").df())} columnas')

# ---------- (C) CONTROLES ----------
t('(C) CONTROLES: uniformidad dentro del dominio, cobertura y sentido')
pz = []
for pct, src, d, dom in RAN:
    pz.append(f"""SELECT '{pct}' AS variable, '{dom or "sin restriccion"}' AS dominio,
      count("{pct}") AS con_rango,
      round(100.0*count("{pct}")/count(*),2) AS pct_cobertura,
      round(quantile_cont("{pct}",0.25),4) AS p25, round(median("{pct}"),4) AS mediana,
      round(quantile_cont("{pct}",0.75),4) AS p75, round(max("{pct}"),4) AS maximo,
      round(100.0*count(*) FILTER (WHERE "{pct}" >= 0.95)/nullif(count("{pct}"),0),2) AS pct_enciende,
      CASE WHEN ('{d}'='ASC' AND corr("{src}","{pct}")>0)
             OR ('{d}'='DESC' AND corr("{src}","{pct}")<0) THEN 'OK' ELSE 'INVERTIDA' END AS sentido
      FROM v6""")
K = con.sql('\nUNION ALL\n'.join(pz)).df()
print(g(K, 'controles_variables_v6').to_string(index=False))
print('\nLa mediana de cada rango debe quedar cerca de 0,5 y el maximo debe llegar a 1,0: eso indica')
print('que el rango puede alcanzar el umbral de alerta. Antes, concurrencia saturaba en 0,8315.')
print('\n-- las cuatro binarias --')
pz = [f"""SELECT '{b}' AS variable, count("{b}") AS con_dato,
      round(100.0*count(*) FILTER (WHERE "{b}")/count("{b}"),2) AS pct_verdadero FROM v6""" for b in BIN]
print(con.sql('\nUNION ALL\n'.join(pz)).df().to_string(index=False))

# ---------- (D) COINCIDENCIA CONTRA INDEPENDENCIA ----------
t('(D) COINCIDENCIA DE ALERTAS CONTRA INDEPENDENCIA, CON EL CONJUNTO NUEVO')
def tablero(nombres, etiqueta):
    sel = ', '.join(f'coalesce("{v}" >= 0.95, false) AS f{i}, ("{v}" IS NOT NULL) AS m{i}'
                    for i, v in enumerate(nombres))
    B = con.sql(f"SELECT {sel} FROM v6").df()
    Fb = B[[f'f{i}' for i in range(len(nombres))]].astype(bool).values
    Mb = B[[f'm{i}' for i in range(len(nombres))]].astype(bool).values
    ps = np.array([Fb[:, i].sum() / max(Mb[:, i].sum(), 1) for i in range(len(nombres))])
    code = (Mb.astype(np.int64) * (1 << np.arange(len(nombres)))).sum(axis=1)
    dist = np.zeros(len(nombres) + 1)
    for c, n in collections.Counter(code.tolist()).items():
        d = np.array([1.0])
        for i in range(len(nombres)):
            if (c >> i) & 1: d = np.convolve(d, [1 - ps[i], ps[i]])
        dist[:len(d)] += n * d
    dist /= dist.sum()
    oc = np.bincount(Fb.sum(axis=1).astype(np.int64), minlength=len(nombres) + 1) / len(Fb)
    print(f'\n{etiqueta}')
    print(f'  tasas de encendido: ' + ', '.join(f'{100*p:.2f}%' for p in ps))
    print(f'  con 3 o mas alertas: observado {100*oc[3:].sum():.3f}%  esperado {100*dist[3:].sum():.3f}%'
          f'  -> {oc[3:].sum()/max(dist[3:].sum(),1e-12):.2f} veces lo esperable')
    return oc, dist
TODOS = [n for n, _, _, _ in RAN]
tablero(TODOS, 'conjunto completo de 9 rangos')
SIN_RED = [n for n in TODOS if n not in ('pct_recurrencia_pos', 'pct_cuota_pos')]
tablero(SIN_RED, 'quitando recurrencia y cuota, los dos rangos que mas se solapan con fraccionamiento')
print('\nSi al quitar los rangos redundantes el exceso baja mucho, buena parte de la coincidencia era')
print('de construccion. Si se mantiene, hay procesos que disparan varios controles por razones reales.')

# ---------- (E) ESCRITURA Y ENMIENDA ----------
t('(E) ESCRITURA Y FIRMA')
out = 'plan_A/planA_indicadores_v6.parquet'
con.sql(f"COPY v6 TO '{out}' (FORMAT PARQUET, COMPRESSION ZSTD)")
h = sha16(out); print(f'  {out} | {round(os.path.getsize(out)/1e6,1)} MB | sha256_16 = {h}')
if os.path.exists('plan_A/planA_indicadores.parquet'):
    os.replace('plan_A/planA_indicadores.parquet', f'{HIST}/planA_indicadores_v4_3.parquet')
os.replace(out, 'plan_A/planA_indicadores.parquet')
h = sha16('plan_A/planA_indicadores.parquet')
print(f'  queda como plan_A/planA_indicadores.parquet con sha256_16 = {h}')
esp = f'{DOC}/especificacion_indicadores.json'
E = json.load(open(esp, encoding='utf-8'))
E['variables_modelo'] = {'binarias': BIN, 'rangos': [n for n, _, _, _ in RAN]}
E.setdefault('enmiendas', []).append({
    'version': 'v6', 'fecha': pd.Timestamp.today().strftime('%Y-%m-%d'),
    'motivo': 'rediseno del conjunto de variables por diagnostico de construccion, anterior a '
              'cualquier ejecucion de modelo',
    'cambios': [
        'el rango percentil no es transformacion adecuada para variables con grandes bloques de '
        'empates: el rango de concurrencia saturaba en 0,8315 y no podia alcanzar nunca el umbral '
        'de alerta de 0,95, y los de plazo y firma se encendian al 2,49% y 0,86% en lugar del 5%',
        'los fenomenos discretos pasan a representarse con las cuatro indicadoras binarias que ya '
        'existian: oferente unico, antecedente en 30 dias, primera vez con la entidad y proveedor '
        'debutante',
        'los cuatro rangos afectados se recalculan unicamente dentro del dominio donde la variable '
        'tiene variacion: concurrencia entre los procesos con dos o mas oferentes, fraccionamiento '
        'entre los que tienen monto previo, cuota y recurrencia entre los que tienen historial',
        'se retiran pct_poca_concurrencia, pct_fraccionamiento, pct_cuota_12m y pct_recurrencia_par',
        'no se elimina ninguna variable por redundancia: la correlacion de 0,974 entre cuota y '
        'recurrencia cae a 0,279 al restringir a las filas donde ambas son mayores que cero, de modo '
        'que era un artefacto de la coincidencia de ceros; la mayor correlacion real es 0,472'],
    'sha256_16_indicadores_v6': h})
json.dump(E, open(esp, 'w', encoding='utf-8'), ensure_ascii=False, indent=2)
M = []
for f in sorted(os.listdir('plan_A')):
    if f.endswith('.parquet'):
        p = f'plan_A/{f}'
        M.append({'archivo': f, 'filas': con.sql(f"SELECT count(*) FROM read_parquet('{p}')").fetchone()[0],
                  'columnas': len(con.sql(f"DESCRIBE SELECT * FROM read_parquet('{p}')").df()),
                  'MB': round(os.path.getsize(p)/1e6,1), 'sha256_16': sha16(p)})
MAN = pd.DataFrame(M); MAN.to_csv(f'{DOC}/manifiesto.csv', index=False, encoding='utf-8-sig')
print('\n' + MAN.to_string(index=False))
print('\nPega aqui toda la salida.')

In [ ]:
# ===========================================================================
# CELDA 45 — correccion de fondo de los rangos: cume_dist + conversion de los
#            conteos a diferencia contra la mediana del grupo.  -> v6.1
# ===========================================================================
import duckdb, pandas as pd, numpy as np, hashlib, datetime, pathlib, shutil

BASE = pathlib.Path(r'C:\SERCOP_FINAL')
PA   = BASE / 'plan_A'
DOC  = PA / 'documentacion';  DOC.mkdir(exist_ok=True)
con  = duckdb.connect()
pd.set_option('display.width', 230); pd.set_option('display.max_columns', 60)

IND = (PA / 'planA_indicadores.parquet').as_posix()
con.execute(f"CREATE VIEW ind6 AS SELECT * FROM read_parquet('{IND}')")
n_total = con.execute("SELECT count(*) FROM ind6").fetchone()[0]

# alias | nombre del rango | fuente | orden | extremo anomalo | filtro de dominio
CFG = [
 ('precio',       'pct_precio_alto',         'ind_desv_precio_unit_z',        'ASC', 'max', None),
 ('capital',      'pct_monto_capital',       'ind_monto_sobre_capital',       'ASC', 'max', None),
 ('antiguedad',   'pct_proveedor_nuevo',     'ind_antiguedad_dias',           'DESC','min', None),
 ('plazo',        'pct_plazo_corto',         'ind_plazo_ofertas_dias',        'DESC','min', None),
 ('firma',        'pct_firma_rapida',        'ind_plazo_adj_firma_dias',      'DESC','min', None),
 ('concurrencia', 'pct_concurrencia_pos',    'ind_n_oferentes',               'DESC','min', 'ind_n_oferentes > 1'),
 ('fracciona',    'pct_fraccionamiento_pos', 'ind_fraccionamiento_monto_30d', 'ASC', 'max', 'ind_fraccionamiento_monto_30d > 0'),
 ('cuota',        'pct_cuota_pos',           'ind_cuota_prov_en_entidad_12m', 'ASC', 'max', 'ind_cuota_prov_en_entidad_12m > 0'),
 ('recurrencia',  'pct_recurrencia_pos',     'ind_recurrencia_par_12m',       'ASC', 'max', 'ind_recurrencia_par_12m > 0'),
]

# el grupo comparable es el mismo de siempre: familia x periodo, con respaldo a
# familia y luego global cuando el grupo no llega a 30 casos
def dominio_sql(src, filtro):
    f = f'AND {filtro}' if filtro else ''
    return f"""
      SELECT ocid, x,
             CASE WHEN n_fp >= 30 THEN familia || '|' || periodo
                  WHEN n_f  >= 30 THEN familia
                  ELSE 'GLOBAL' END AS grupo
      FROM (
        SELECT ocid, familia, periodo, "{src}" AS x,
               count(*) OVER (PARTITION BY familia, periodo) AS n_fp,
               count(*) OVER (PARTITION BY familia)          AS n_f
        FROM ind6 WHERE "{src}" IS NOT NULL {f}
      )
    """

print('='*105)
print('(A) DIAGNOSTICO: cuanta masa esta atrapada en el valor MAS ANOMALO de su grupo')
print('='*105)

diag = []
for alias, pct, src, orden, ext, filtro in CFG:
    agg = 'max' if ext == 'max' else 'min'
    q = f"""
    WITH e AS ({dominio_sql(src, filtro)}),
         g AS (SELECT grupo, {agg}(x) AS xe FROM e GROUP BY grupo),
         r AS (SELECT (e.x = g.xe) AS en_extremo, e.x AS x,
                      percent_rank() OVER (PARTITION BY e.grupo ORDER BY e.x {orden}) AS pr,
                      cume_dist()    OVER (PARTITION BY e.grupo ORDER BY e.x {orden}) AS cd
               FROM e JOIN g ON g.grupo = e.grupo)
    SELECT count(*) AS filas, count(DISTINCT x) AS valores,
           avg(CASE WHEN en_extremo THEN 1.0 ELSE 0.0 END) AS masa,
           max(pr) AS max_pr, max(cd) AS max_cd,
           avg(CASE WHEN pr >= 0.95 THEN 1.0 ELSE 0.0 END) AS enc_pr,
           avg(CASE WHEN cd >= 0.95 THEN 1.0 ELSE 0.0 END) AS enc_cd
    FROM r
    """
    r = con.execute(q).df().iloc[0]
    diag.append(dict(variable=pct, filas=int(r.filas), valores_distintos=int(r.valores),
                     masa_valor_extremo=round(100*r.masa, 2),
                     max_pct_rank=round(r.max_pr, 4), max_cume_dist=round(r.max_cd, 4),
                     enc_pct_rank=round(100*r.enc_pr, 2), enc_cume_dist=round(100*r.enc_cd, 2),
                     decision='rango' if r.masa <= 0.05 else 'CONVERTIR'))
diag = pd.DataFrame(diag)
print(diag.to_string(index=False))
print("""
masa_valor_extremo: % del dominio que comparte el valor mas anomalo de su grupo.
Si pasa de 5%, no existe ningun percentil que deje al 5% mas extremo por encima de
0,95: la variable no tiene la resolucion que un rango finge tener, y se convierte a
diferencia contra la mediana del grupo. Si no pasa de 5%, basta cambiar percent_rank
por cume_dist, que es la funcion de distribucion empirica y asigna al bloque de
empates el percentil de su ULTIMO miembro en lugar del primero.""")

APTAS   = [c for c in CFG if diag.loc[diag.variable == c[1], 'decision'].iloc[0] == 'rango']
CONVERT = [c for c in CFG if diag.loc[diag.variable == c[1], 'decision'].iloc[0] == 'CONVERTIR']

# ---------------------------------------------------------------------------
# (B) RECONSTRUCCION
# ---------------------------------------------------------------------------
print('\n' + '='*105)
print('(B) RECONSTRUCCION DEL CONJUNTO  -> v6.1')
print('='*105)

d = con.execute("SELECT * FROM ind6").df()
fuera = [c[1] for c in CFG] + ['pct_poca_concurrencia', 'pct_fraccionamiento',
                               'pct_cuota_12m', 'pct_recurrencia_par']
d = d.drop(columns=[c for c in fuera if c in d.columns])

rangos_finales = []
for alias, pct, src, orden, ext, filtro in APTAS:
    q = (f'WITH e AS ({dominio_sql(src, filtro)}) '
         f'SELECT ocid, cume_dist() OVER (PARTITION BY grupo ORDER BY x {orden}) AS "{pct}" FROM e')
    d = d.merge(con.execute(q).df(), on='ocid', how='left')
    rangos_finales.append(pct)
    print(f'  rango  {pct:<26s} <- {src:<32s} {orden:<4s} cume_dist')

difs_finales = []
for alias, pct, src, orden, ext, filtro in CONVERT:
    nombre = f'dif_{alias}_grupo'
    # orientado siempre a "mas alto = mas anomalo"
    expr = '(m.med - e.x)' if orden == 'DESC' else '(e.x - m.med)'
    q = f"""
    WITH e AS ({dominio_sql(src, filtro)}),
         m AS (SELECT grupo, median(x) AS med FROM e GROUP BY grupo)
    SELECT e.ocid, {expr} AS "{nombre}", m.med AS "ref_{alias}_grupo"
    FROM e JOIN m ON m.grupo = e.grupo
    """
    d = d.merge(con.execute(q).df(), on='ocid', how='left')
    difs_finales.append(nombre)
    print(f'  DIF    {nombre:<26s} <- {src:<32s} mediana del grupo como referencia')

print(f'\nv6.1: {len(d):,} filas, {d.shape[1]} columnas')
print(f'  rangos: {len(rangos_finales)}   diferencias: {len(difs_finales)}   binarias: 4')

# ---------------------------------------------------------------------------
# (C) CONTROLES
# ---------------------------------------------------------------------------
print('\n' + '='*105)
print('(C) CONTROLES DE UNIFORMIDAD Y ENCENDIDO')
print('='*105)

ctrl = []
for pct in rangos_finales:
    s = pd.to_numeric(d[pct], errors='coerce').dropna()
    ctrl.append(dict(variable=pct, con_rango=len(s), pct_cobertura=round(100*len(s)/n_total, 2),
                     p25=round(s.quantile(.25), 4), mediana=round(s.median(), 4),
                     p75=round(s.quantile(.75), 4), maximo=round(s.max(), 4),
                     pct_enciende=round(100*(s >= 0.95).mean(), 2),
                     veredicto='OK' if abs(s.median()-.5) < .06 and s.max() > .999
                                       and 3.5 <= 100*(s >= .95).mean() <= 8 else 'REVISAR'))
print(pd.DataFrame(ctrl).to_string(index=False))

if difs_finales:
    print('\n-- las variables convertidas a diferencia contra la mediana del grupo --')
    dd = []
    for nm in difs_finales:
        s = pd.to_numeric(d[nm], errors='coerce').dropna()
        dd.append(dict(variable=nm, con_dato=len(s), pct_cobertura=round(100*len(s)/n_total, 2),
                       p25=round(s.quantile(.25), 2), mediana=round(s.median(), 2),
                       p75=round(s.quantile(.75), 2), p95=round(s.quantile(.95), 2),
                       maximo=round(s.max(), 2),
                       pct_bajo_la_mediana_del_grupo=round(100*(s > 0).mean(), 2)))
    print(pd.DataFrame(dd).to_string(index=False))
    print('Valor positivo = el proceso quedo en el lado anomalo de la mediana de su grupo.')

print('\n-- las cuatro binarias (sin cambio) --')
BIN = ['ind_oferente_unico','ind_fraccionamiento_marca',
       'ind_primera_vez_con_la_entidad','ind_proveedor_debutante']
print(pd.DataFrame([dict(variable=b, con_dato=int(d[b].notna().sum()),
                         pct_verdadero=round(100*pd.to_numeric(d[b], errors='coerce').mean(), 2))
                    for b in BIN]).to_string(index=False))

# ---------------------------------------------------------------------------
# (D) COINCIDENCIA DE ALERTAS CONTRA INDEPENDENCIA
# ---------------------------------------------------------------------------
print('\n' + '='*105)
print('(D) COINCIDENCIA DE ALERTAS CONTRA INDEPENDENCIA (Poisson-binomial exacta)')
print('='*105)

def exceso(cols, k=3):
    X = np.column_stack([pd.to_numeric(d[c], errors='coerce').to_numpy(dtype='float64') for c in cols])
    A = X >= 0.95                      # alerta   (nan compara False)
    M = ~np.isnan(X)                   # medible
    p = np.array([A[M[:, j], j].mean() for j in range(len(cols))])
    obs = (A.sum(1) >= k).mean()
    clave = M.astype(np.int64) @ (1 << np.arange(len(cols), dtype=np.int64))
    vals, cnt = np.unique(clave, return_counts=True)
    esp = 0.0
    for v, c in zip(vals, cnt):
        dist = np.array([1.0])
        for j in range(len(cols)):
            if (v >> j) & 1:
                dist = np.convolve(dist, [1 - p[j], p[j]])
        esp += (c / len(d)) * dist[k:].sum()
    return p, obs, esp

for etiqueta, cols in [('conjunto completo de rangos', rangos_finales),
                       ('quitando recurrencia y cuota (las dos que se solapan)',
                        [c for c in rangos_finales if c not in ('pct_recurrencia_pos','pct_cuota_pos')])]:
    p, obs, esp = exceso(cols)
    print(f'\n{etiqueta}  ({len(cols)} variables)')
    print('  tasas de encendido: ' + ', '.join(f'{100*v:.2f}%' for v in p))
    print(f'  con 3 o mas alertas: observado {100*obs:.3f}%  esperado {100*esp:.3f}%'
          f'  -> {obs/esp:.2f} veces lo esperable' if esp > 0 else '  esperado 0')

# ---------------------------------------------------------------------------
# (E) ESCRITURA, FIRMA Y ENMIENDA
# ---------------------------------------------------------------------------
print('\n' + '='*105)
print('(E) ESCRITURA Y FIRMA')
print('='*105)

def sha16(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(1 << 20), b''):
            h.update(b)
    return h.hexdigest()[:16]

con.register('v61', d)
ver = PA / 'planA_indicadores_v6_1.parquet'
con.execute(f"COPY v61 TO '{ver.as_posix()}' (FORMAT PARQUET, COMPRESSION ZSTD)")
shutil.copyfile(ver, PA / 'planA_indicadores.parquet')
print(f'  {ver.name} | {ver.stat().st_size/1e6:.1f} MB | sha256_16 = {sha16(ver)}')
print(f'  queda como planA_indicadores.parquet con sha256_16 = '
      f'{sha16(PA / "planA_indicadores.parquet")}')

man = []
for p in sorted(PA.glob('planA_*.parquet')):
    f, c = con.execute(f"SELECT count(*), count(*) FROM read_parquet('{p.as_posix()}')").fetchone()[0], \
           len(con.execute(f"DESCRIBE SELECT * FROM read_parquet('{p.as_posix()}')").df())
    man.append(dict(archivo=p.name, filas=f, columnas=c,
                    MB=round(p.stat().st_size/1e6, 1), sha256_16=sha16(p)))
man = pd.DataFrame(man)
print('\n' + man.to_string(index=False))
man.insert(0, 'fecha', datetime.date.today().isoformat())
man.to_csv(DOC / 'manifiesto.csv', index=False, encoding='utf-8')

with open(DOC / 'enmiendas.md', 'a', encoding='utf-8') as f:
    f.write(f"\n## Enmienda v6.1 — {datetime.date.today().isoformat()}\n"
            f"- Motivo: `percent_rank()` asigna a un bloque de empates el percentil de su primer\n"
            f"  miembro, por lo que los rangos con masa en el extremo anomalo no alcanzaban 0,95.\n"
            f"- Correccion 1: todos los rangos pasan a `cume_dist()` (funcion de distribucion\n"
            f"  empirica). Rangos vigentes: {', '.join(rangos_finales)}.\n"
            f"- Correccion 2: las variables con mas del 5% del dominio en el valor extremo de su\n"
            f"  grupo dejan de representarse como rango y pasan a diferencia contra la mediana del\n"
            f"  grupo comparable: {', '.join(difs_finales) if difs_finales else 'ninguna'}.\n"
            f"- Se retiran definitivamente: pct_poca_concurrencia, pct_fraccionamiento,\n"
            f"  pct_cuota_12m, pct_recurrencia_par, pct_concurrencia_pos.\n"
            f"- Firma planA_indicadores.parquet = {sha16(PA / 'planA_indicadores.parquet')}\n")
print(f'\n  enmienda registrada en {DOC / "enmiendas.md"}')
print(f'  manifiesto reescrito: {len(man)} archivos')
print('\nPega aqui toda la salida.')

In [ ]:
# ===========================================================================
# CELDA 46 — percentil a mitad de salto (mid-distribution) para todos los
#            rangos. Regla de decision por tasa de encendido.  -> v6.2
# ===========================================================================
import duckdb, pandas as pd, numpy as np, hashlib, datetime, pathlib, shutil

BASE = pathlib.Path(r'C:\SERCOP_FINAL')
PA   = BASE / 'plan_A'
DOC  = PA / 'documentacion';  DOC.mkdir(exist_ok=True)
VER  = PA / '_versiones';     VER.mkdir(exist_ok=True)
con  = duckdb.connect()
pd.set_option('display.width', 235); pd.set_option('display.max_columns', 60)

IND = (PA / 'planA_indicadores.parquet').as_posix()
con.execute(f"CREATE VIEW ind6 AS SELECT * FROM read_parquet('{IND}')")
n_total = con.execute("SELECT count(*) FROM ind6").fetchone()[0]

# alias | nombre del rango | fuente | orden | filtro de dominio
CFG = [
 ('precio',       'pct_precio_alto',         'ind_desv_precio_unit_z',        'ASC', None),
 ('capital',      'pct_monto_capital',       'ind_monto_sobre_capital',       'ASC', None),
 ('antiguedad',   'pct_proveedor_nuevo',     'ind_antiguedad_dias',           'DESC',None),
 ('plazo',        'pct_plazo_corto',         'ind_plazo_ofertas_dias',        'DESC',None),
 ('firma',        'pct_firma_rapida',        'ind_plazo_adj_firma_dias',      'DESC',None),
 ('concurrencia', 'pct_concurrencia_pos',    'ind_n_oferentes',               'DESC','ind_n_oferentes > 1'),
 ('fracciona',    'pct_fraccionamiento_pos', 'ind_fraccionamiento_monto_30d', 'ASC', 'ind_fraccionamiento_monto_30d > 0'),
 ('cuota',        'pct_cuota_pos',           'ind_cuota_prov_en_entidad_12m', 'ASC', 'ind_cuota_prov_en_entidad_12m > 0'),
 ('recurrencia',  'pct_recurrencia_pos',     'ind_recurrencia_par_12m',       'ASC', 'ind_recurrencia_par_12m > 0'),
]

def dominio_sql(src, filtro):
    f = f'AND {filtro}' if filtro else ''
    return f"""
      SELECT ocid, x,
             CASE WHEN n_fp >= 30 THEN familia || '|' || periodo
                  WHEN n_f  >= 30 THEN familia
                  ELSE 'GLOBAL' END AS grupo
      FROM (
        SELECT ocid, familia, periodo, "{src}" AS x,
               count(*) OVER (PARTITION BY familia, periodo) AS n_fp,
               count(*) OVER (PARTITION BY familia)          AS n_f
        FROM ind6 WHERE "{src}" IS NOT NULL {f}
      )
    """

# el percentil a mitad de salto: cume_dist menos media anchura del bloque de empates
def mid_sql(src, orden, filtro, nombre):
    return f"""
    WITH e AS ({dominio_sql(src, filtro)})
    SELECT ocid,
           cume_dist() OVER (PARTITION BY grupo ORDER BY x {orden})
           - (count(*) OVER (PARTITION BY grupo, x))::DOUBLE
             / (2.0 * count(*) OVER (PARTITION BY grupo))  AS "{nombre}",
           (count(*) OVER (PARTITION BY grupo, x))::DOUBLE
             / (count(*) OVER (PARTITION BY grupo))        AS "_bloque"
    FROM e
    """

print('='*110)
print('(A) LAS TRES FORMULAS SOBRE LAS MISMAS VARIABLES, Y LA DECISION')
print('='*110)

cand, diag = {}, []
for alias, pct, src, orden, filtro in CFG:
    q = f"""
    WITH m AS ({mid_sql(src, orden, filtro, 'mid')}),
         c AS (SELECT ocid,
                      percent_rank() OVER (PARTITION BY grupo ORDER BY x {orden}) AS pr,
                      cume_dist()    OVER (PARTITION BY grupo ORDER BY x {orden}) AS cd
               FROM ({dominio_sql(src, filtro)}))
    SELECT m.ocid, m.mid, m."_bloque" AS bloque, c.pr, c.cd
    FROM m JOIN c ON c.ocid = m.ocid
    """
    t = con.execute(q).df()
    cand[pct] = t[['ocid', 'mid']].rename(columns={'mid': pct})
    enc = lambda s: round(100*(s >= 0.95).mean(), 2)
    ok = (abs(t['mid'].mean() - 0.5) < 0.01) and (t['mid'].max() >= 0.95) and (3.5 <= enc(t['mid']) <= 8.0)
    diag.append(dict(variable=pct, filas=len(t), valores=int(t['mid'].nunique()),
                     bloque_medio=round(100*t['bloque'].mean(), 2),
                     media_mid=round(t['mid'].mean(), 4), max_mid=round(t['mid'].max(), 4),
                     enc_percent_rank=enc(t['pr']), enc_cume_dist=enc(t['cd']),
                     enc_mid=enc(t['mid']),
                     decision='rango' if ok else 'CONVERTIR'))
diag = pd.DataFrame(diag)
print(diag.to_string(index=False))
print("""
bloque_medio: % del grupo que, en promedio, comparte el valor de un proceso. Mide cuan
  discreta es la variable en la practica; con bloques grandes percent_rank subestima y
  cume_dist sobreestima, cada uno por media anchura de bloque.
media_mid: control duro. El percentil a mitad de salto tiene media exactamente 0,5000
  por construccion, haya los empates que haya. Si no sale 0,5000 hay error de calculo.
decision: se conserva como rango si la media es 0,5000, el maximo alcanza 0,95 y la tasa
  de encendido cae entre 3,5% y 8%. Lo demas se convierte a diferencia contra la mediana.""")

APTAS   = [c for c in CFG if diag.loc[diag.variable == c[1], 'decision'].iloc[0] == 'rango']
CONVERT = [c for c in CFG if diag.loc[diag.variable == c[1], 'decision'].iloc[0] == 'CONVERTIR']

# ---------------------------------------------------------------------------
# (B) RECONSTRUCCION
# ---------------------------------------------------------------------------
print('\n' + '='*110)
print('(B) RECONSTRUCCION  -> v6.2')
print('='*110)

d = con.execute("SELECT * FROM ind6").df()
fuera = ([c[1] for c in CFG] +
         ['pct_poca_concurrencia','pct_fraccionamiento','pct_cuota_12m','pct_recurrencia_par'] +
         [f'dif_{a}_grupo' for a,*_ in CFG] + [f'ref_{a}_grupo' for a,*_ in CFG])
d = d.drop(columns=[c for c in fuera if c in d.columns])

rangos_finales = []
for alias, pct, src, orden, filtro in APTAS:
    d = d.merge(cand[pct], on='ocid', how='left')
    rangos_finales.append(pct)
    print(f'  rango  {pct:<26s} <- {src:<32s} {orden:<4s} mid-distribution')

difs_finales = []
for alias, pct, src, orden, filtro in CONVERT:
    nombre = f'dif_{alias}_grupo'
    expr = '(m.med - e.x)' if orden == 'DESC' else '(e.x - m.med)'
    q = f"""
    WITH e AS ({dominio_sql(src, filtro)}),
         m AS (SELECT grupo, median(x) AS med FROM e GROUP BY grupo)
    SELECT e.ocid, {expr} AS "{nombre}", m.med AS "ref_{alias}_grupo"
    FROM e JOIN m ON m.grupo = e.grupo
    """
    d = d.merge(con.execute(q).df(), on='ocid', how='left')
    difs_finales.append(nombre)
    print(f'  DIF    {nombre:<26s} <- {src:<32s} mediana del grupo como referencia')

print(f'\nv6.2: {len(d):,} filas, {d.shape[1]} columnas'
      f'   | rangos: {len(rangos_finales)}  diferencias: {len(difs_finales)}  binarias: 4')

# ---------------------------------------------------------------------------
# (C) CONTROLES
# ---------------------------------------------------------------------------
print('\n' + '='*110)
print('(C) CONTROLES')
print('='*110)
ctrl = []
for pct in rangos_finales:
    s = pd.to_numeric(d[pct], errors='coerce').dropna()
    ctrl.append(dict(variable=pct, con_rango=len(s), pct_cobertura=round(100*len(s)/n_total, 2),
                     media=round(s.mean(), 4), p25=round(s.quantile(.25), 4),
                     mediana=round(s.median(), 4), p75=round(s.quantile(.75), 4),
                     maximo=round(s.max(), 4), pct_enciende=round(100*(s >= .95).mean(), 2),
                     veredicto='OK' if abs(s.mean()-.5) < .01 and s.max() >= .95
                                       and 3.5 <= 100*(s >= .95).mean() <= 8 else 'REVISAR'))
print(pd.DataFrame(ctrl).to_string(index=False))

if difs_finales:
    print('\n-- convertidas a diferencia contra la mediana del grupo (positivo = lado anomalo) --')
    print(pd.DataFrame([dict(variable=nm,
            con_dato=int(pd.to_numeric(d[nm], errors='coerce').notna().sum()),
            pct_cobertura=round(100*pd.to_numeric(d[nm], errors='coerce').notna().mean(), 2),
            p25=round(pd.to_numeric(d[nm], errors='coerce').quantile(.25), 2),
            mediana=round(pd.to_numeric(d[nm], errors='coerce').median(), 2),
            p75=round(pd.to_numeric(d[nm], errors='coerce').quantile(.75), 2),
            p95=round(pd.to_numeric(d[nm], errors='coerce').quantile(.95), 2),
            maximo=round(pd.to_numeric(d[nm], errors='coerce').max(), 2),
            pct_lado_anomalo=round(100*(pd.to_numeric(d[nm], errors='coerce') > 0).mean(), 2))
          for nm in difs_finales]).to_string(index=False))

BIN = ['ind_oferente_unico','ind_fraccionamiento_marca',
       'ind_primera_vez_con_la_entidad','ind_proveedor_debutante']
print('\n-- binarias --')
print(pd.DataFrame([dict(variable=b, con_dato=int(d[b].notna().sum()),
                         pct_verdadero=round(100*pd.to_numeric(d[b], errors='coerce').mean(), 2))
                    for b in BIN]).to_string(index=False))

# ---------------------------------------------------------------------------
# (D) COINCIDENCIA CONTRA INDEPENDENCIA, YA CON TODAS LAS TASAS CALIBRADAS
# ---------------------------------------------------------------------------
print('\n' + '='*110)
print('(D) COINCIDENCIA DE ALERTAS CONTRA INDEPENDENCIA (Poisson-binomial exacta)')
print('='*110)

def exceso(cols, k=3):
    X = np.column_stack([pd.to_numeric(d[c], errors='coerce').to_numpy(dtype='float64') for c in cols])
    A, M = X >= 0.95, ~np.isnan(X)
    p = np.array([A[M[:, j], j].mean() for j in range(len(cols))])
    obs = (A.sum(1) >= k).mean()
    clave = M.astype(np.int64) @ (1 << np.arange(len(cols), dtype=np.int64))
    vals, cnt = np.unique(clave, return_counts=True)
    esp = 0.0
    for v, c in zip(vals, cnt):
        dist = np.array([1.0])
        for j in range(len(cols)):
            if (v >> j) & 1:
                dist = np.convolve(dist, [1 - p[j], p[j]])
        esp += (c / len(d)) * dist[k:].sum()
    return p, obs, esp

print('\n banderas  observado_pct  esperado_pct  veces')
X = np.column_stack([pd.to_numeric(d[c], errors='coerce').to_numpy(dtype='float64') for c in rangos_finales])
for k in range(1, len(rangos_finales) + 1):
    _, o, e = exceso(rangos_finales, k)
    if o == 0 and e < 1e-9: break
    print(f'  {k} o mas      {100*o:9.3f}     {100*e:9.3f}   {o/e:7.2f}' if e > 0
          else f'  {k} o mas      {100*o:9.3f}     {0:9.3f}        -')

for etiqueta, cols in [('conjunto completo', rangos_finales),
                       ('quitando recurrencia y cuota',
                        [c for c in rangos_finales if c not in ('pct_recurrencia_pos','pct_cuota_pos')])]:
    p, o, e = exceso(cols)
    print(f'\n{etiqueta} ({len(cols)} variables)')
    print('  tasas: ' + ', '.join(f'{100*v:.2f}%' for v in p))
    print(f'  3 o mas alertas: observado {100*o:.3f}%  esperado {100*e:.3f}%  -> {o/e:.2f} veces')

# ---------------------------------------------------------------------------
# (E) ESCRITURA, ORDEN DE VERSIONES Y FIRMA
# ---------------------------------------------------------------------------
print('\n' + '='*110)
print('(E) ESCRITURA Y FIRMA')
print('='*110)

def sha16(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(1 << 20), b''):
            h.update(b)
    return h.hexdigest()[:16]

# las copias de version no son archivos de entrega: se guardan aparte
for p in list(PA.glob('planA_indicadores_v*.parquet')):
    shutil.move(str(p), str(VER / p.name))
    print(f'  archivado  _versiones/{p.name}')

con.register('v62', d)
ver = VER / 'planA_indicadores_v6_2.parquet'
con.execute(f"COPY v62 TO '{ver.as_posix()}' (FORMAT PARQUET, COMPRESSION ZSTD)")
shutil.copyfile(ver, PA / 'planA_indicadores.parquet')
print(f'\n  {ver.name} | {ver.stat().st_size/1e6:.1f} MB | sha256_16 = {sha16(ver)}')
print(f'  queda como planA_indicadores.parquet con sha256_16 = '
      f'{sha16(PA / "planA_indicadores.parquet")}')

man = []
for p in sorted(PA.glob('planA_*.parquet')):
    f = con.execute(f"SELECT count(*) FROM read_parquet('{p.as_posix()}')").fetchone()[0]
    c = len(con.execute(f"DESCRIBE SELECT * FROM read_parquet('{p.as_posix()}')").df())
    man.append(dict(archivo=p.name, filas=f, columnas=c,
                    MB=round(p.stat().st_size/1e6, 1), sha256_16=sha16(p)))
man = pd.DataFrame(man)
print('\n' + man.to_string(index=False))
man.insert(0, 'fecha', datetime.date.today().isoformat())
man.to_csv(DOC / 'manifiesto.csv', index=False, encoding='utf-8')

with open(DOC / 'enmiendas.md', 'a', encoding='utf-8') as f:
    f.write(f"\n## Enmienda v6.2 — {datetime.date.today().isoformat()}\n"
            f"- Motivo: con variables discretas, percent_rank subestima el percentil de un\n"
            f"  bloque de empates y cume_dist lo sobreestima, cada uno por media anchura de\n"
            f"  bloque. v6 encendia 2,49% y 0,86% en plazo y firma; v6.1 encendio 11,38% y 10,24%.\n"
            f"- Correccion: todos los rangos usan el percentil a mitad de salto\n"
            f"  (mid-distribution function), cume_dist - b/(2n), de media exactamente 0,5.\n"
            f"- Regla de conservacion: media 0,5000, maximo >= 0,95 y encendido entre 3,5% y 8%.\n"
            f"- Rangos vigentes: {', '.join(rangos_finales)}.\n"
            f"- Convertidas a diferencia contra la mediana del grupo: "
            f"{', '.join(difs_finales) if difs_finales else 'ninguna'}.\n"
            f"- Las copias de version salen de plan_A/ a plan_A/_versiones/; el manifiesto\n"
            f"  vuelve a listar solo los archivos de entrega.\n"
            f"- Firma planA_indicadores.parquet = {sha16(PA / 'planA_indicadores.parquet')}\n")
print(f'\n  enmienda registrada | manifiesto: {len(man)} archivos de entrega')
print('\nPega aqui toda la salida.')

In [ ]:
# ---------------------------------------------------------------------------
# CELDA 46-bis — datos de las dos entidades que no se pueden clasificar
#                por el nombre, para buscarlas en el portal
# ---------------------------------------------------------------------------
q = """
SELECT entidad_ruc, entidad_nombre,
       count(*) AS procesos,
       round(sum(monto_final)/1e6, 1) AS millones,
       min(anio_proceso) AS desde, max(anio_proceso) AS hasta,
       min(ocid) AS ocid_ejemplo_1,
       max(ocid) AS ocid_ejemplo_2
FROM read_parquet('C:/SERCOP_FINAL/plan_A/planA_procesos_modelo.parquet')
WHERE upper(entidad_nombre) LIKE '%COORDINACION ZONAL 4%'
   OR upper(entidad_nombre) LIKE '%COORDINACIÓN ZONAL 4%'
   OR upper(entidad_nombre) LIKE '%DIGMAT%'
GROUP BY 1, 2
ORDER BY millones DESC
"""
print(con.execute(q).df().to_string(index=False))

In [ ]:
# ===========================================================================
# CELDA 47 — tipo institucional persistido en planA_dim_entidad, por RUC.
#            Incluye nombre canonico y conteo de nombres alternos.
# ===========================================================================
import duckdb, pandas as pd, numpy as np, hashlib, datetime, pathlib, shutil, re, unicodedata

BASE = pathlib.Path(r'C:\SERCOP_FINAL')
PA   = BASE / 'plan_A'
DOC  = PA / 'documentacion'
VER  = PA / '_versiones';  VER.mkdir(exist_ok=True)
con  = duckdb.connect()
pd.set_option('display.width', 235); pd.set_option('display.max_columns', 60)

PM  = (PA / 'planA_procesos_modelo.parquet').as_posix()
DE  = (PA / 'planA_dim_entidad.parquet').as_posix()
con.execute(f"CREATE VIEW pm AS SELECT * FROM read_parquet('{PM}')")
con.execute(f"CREATE VIEW de AS SELECT * FROM read_parquet('{DE}')")
print('columnas de dim_entidad:', list(con.execute("DESCRIBE SELECT * FROM de").df()['column_name']))

# --- nombre canonico: el del proceso mas reciente de cada RUC -------------
nom = con.execute("""
WITH n AS (
  SELECT entidad_ruc, entidad_nombre, max(anio_proceso) AS ult, count(*) AS proc
  FROM pm GROUP BY 1, 2
),
r AS (
  SELECT *, row_number() OVER (PARTITION BY entidad_ruc ORDER BY ult DESC, proc DESC) AS rn
  FROM n
)
SELECT entidad_ruc,
       max(CASE WHEN rn = 1 THEN entidad_nombre END) AS entidad_nombre_canonico,
       count(*) AS n_nombres_distintos
FROM r GROUP BY entidad_ruc
""").df()
print(f'\nRUC con mas de un nombre historico: '
      f'{int((nom.n_nombres_distintos > 1).sum()):,} de {len(nom):,}')
print(nom[nom.n_nombres_distintos > 1].nlargest(8, 'n_nombres_distintos').to_string(index=False))

# --- normalizacion: mayusculas, sin tildes, sin puntos --------------------
def norm(s):
    if s is None or (isinstance(s, float) and np.isnan(s)): return ''
    s = unicodedata.normalize('NFKD', str(s)).encode('ascii', 'ignore').decode()
    return re.sub(r'\s+', ' ', s.upper().replace('.', '')).strip()
# quitar los puntos es lo que permite reconocer A.N.D.E.C. y E.P.

# --- correcciones confirmadas una por una, por RUC (prevalecen sobre todo)
MANUAL = {
 '1768011230001': 'Seguridad y defensa',   # DIGMAT - Direccion General del Material, Armada
 '1360066490001': 'Educacion',             # Coordinacion Zonal 4 del Ministerio de Educacion
}

# --- reglas por nombre, en orden: la primera que coincide gana ------------
REGLAS = [
 ('Banca publica', r'\bBANECUADOR\b|BANCO CENTRAL|CORPORACION FINANCIERA NACIONAL|\bCFN\b|'
                   r'\bBIESS\b|BANCO NACIONAL DE FOMENTO|BANCO DE DESARROLLO|BANCO DEL ESTADO'),
 ('Salud',         r'HOSPITAL|CENTRO DE SALUD|MINISTERIO DE SALUD|\bMSP\b|DISTRITO.*SALUD|'
                   r'ZONAL.*SALUD|\bARCSA\b|\bSOLCA\b|INSTITUTO NACIONAL DE DONACION|'
                   r'UNIDAD.*SALUD|MATERNIDAD|DISPENSARIO'),
 ('Seguridad y defensa',
                   r'COMANDANCIA|SUBZONA|FUERZA TERRESTRE|FUERZA AEREA|FUERZA NAVAL|\bARMADA\b|'
                   r'POLICIA|MINISTERIO DE DEFENSA|\bDIGMAT\b|\bFAME\b|SEGURIDAD CIUDADANA|'
                   r'COMISION DE TRANSITO|\bCTE\b|CUERPO DE BOMBEROS|\bISSFA\b|\bISSPOL\b'),
 ('Educacion superior',
                   r'UNIVERSIDAD|ESCUELA SUPERIOR POLITECNICA|ESCUELA POLITECNICA|\bESPOL\b|'
                   r'\bESPE\b|\bULEAM\b|\bUTM\b|\bUTE\b|SENESCYT|EDUCACION SUPERIOR|'
                   r'INSTITUTO SUPERIOR|INSTITUTO TECNOLOGICO SUPERIOR'),
 ('Educacion',     r'MINISTERIO DE EDUCACION|\bMINEDUC\b|DISTRITO.*EDUCACION|UNIDAD EDUCATIVA|'
                   r'ZONAL.*EDUCACION|COLEGIO|ESCUELA FISCAL'),
 ('Empresa publica',
                   r'EMPRESA PUBLICA|EMPRESA MUNICIPAL|\bEP\b|\bEMMOP\b|\bEMUVI\b|\bEMAPA\b|'
                   r'\bETAPA\b|\bCELEC\b|\bCNEL\b|\bCNT\b|PETROECUADOR|\bANDEC\b|AUSTROGAS|'
                   r'AUTORIDAD PORTUARIA|ELECTRO|ECONOMIA MIXTA|TERMINAL TERRESTRE|'
                   r'CORPORACION ELECTRICA|CORPORACION NACIONAL DE TELECOMUNICACIONES|'
                   r'ELECTRICA|AGUA POTABLE|\bSA\b|\bCEM\b'),
 ('Gobierno local',
                   r'MUNICIP|\bGADM?\b|GOBIERNO AUTONOMO|PREFECTURA|CONSEJO PROVINCIAL|'
                   r'GOBIERNO PROVINCIAL|GOB PROV|JUNTA PARROQUIAL|DISTRITO METROPOLITANO|'
                   r'\bCANTON\b|MANCOMUNIDAD'),
 ('Gobierno central',
                   r'MINISTERIO|SECRETARIA|SERVICIO DE RENTAS|\bSRI\b|SERVICIO DE CONTRATACION|'
                   r'\bINSTITUTO\b|\bAGENCIA\b|DIRECCION DISTRITAL|COORDINACION ZONAL|'
                   r'PARQUE NACIONAL|\bSENAE\b|REGISTRO CIVIL|DIGERCIC|\bIESS\b|SEGURIDAD SOCIAL|'
                   r'SUPERINTENDENCIA|DEFENSORIA|FISCALIA|CONSEJO DE LA JUDICATURA|\bCORTE\b|'
                   r'ASAMBLEA|PRESIDENCIA|VICEPRESIDENCIA|GOBERNACION|\bMIES\b|'
                   r'DESARROLLO HUMANO|CONSEJO NACIONAL|SERVICIO NACIONAL|\bSENPLADES\b'),
]

def clasificar(ruc, nombre):
    if ruc in MANUAL: return MANUAL[ruc], 'manual'
    n = norm(nombre)
    for tipo, pat in REGLAS:
        if re.search(pat, n): return tipo, 'patron'
    return 'Otros', 'residuo'

# --- aplicar sobre dim_entidad --------------------------------------------
dim = con.execute("SELECT * FROM de").df()
col_ruc = 'entidad_ruc' if 'entidad_ruc' in dim.columns else dim.columns[0]
col_nom = next(c for c in ['entidad_nombre','nombre','entidad'] if c in dim.columns)
dim[col_ruc] = dim[col_ruc].astype(str)
nom['entidad_ruc'] = nom['entidad_ruc'].astype(str)

dim = dim.drop(columns=[c for c in ['tipo_entidad','tipo_entidad_origen',
                                    'entidad_nombre_canonico','n_nombres_distintos']
                        if c in dim.columns])
dim = dim.merge(nom, left_on=col_ruc, right_on='entidad_ruc', how='left',
                suffixes=('', '_y'))
if 'entidad_ruc_y' in dim.columns: dim = dim.drop(columns=['entidad_ruc_y'])
dim['entidad_nombre_canonico'] = dim['entidad_nombre_canonico'].fillna(dim[col_nom])
dim['n_nombres_distintos'] = dim['n_nombres_distintos'].fillna(1).astype(int)

res = [clasificar(r, n) for r, n in zip(dim[col_ruc], dim['entidad_nombre_canonico'])]
dim['tipo_entidad']        = [t for t, _ in res]
dim['tipo_entidad_origen'] = [o for _, o in res]

print('\n' + '='*110)
print('(A) COMO QUEDO LA CLASIFICACION, MEDIDA SOBRE LOS PROCESOS')
print('='*110)
con.register('tip', dim[[col_ruc, 'tipo_entidad', 'tipo_entidad_origen',
                         'entidad_nombre_canonico']].rename(columns={col_ruc: 'ruc'}))
print(con.execute("""
SELECT coalesce(t.tipo_entidad, 'SIN FICHA') AS tipo,
       count(DISTINCT pm.entidad_ruc) AS entidades, count(*) AS procesos,
       round(100.0*count(*)/(SELECT count(*) FROM pm), 2) AS pct_procesos,
       round(sum(pm.monto_final)/1e6, 1) AS millones,
       round(100.0*sum(pm.monto_final)/(SELECT sum(monto_final) FROM pm), 2) AS pct_monto,
       round(median(pm.monto_final), 0) AS mediana_usd,
       round(100.0*avg(CASE WHEN pm.ind_oferente_unico THEN 1.0 ELSE 0.0 END), 2) AS pct_oferente_unico
FROM pm LEFT JOIN tip t ON t.ruc = pm.entidad_ruc
GROUP BY 1 ORDER BY millones DESC
""").df().to_string(index=False))

print('\n-- de donde sale cada asignacion --')
print(dim.groupby('tipo_entidad_origen').size().rename('entidades').to_frame().to_string())

print('\n-- lo que sigue en "Otros": las 15 mas grandes --')
print(con.execute("""
SELECT t.entidad_nombre_canonico AS entidad, pm.entidad_ruc AS ruc,
       count(*) AS procesos, round(sum(pm.monto_final)/1e6, 1) AS millones
FROM pm JOIN tip t ON t.ruc = pm.entidad_ruc
WHERE t.tipo_entidad = 'Otros'
GROUP BY 1, 2 ORDER BY millones DESC LIMIT 15
""").df().to_string(index=False))

# --- escritura y firma ----------------------------------------------------
print('\n' + '='*110)
print('(B) ESCRITURA Y FIRMA')
print('='*110)
def sha16(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(1 << 20), b''): h.update(b)
    return h.hexdigest()[:16]

shutil.copyfile(PA / 'planA_dim_entidad.parquet',
                VER / 'planA_dim_entidad_previo_tipologia.parquet')
con.register('dimout', dim)
con.execute(f"COPY dimout TO '{(PA/'planA_dim_entidad.parquet').as_posix()}' "
            f"(FORMAT PARQUET, COMPRESSION ZSTD)")
print(f'  planA_dim_entidad.parquet | {dim.shape[0]:,} filas, {dim.shape[1]} columnas '
      f'| sha256_16 = {sha16(PA / "planA_dim_entidad.parquet")}')

man = []
for p in sorted(PA.glob('planA_*.parquet')):
    f = con.execute(f"SELECT count(*) FROM read_parquet('{p.as_posix()}')").fetchone()[0]
    c = len(con.execute(f"DESCRIBE SELECT * FROM read_parquet('{p.as_posix()}')").df())
    man.append(dict(archivo=p.name, filas=f, columnas=c,
                    MB=round(p.stat().st_size/1e6, 1), sha256_16=sha16(p)))
man = pd.DataFrame(man); print('\n' + man.to_string(index=False))
man.insert(0, 'fecha', datetime.date.today().isoformat())
man.to_csv(DOC / 'manifiesto.csv', index=False, encoding='utf-8')

with open(DOC / 'enmiendas.md', 'a', encoding='utf-8') as f:
    f.write(f"\n## Enmienda tipologia institucional — {datetime.date.today().isoformat()}\n"
            f"- planA_dim_entidad.parquet gana cuatro columnas: tipo_entidad,\n"
            f"  tipo_entidad_origen (manual / patron / residuo), entidad_nombre_canonico\n"
            f"  (el nombre del proceso mas reciente del RUC) y n_nombres_distintos.\n"
            f"- La tipologia se asigna por RUC, no por nombre: un mismo RUC aparece con\n"
            f"  nombres distintos segun el anio (MIES -> Ministerio de Desarrollo Humano;\n"
            f"  Servicio Nacional de Gestion de Riesgos -> Gestion de Riesgos), de modo que\n"
            f"  contar entidades por nombre infla el numero de entidades.\n"
            f"- Dos asignaciones verificadas una por una contra el SRI y el portal:\n"
            f"  1768011230001 DIGMAT = Direccion General del Material (Armada) -> Seguridad\n"
            f"  y defensa; 1360066490001 Coordinacion Zonal 4 = Ministerio de Educacion ->\n"
            f"  Educacion. Existen doce coordinaciones zonales 4 distintas, una por ministerio.\n"
            f"- La clasificacion es heuristica sobre el nombre salvo las verificadas, y debe\n"
            f"  declararse como tal: sirve para formar grupos comparables, no es dato oficial.\n"
            f"- Firma planA_dim_entidad.parquet = {sha16(PA / 'planA_dim_entidad.parquet')}\n")
print(f'\n  enmienda registrada | manifiesto: {len(man)} archivos de entrega')
print('\nPega aqui toda la salida.')

In [ ]:
# ===========================================================================
# CELDA 47b — continua la 47 (dim ya esta en memoria): la bandera de oferente
#             unico esta en indicadores, no en procesos_modelo
# ===========================================================================
con.execute(f"CREATE OR REPLACE VIEW ix AS SELECT * FROM "
            f"read_parquet('{(PA/'planA_indicadores.parquet').as_posix()}')")

con.register('tip', dim[[col_ruc, 'tipo_entidad', 'tipo_entidad_origen',
                         'entidad_nombre_canonico']].rename(columns={col_ruc: 'ruc'}))

print('='*110)
print('(A) COMO QUEDO LA CLASIFICACION, MEDIDA SOBRE LOS PROCESOS')
print('='*110)
print(con.execute("""
SELECT coalesce(t.tipo_entidad, 'SIN FICHA') AS tipo,
       count(DISTINCT pm.entidad_ruc) AS entidades,
       count(*) AS procesos,
       round(100.0*count(*)/(SELECT count(*) FROM pm), 2) AS pct_procesos,
       round(sum(pm.monto_final)/1e6, 1) AS millones,
       round(100.0*sum(pm.monto_final)/(SELECT sum(monto_final) FROM pm), 2) AS pct_monto,
       round(median(pm.monto_final), 0) AS mediana_usd,
       round(100.0*avg(CASE WHEN ix.ind_oferente_unico THEN 1.0 ELSE 0.0 END), 2) AS pct_oferente_unico
FROM pm
LEFT JOIN tip t ON t.ruc = pm.entidad_ruc
LEFT JOIN ix    ON ix.ocid = pm.ocid
GROUP BY 1 ORDER BY millones DESC
""").df().to_string(index=False))

print('\n-- de donde sale cada asignacion --')
print(dim.groupby('tipo_entidad_origen').size().rename('entidades').to_frame().to_string())

print('\n-- lo que sigue en "Otros": las 15 mas grandes --')
print(con.execute("""
SELECT t.entidad_nombre_canonico AS entidad, pm.entidad_ruc AS ruc,
       count(*) AS procesos, round(sum(pm.monto_final)/1e6, 1) AS millones
FROM pm JOIN tip t ON t.ruc = pm.entidad_ruc
WHERE t.tipo_entidad = 'Otros'
GROUP BY 1, 2 ORDER BY millones DESC LIMIT 15
""").df().to_string(index=False))

# control: ningun proceso deberia quedar sin ficha de entidad
sin_ficha = con.execute("""
SELECT count(*) FROM pm LEFT JOIN tip t ON t.ruc = pm.entidad_ruc
WHERE t.ruc IS NULL
""").fetchone()[0]
print(f'\nprocesos cuya entidad no tiene ficha en dim_entidad: {sin_ficha:,}')

# ---------------------------------------------------------------------------
# (B) ESCRITURA Y FIRMA
# ---------------------------------------------------------------------------
print('\n' + '='*110)
print('(B) ESCRITURA Y FIRMA')
print('='*110)
def sha16(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(1 << 20), b''): h.update(b)
    return h.hexdigest()[:16]

shutil.copyfile(PA / 'planA_dim_entidad.parquet',
                VER / 'planA_dim_entidad_previo_tipologia.parquet')
con.register('dimout', dim)
con.execute(f"COPY dimout TO '{(PA/'planA_dim_entidad.parquet').as_posix()}' "
            f"(FORMAT PARQUET, COMPRESSION ZSTD)")
print(f'  planA_dim_entidad.parquet | {dim.shape[0]:,} filas, {dim.shape[1]} columnas '
      f'| sha256_16 = {sha16(PA / "planA_dim_entidad.parquet")}')

man = []
for p in sorted(PA.glob('planA_*.parquet')):
    f = con.execute(f"SELECT count(*) FROM read_parquet('{p.as_posix()}')").fetchone()[0]
    c = len(con.execute(f"DESCRIBE SELECT * FROM read_parquet('{p.as_posix()}')").df())
    man.append(dict(archivo=p.name, filas=f, columnas=c,
                    MB=round(p.stat().st_size/1e6, 1), sha256_16=sha16(p)))
man = pd.DataFrame(man); print('\n' + man.to_string(index=False))
man.insert(0, 'fecha', datetime.date.today().isoformat())
man.to_csv(DOC / 'manifiesto.csv', index=False, encoding='utf-8')

with open(DOC / 'enmiendas.md', 'a', encoding='utf-8') as f:
    f.write(f"\n## Enmienda tipologia institucional — {datetime.date.today().isoformat()}\n"
            f"- planA_dim_entidad.parquet gana cuatro columnas: tipo_entidad,\n"
            f"  tipo_entidad_origen (manual / patron / residuo), entidad_nombre_canonico\n"
            f"  (el nombre del proceso mas reciente del RUC) y n_nombres_distintos.\n"
            f"- La tipologia se asigna por RUC, no por nombre: un mismo RUC aparece con\n"
            f"  nombres distintos segun el anio (MIES -> Ministerio de Desarrollo Humano;\n"
            f"  Servicio Nacional de Gestion de Riesgos -> Gestion de Riesgos), de modo que\n"
            f"  contar entidades por nombre infla el numero de entidades.\n"
            f"- Dos asignaciones verificadas una por una contra el SRI y el portal:\n"
            f"  1768011230001 DIGMAT = Direccion General del Material (Armada) -> Seguridad\n"
            f"  y defensa; 1360066490001 Coordinacion Zonal 4 = Ministerio de Educacion ->\n"
            f"  Educacion. Existen doce coordinaciones zonales 4 distintas, una por ministerio.\n"
            f"- La clasificacion es heuristica sobre el nombre salvo las verificadas, y debe\n"
            f"  declararse como tal: sirve para formar grupos comparables, no es dato oficial.\n"
            f"- Firma planA_dim_entidad.parquet = {sha16(PA / 'planA_dim_entidad.parquet')}\n")
print(f'\n  enmienda registrada | manifiesto: {len(man)} archivos de entrega')
print('\nPega aqui toda la salida.')

In [ ]:
# las vistas se recrean siempre: asi la celda no depende de lo que quedo en memoria
con.execute(f"CREATE OR REPLACE VIEW pm AS SELECT * FROM read_parquet('{(PA/'planA_procesos_modelo.parquet').as_posix()}')")
con.execute(f"CREATE OR REPLACE VIEW de AS SELECT * FROM read_parquet('{(PA/'planA_dim_entidad.parquet').as_posix()}')")
con.execute(f"CREATE OR REPLACE VIEW ix AS SELECT * FROM read_parquet('{(PA/'planA_indicadores.parquet').as_posix()}')")
# ===========================================================================
# CELDA 48 — tipologia sobre TODOS los nombres historicos del RUC, con
#            registro del patron que clasifico y matriz de cambios
# ===========================================================================
import re, unicodedata, hashlib, datetime, shutil
import pandas as pd, numpy as np

# --- todos los nombres de cada RUC, y el canonico por fecha real ----------
# --- todos los nombres de cada RUC, y el canonico por fecha real ----------
nombres = con.execute("""
WITH n AS (
  SELECT entidad_ruc, entidad_nombre,
         max(fecha_inicio_tender) AS ult, count(*) AS proc
  FROM pm GROUP BY 1, 2
)
SELECT entidad_ruc, entidad_nombre, ult, proc,
       row_number() OVER (PARTITION BY entidad_ruc ORDER BY ult DESC, proc DESC) AS rn
FROM n
""").df()
nombres['entidad_ruc'] = nombres['entidad_ruc'].astype(str)
POR_RUC = nombres.groupby('entidad_ruc')['entidad_nombre'].apply(list).to_dict()
CANON   = (nombres[nombres.rn == 1].set_index('entidad_ruc')['entidad_nombre'].to_dict())
N_NOM   = nombres.groupby('entidad_ruc').size().to_dict()
print(f'RUC en procesos: {len(POR_RUC):,} | con mas de un nombre: '
      f'{sum(1 for v in N_NOM.values() if v > 1):,}')

def norm(s):
    if s is None or (isinstance(s, float) and pd.isna(s)): return ''
    s = unicodedata.normalize('NFKD', str(s)).encode('ascii', 'ignore').decode()
    return re.sub(r'\s+', ' ', s.upper().replace('.', '')).strip()

MANUAL = {'1768011230001': 'Seguridad y defensa',   # DIGMAT, Armada del Ecuador
          '1360066490001': 'Educacion'}            # Coord. Zonal 4 del MINEDUC

# el orden importa: la primera regla que coincide gana
REGLAS = [
 ('Banca publica', ['BANECUADOR','BANCO CENTRAL','CORPORACION FINANCIERA NACIONAL',
    r'\bCFN\b',r'\bBIESS\b','BANCO NACIONAL DE FOMENTO','BANCO DE DESARROLLO',
    'BANCO DEL ESTADO','BANCO DEL INSTITUTO']),
 ('Salud', ['HOSPITAL','CENTRO DE SALUD','CENTRO MEDICO','CENTRO DE ESPECIALIDADES',
    'DIALISIS','MINISTERIO DE SALUD',r'\bMSP\b',r'DISTRITO.*SALUD',r'DIRECCION.*SALUD',
    r'ZONAL.*SALUD',r'UNIDAD.*SALUD',r'\bARCSA\b',r'\bSOLCA\b','MATERNIDAD','DISPENSARIO',
    'INSTITUTO NACIONAL DE DONACION','ONCOLOG','PSIQUIATR','INSTITUTO NACIONAL DE SALUD']),
 ('Seguridad y defensa', ['COMANDANCIA','COMANDO',r'\bEJERCITO\b','CUERPO DE INGENIEROS',
    'FUERZA TERRESTRE','FUERZA AEREA','FUERZA NAVAL',r'\bFAE\b',r'\bARMADA\b','SUBZONA',
    'POLICIA','MINISTERIO DE DEFENSA',r'\bDIGMAT\b',r'\bFAME\b','SEGURIDAD CIUDADANA',
    'ECU 911','ECU911','ANTIDROGAS','DIRECCION DE MOVILIZACION','SERVICIO DE DRAGAS',
    'COMISION DE TRANSITO',r'\bCTE\b','CUERPO DE BOMBEROS',r'\bISSFA\b',r'\bISSPOL\b',
    'MINISTERIO DEL INTERIOR',r'\bSNAI\b','REHABILITACION SOCIAL']),
 ('Educacion superior', ['UNIVERSIDAD','ESCUELA SUPERIOR POLITECNICA','ESCUELA POLITECNICA',
    r'\bESPOL\b',r'\bESPE\b',r'\bULEAM\b',r'\bUTM\b',r'\bUTE\b','SENESCYT',
    'EDUCACION SUPERIOR','INSTITUTO SUPERIOR','INSTITUTO TECNOLOGICO SUPERIOR',
    r'\bYACHAY\b',r'\bIKIAM\b']),
 ('Educacion', ['MINISTERIO DE EDUCACION',r'\bMINEDUC\b',r'DISTRITO.*EDUCACION',
    r'ZONAL.*EDUCACION',r'DIRECCION.*EDUCACION','UNIDAD EDUCATIVA','COLEGIO',
    'ESCUELA FISCAL']),
 ('Empresa publica', ['EMPRESA PUBLICA','EMPRESA MUNICIPAL',r'\bEP\b',r'\bEMMOP\b',
    r'\bEMUVI\b',r'\bEMAPA\b',r'\bETAPA\b',r'\bCELEC\b',r'\bCNEL\b',r'\bCNT\b',
    'PETROECUADOR',r'\bANDEC\b','AUSTROGAS','AUTORIDAD PORTUARIA','ELECTRO',
    'ECONOMIA MIXTA','TERMINAL TERRESTRE','CORPORACION ELECTRICA',
    'CORPORACION NACIONAL DE TELECOMUNICACIONES','ELECTRICA','AGUA POTABLE',
    'ALCANTARILLADO','MERCADO MAYORISTA',r'\bSA\b',r'\bCEM\b','FERROCARRILES',
    'CORREOS DEL ECUADOR',r'\bFLOPEC\b',r'\bTAME\b','ASTILLEROS']),
 ('Gobierno local', ['MUNICIP',r'\bGADM?\b','GOBIERNO AUTONOMO','PREFECTURA',
    'CONSEJO PROVINCIAL','GOBIERNO PROVINCIAL','GOB PROV','JUNTA PARROQUIAL',
    'DISTRITO METROPOLITANO',r'\bCANTON\b','MANCOMUNIDAD',
    'REGIMEN ESPECIAL DE GALAPAGOS','CORPOSUCUMBIOS','PATRONATO']),
 ('Gobierno central', ['MINISTERIO','SECRETARIA','SERVICIO DE RENTAS',r'\bSRI\b',
    'SERVICIO DE CONTRATACION',r'\bINSTITUTO\b',r'\bAGENCIA\b','DIRECCION DISTRITAL',
    'COORDINACION ZONAL','PARQUE NACIONAL',r'\bSENAE\b','REGISTRO CIVIL','DIGERCIC',
    r'\bIESS\b','SEGURIDAD SOCIAL','SUPERINTENDENCIA','DEFENSORIA','FISCALIA',
    'CONSEJO DE LA JUDICATURA',r'\bCORTE\b','ASAMBLEA','PRESIDENCIA','VICEPRESIDENCIA',
    'GOBERNACION',r'\bMIES\b','DESARROLLO HUMANO','CONSEJO NACIONAL','SERVICIO NACIONAL',
    r'\bSENPLADES\b','CONTRALORIA','AVIACION CIVIL',r'\bDGAC\b','DIRECCION GENERAL',
    'DIRECCION NACIONAL','DIRECCION PROVINCIAL','TRIBUNAL','PROCURADURIA',
    'CONSEJO DE PARTICIPACION']),
]

def clasificar(ruc, nombre_fallback):
    if ruc in MANUAL:
        return MANUAL[ruc], 'manual', 'RUC verificado'
    # se prueban todos los nombres historicos; gana la regla de menor indice
    cands = [norm(x) for x in POR_RUC.get(ruc, [])] or [norm(nombre_fallback)]
    mejor = None
    for i, (tipo, pats) in enumerate(REGLAS):
        for pat in pats:
            for nm in cands:
                if re.search(pat, nm):
                    mejor = (i, tipo, pat); break
            if mejor: break
        if mejor: break
    return (mejor[1], 'patron', mejor[2]) if mejor else ('Otros', 'residuo', '-')

dim = con.execute("SELECT * FROM de").df()          # trae el tipo de la corrida anterior
dim['ruc'] = dim['ruc'].astype(str)
dim = dim.rename(columns={'tipo_entidad': 'tipo_previo'})
dim = dim.drop(columns=[c for c in ['tipo_entidad_origen','entidad_nombre_canonico',
                                    'n_nombres_distintos','tipo_entidad_regla']
                        if c in dim.columns])
dim['entidad_nombre_canonico'] = [CANON.get(r, n) for r, n in zip(dim['ruc'], dim['entidad_nombre'])]
dim['n_nombres_distintos']     = [N_NOM.get(r, 1) for r in dim['ruc']]
res = [clasificar(r, n) for r, n in zip(dim['ruc'], dim['entidad_nombre_canonico'])]
dim['tipo_entidad']        = [a for a, _, _ in res]
dim['tipo_entidad_origen'] = [b for _, b, _ in res]
dim['tipo_entidad_regla']  = [c for _, _, c in res]

con.register('tip', dim[['ruc','tipo_entidad','tipo_entidad_origen','tipo_entidad_regla',
                         'entidad_nombre_canonico']])

print('\n' + '='*110)
print('(A) CLASIFICACION FINAL, MEDIDA SOBRE LOS PROCESOS')
print('='*110)
print(con.execute("""
SELECT t.tipo_entidad AS tipo, count(DISTINCT pm.entidad_ruc) AS entidades,
       count(*) AS procesos,
       round(100.0*count(*)/(SELECT count(*) FROM pm), 2) AS pct_procesos,
       round(sum(pm.monto_final)/1e6, 1) AS millones,
       round(100.0*sum(pm.monto_final)/(SELECT sum(monto_final) FROM pm), 2) AS pct_monto,
       round(median(pm.monto_final), 0) AS mediana_usd,
       round(100.0*avg(CASE WHEN ix.ind_oferente_unico THEN 1.0 ELSE 0.0 END), 2) AS pct_oferente_unico
FROM pm JOIN tip t ON t.ruc = pm.entidad_ruc
        LEFT JOIN ix ON ix.ocid = pm.ocid
GROUP BY 1 ORDER BY millones DESC
""").df().to_string(index=False))
print('\norigen de la asignacion:')
print(dim.groupby('tipo_entidad_origen').size().rename('entidades').to_frame().to_string())

print('\n' + '='*110)
print('(B) QUE SE MOVIO RESPECTO A LA CORRIDA ANTERIOR (entidades)')
print('='*110)
mv = pd.crosstab(dim['tipo_previo'].fillna('-'), dim['tipo_entidad'])
print(mv.to_string())
print(f"\nentidades que cambiaron de tipo: "
      f"{int((dim['tipo_previo'].fillna('-') != dim['tipo_entidad']).sum()):,}")

print('\n' + '='*110)
print('(C) AUDITORIA: que patron clasifico a cada entidad, por peso en monto')
print('='*110)
aud = con.execute("""
SELECT t.tipo_entidad AS tipo, t.tipo_entidad_regla AS patron,
       count(DISTINCT pm.entidad_ruc) AS entidades, count(*) AS procesos,
       round(sum(pm.monto_final)/1e6, 1) AS millones
FROM pm JOIN tip t ON t.ruc = pm.entidad_ruc
WHERE t.tipo_entidad_origen = 'patron'
GROUP BY 1, 2 ORDER BY millones DESC LIMIT 30
""").df()
print(aud.to_string(index=False))

print('\n-- los dos patrones que mas me preocupan: que capturaron exactamente --')
for pat in [r'\bSA\b', r'\bEP\b']:
    sub = con.execute(f"""
    SELECT t.entidad_nombre_canonico AS entidad, count(*) AS procesos,
           round(sum(pm.monto_final)/1e6, 1) AS millones
    FROM pm JOIN tip t ON t.ruc = pm.entidad_ruc
    WHERE t.tipo_entidad_regla = '{pat}'
    GROUP BY 1 ORDER BY millones DESC LIMIT 12
    """).df()
    print(f'\npatron {pat}  ({len(sub)} mostradas)')
    print(sub.to_string(index=False) if len(sub) else '  (ninguna)')

print('\n' + '='*110)
print('(D) RESIDUO')
print('='*110)
r = con.execute("""
SELECT count(DISTINCT pm.entidad_ruc) AS entidades, count(*) AS procesos,
       round(100.0*count(*)/(SELECT count(*) FROM pm), 2) AS pct_procesos,
       round(sum(pm.monto_final)/1e6, 1) AS millones,
       round(100.0*sum(pm.monto_final)/(SELECT sum(monto_final) FROM pm), 2) AS pct_monto
FROM pm JOIN tip t ON t.ruc = pm.entidad_ruc WHERE t.tipo_entidad = 'Otros'
""").df()
print(r.to_string(index=False))
print('referencia: el CASE en linea de la celda 44 dejaba 920 entidades, '
      '6,28% de procesos y 3,18% del monto. Esto tiene que quedar por debajo.')
print('\n-- las 15 mas grandes que siguen sin clasificar --')
print(con.execute("""
SELECT t.entidad_nombre_canonico AS entidad, pm.entidad_ruc AS ruc,
       count(*) AS procesos, round(sum(pm.monto_final)/1e6, 1) AS millones
FROM pm JOIN tip t ON t.ruc = pm.entidad_ruc
WHERE t.tipo_entidad = 'Otros'
GROUP BY 1, 2 ORDER BY millones DESC LIMIT 15
""").df().to_string(index=False))

# --- escritura ------------------------------------------------------------
print('\n' + '='*110)
print('(E) ESCRITURA Y FIRMA')
print('='*110)
def sha16(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(1 << 20), b''): h.update(b)
    return h.hexdigest()[:16]

salida = dim.drop(columns=['tipo_previo'])
con.register('dimout', salida)
con.execute(f"COPY dimout TO '{(PA/'planA_dim_entidad.parquet').as_posix()}' "
            f"(FORMAT PARQUET, COMPRESSION ZSTD)")
print(f'  planA_dim_entidad.parquet | {salida.shape[0]:,} filas, {salida.shape[1]} columnas '
      f'| sha256_16 = {sha16(PA / "planA_dim_entidad.parquet")}')

man = []
for p in sorted(PA.glob('planA_*.parquet')):
    f = con.execute(f"SELECT count(*) FROM read_parquet('{p.as_posix()}')").fetchone()[0]
    c = len(con.execute(f"DESCRIBE SELECT * FROM read_parquet('{p.as_posix()}')").df())
    man.append(dict(archivo=p.name, filas=f, columnas=c,
                    MB=round(p.stat().st_size/1e6, 1), sha256_16=sha16(p)))
man = pd.DataFrame(man); print('\n' + man.to_string(index=False))
man.insert(0, 'fecha', datetime.date.today().isoformat())
man.to_csv(DOC / 'manifiesto.csv', index=False, encoding='utf-8')
print('\nPega aqui toda la salida.')

In [ ]:
# ===========================================================================
# CELDA 49 — TIPOLOGIA INSTITUCIONAL, VERSION DEFINITIVA.
#            Reemplaza a las celdas 47 y 48. Autocontenida y reproducible.
# ===========================================================================
import duckdb, pandas as pd, numpy as np, re, unicodedata, hashlib, datetime, pathlib, shutil

BASE = pathlib.Path(r'C:\SERCOP_FINAL')
PA   = BASE / 'plan_A'
DOC  = PA / 'documentacion';  DOC.mkdir(exist_ok=True)
VER  = PA / '_versiones';     VER.mkdir(exist_ok=True)
con  = duckdb.connect()
pd.set_option('display.width', 235); pd.set_option('display.max_columns', 60)

# ---- la decision discutible, en un solo lugar y a la vista ---------------
TIPO_BOMBEROS = 'Gobierno local'   # cambiar a 'Seguridad y defensa' si se prefiere

for v, f in [('pm','planA_procesos_modelo'), ('de','planA_dim_entidad'),
             ('ix','planA_indicadores'), ('dp','planA_dim_proveedor')]:
    con.execute(f"CREATE OR REPLACE VIEW {v} AS SELECT * FROM "
                f"read_parquet('{(PA/(f+'.parquet')).as_posix()}')")

cols_dp = list(con.execute("DESCRIBE SELECT * FROM dp").df()['column_name'])
col_prov_ruc = next((c for c in ['ruc','proveedor_ruc'] if c in cols_dp), cols_dp[0])
print('clave de dim_proveedor:', col_prov_ruc)

# ---------------------------------------------------------------------------
# (A) LAS EMPRESAS PRIVADAS DEL LADO COMPRADOR: ¿SON REALES O SON EL DEFECTO
#     CONOCIDO DE IDS DE buyer/parties?
# ---------------------------------------------------------------------------


print('\n' + '='*110)
print('(A) RUC QUE APARECEN A LA VEZ COMO ENTIDAD COMPRADORA Y COMO PROVEEDOR')
print('='*110)
print(con.execute(f"""
WITH comp AS (
  SELECT entidad_ruc AS ruc, max(entidad_nombre) AS nombre,
         count(*) AS procesos_como_comprador,
         round(sum(monto_final)/1e6, 2) AS mill_como_comprador
  FROM pm GROUP BY 1
)
SELECT c.nombre, c.ruc, c.procesos_como_comprador, c.mill_como_comprador,
       (p.{col_prov_ruc} IS NOT NULL) AS tambien_es_proveedor
FROM comp c LEFT JOIN dp p ON p.{col_prov_ruc} = c.ruc
WHERE upper(c.nombre) SIMILAR TO '.*(EASYNET|QUITOLINDO|LA CUADRA|HIDROVICTORIA|HOLDINGDINE|
      TRANSNEXA|HDINEAGROS|COSSFA|EDITOGRAN|UCEM|EXPLOCEN|TRANSNAVE).*'
ORDER BY c.mill_como_comprador DESC
""".replace('\n      ', '')).df().to_string(index=False))

tot = con.execute(f"""
SELECT count(*) AS ruc_comprador,
       sum(CASE WHEN p.{col_prov_ruc} IS NOT NULL THEN 1 ELSE 0 END) AS tambien_proveedor
FROM (SELECT DISTINCT entidad_ruc AS ruc FROM pm) c
LEFT JOIN dp p ON p.{col_prov_ruc} = c.ruc
""").df()
print('\nen total, sobre todas las entidades compradoras:')
print(tot.to_string(index=False))
print("""Un RUC en los dos lados no es por si mismo un error: CELEC le vende a CNEL y las dos
publican. Lo que habria que mirar es si una empresa claramente privada aparece comprando,
porque eso apuntaria al defecto de ids de buyer/parties ya documentado en esta fuente.""")
# ¿el lado comprador de esos RUC es marginal frente a su lado proveedor?
print('\n-- los 15 RUC que aparecen en los dos lados, ordenados por desbalance --')
print(con.execute(f"""
WITH comp AS (SELECT entidad_ruc AS ruc, max(entidad_nombre) AS nombre,
                     count(*) AS proc_comprando,
                     round(sum(monto_final)/1e6, 2) AS mill_comprando
              FROM pm GROUP BY 1),
     prov AS (SELECT proveedor_ruc AS ruc, count(*) AS proc_vendiendo,
                     round(sum(monto_final)/1e6, 2) AS mill_vendiendo
              FROM pm GROUP BY 1)
SELECT c.nombre, c.ruc, c.proc_comprando, c.mill_comprando,
       p.proc_vendiendo, p.mill_vendiendo,
       round(p.mill_vendiendo / nullif(c.mill_comprando, 0), 1) AS veces_mas_vendiendo
FROM comp c JOIN prov p ON p.ruc = c.ruc
ORDER BY veces_mas_vendiendo DESC NULLS LAST LIMIT 15
""").df().to_string(index=False))
# ---------------------------------------------------------------------------
# (B) LAS REGLAS
# ---------------------------------------------------------------------------
def norm(s):
    if s is None or (isinstance(s, float) and pd.isna(s)): return ''
    s = unicodedata.normalize('NFKD', str(s)).encode('ascii', 'ignore').decode()
    return re.sub(r'\s+', ' ', s.upper().replace('.', '')).strip()

MANUAL = {'1768011230001': 'Seguridad y defensa',   # DIGMAT, Armada del Ecuador (SRI)
          '1360066490001': 'Educacion'}            # Coord. Zonal 4 del MINEDUC (codigo del proceso)

REGLAS = [
 ('Banca publica', ['BANECUADOR','BANCO CENTRAL','CORPORACION FINANCIERA NACIONAL',
    r'\bCFN\b',r'\bBIESS\b','BANCO NACIONAL DE FOMENTO','BANCO DE DESARROLLO',
    'BANCO DEL ESTADO','BANCO DEL INSTITUTO']),
 ('Salud', ['HOSPITAL','CENTRO DE SALUD','CENTRO MEDICO','CENTRO DE ESPECIALIDADES',
    'DIALISIS','MINISTERIO DE SALUD',r'\bMSP\b',r'DISTRITO.*SALUD',r'DIRECCION.*SALUD',
    r'ZONAL.*SALUD',r'UNIDAD.*SALUD',r'\bARCSA\b',r'\bSOLCA\b','MATERNIDAD','DISPENSARIO',
    'INSTITUTO NACIONAL DE DONACION','ONCOLOG','PSIQUIATR','INSTITUTO NACIONAL DE SALUD']),
 ('Seguridad y defensa', ['COMANDANCIA','COMANDO',r'\bEJERCITO\b','CUERPO DE INGENIEROS',
    'FUERZA TERRESTRE','FUERZA AEREA','FUERZA NAVAL',r'\bFAE\b',r'\bARMADA\b','SUBZONA',
    'POLICIA','MINISTERIO DE DEFENSA',r'\bDIGMAT\b',r'\bFAME\b','SEGURIDAD CIUDADANA',
    'ECU 911','ECU911','ANTIDROGAS','DIRECCION DE MOVILIZACION','SERVICIO DE DRAGAS',
    'COMISION DE TRANSITO',r'\bCTE\b',r'\bISSFA\b',r'\bISSPOL\b','MINISTERIO DEL INTERIOR',
    r'\bSNAI\b','REHABILITACION SOCIAL',
    # unidades militares que el residuo dejo ver
    'ESTACION NAVAL','BASE NAVAL','DISTRITO NAVAL','BRIGADA','BATALLON','ALA DE COMBATE',
    'GRUPO AEREO','CAPITANIA DE PUERTO','ESCUELA MILITAR','ESCUELA NAVAL',
    'ESCUELA SUPERIOR DE POLICIA','FUERZAS ARMADAS']),
 ('Educacion superior', ['UNIVERSIDAD','ESCUELA SUPERIOR POLITECNICA','ESCUELA POLITECNICA',
    r'\bESPOL\b',r'\bESPE\b',r'\bULEAM\b',r'\bUTM\b',r'\bUTE\b','SENESCYT',
    'EDUCACION SUPERIOR','INSTITUTO SUPERIOR','INSTITUTO TECNOLOGICO SUPERIOR',
    r'\bYACHAY\b',r'\bIKIAM\b']),
 ('Educacion', ['MINISTERIO DE EDUCACION',r'\bMINEDUC\b',r'DISTRITO.*EDUCACION',
    r'ZONAL.*EDUCACION',r'DIRECCION.*EDUCACION','UNIDAD EDUCATIVA','COLEGIO',
    'ESCUELA FISCAL']),
 ('Empresa publica', ['EMPRESA PUBLICA','EMPRESA MUNICIPAL',r'\bEP\b',r'\bEMMOP\b',
    r'\bEMUVI\b',r'\bEMAPA\b',r'\bETAPA\b',r'\bCELEC\b',r'\bCNEL\b',r'\bCNT\b',
    'PETROECUADOR',r'\bANDEC\b','AUSTROGAS','AUTORIDAD PORTUARIA','ELECTRO',
    'ECONOMIA MIXTA','TERMINAL TERRESTRE','CORPORACION ELECTRICA',
    'CORPORACION NACIONAL DE TELECOMUNICACIONES','ELECTRICA','AGUA POTABLE',
    'ALCANTARILLADO','MERCADO MAYORISTA',r'\bCEM\b','FERROCARRILES',
    'CORREOS DEL ECUADOR',r'\bFLOPEC\b',r'\bTAME\b','ASTILLEROS',
    # en lugar de \bSA\b, que capturaba empresas privadas: la lista explicita
    'HOLDINGDINE','HDINE','COSSFA','EXPLOCEN','UCEM','UNION CEMENTERA','TRANSNEXA',
    'HIDROVICTORIA','EDITOGRAN','TRANSNAVE',r'\bCENACE\b','OPERADOR NACIONAL DE ELECTRICIDAD',
    'CORPORACION AEROPORTUARIA','FABRICAMOS ECUADOR']),
 ('Gobierno local', ['MUNICIP',r'\bGADM?\b','GOBIERNO AUTONOMO','PREFECTURA',
    'CONSEJO PROVINCIAL','GOBIERNO PROVINCIAL','GOB PROV','JUNTA PARROQUIAL',
    'DISTRITO METROPOLITANO',r'\bCANTON\b','MANCOMUNIDAD',
    'REGIMEN ESPECIAL DE GALAPAGOS','CORPOSUCUMBIOS','PATRONATO',
    r'^GOBIERNO\b','FUNDACION MUNICIPAL','FUNDACION MUSEOS','MALECON 2000']),
 ('Gobierno central', ['MINISTERIO','SECRETARIA','SERVICIO DE RENTAS',r'\bSRI\b',
    'SERVICIO DE CONTRATACION',r'\bINSTITUTO\b',r'\bAGENCIA\b','DIRECCION DISTRITAL',
    'COORDINACION ZONAL','PARQUE NACIONAL',r'\bSENAE\b','REGISTRO CIVIL','DIGERCIC',
    r'\bIESS\b','SEGURIDAD SOCIAL','SUPERINTENDENCIA','DEFENSORIA','FISCALIA',
    'CONSEJO DE LA JUDICATURA',r'\bCORTE\b','ASAMBLEA','PRESIDENCIA','VICEPRESIDENCIA',
    'GOBERNACION',r'\bMIES\b','DESARROLLO HUMANO','CONSEJO NACIONAL','SERVICIO NACIONAL',
    r'\bSENPLADES\b','CONTRALORIA','AVIACION CIVIL',r'\bDGAC\b','DIRECCION GENERAL',
    'DIRECCION NACIONAL','DIRECCION PROVINCIAL','TRIBUNAL','PROCURADURIA',
    'CONSEJO DE PARTICIPACION',r'\bMIDUVI\b','PAGO DE ACCIDENTES',r'\bSPPAT\b',
    'FEDERACION DEPORTIVA','COMITE OLIMPICO','CASA DE LA CULTURA']),
]
# los bomberos van donde diga la variable del inicio
REGLAS = [(t, p + (['CUERPO DE BOMBEROS'] if t == TIPO_BOMBEROS else [])) for t, p in REGLAS]
print(f'\nlos cuerpos de bomberos se clasifican como: {TIPO_BOMBEROS}')

# ---- nombres historicos: la clasificacion usa TODOS los del RUC ----------
nombres = con.execute("""
WITH n AS (SELECT entidad_ruc, entidad_nombre, max(fecha_inicio_tender) AS ult,
                  count(*) AS proc FROM pm GROUP BY 1, 2)
SELECT entidad_ruc, entidad_nombre, proc,
       row_number() OVER (PARTITION BY entidad_ruc ORDER BY ult DESC, proc DESC) AS rn
FROM n
""").df()
nombres['entidad_ruc'] = nombres['entidad_ruc'].astype(str)
POR_RUC = nombres.groupby('entidad_ruc')['entidad_nombre'].apply(list).to_dict()
CANON   = nombres[nombres.rn == 1].set_index('entidad_ruc')['entidad_nombre'].to_dict()
N_NOM   = nombres.groupby('entidad_ruc').size().to_dict()

def clasificar(ruc, fallback):
    if ruc in MANUAL: return MANUAL[ruc], 'manual', 'RUC verificado'
    cands = [norm(x) for x in POR_RUC.get(ruc, [])] or [norm(fallback)]
    for tipo, pats in REGLAS:
        for pat in pats:
            for nm in cands:
                if re.search(pat, nm): return tipo, 'patron', pat
    return 'Otros', 'residuo', '-'

dim = con.execute("SELECT * FROM de").df()
dim['ruc'] = dim['ruc'].astype(str)
dim = dim.rename(columns={'tipo_entidad': 'tipo_previo'})
dim = dim.drop(columns=[c for c in ['tipo_entidad_origen','tipo_entidad_regla',
                                    'entidad_nombre_canonico','n_nombres_distintos']
                        if c in dim.columns])
dim['entidad_nombre_canonico'] = [CANON.get(r, n) for r, n in zip(dim['ruc'], dim['entidad_nombre'])]
dim['n_nombres_distintos']     = [N_NOM.get(r, 1) for r in dim['ruc']]
res = [clasificar(r, n) for r, n in zip(dim['ruc'], dim['entidad_nombre_canonico'])]
dim['tipo_entidad'], dim['tipo_entidad_origen'], dim['tipo_entidad_regla'] = \
    [a for a,_,_ in res], [b for _,b,_ in res], [c for _,_,c in res]
con.register('tip', dim[['ruc','tipo_entidad','tipo_entidad_origen','tipo_entidad_regla',
                         'entidad_nombre_canonico']])
print('\n' + '='*110)
print('(C) CLASIFICACION FINAL')
print('='*110)
print(con.execute("""
SELECT t.tipo_entidad AS tipo, count(DISTINCT pm.entidad_ruc) AS entidades, count(*) AS procesos,
       round(100.0*count(*)/(SELECT count(*) FROM pm), 2) AS pct_procesos,
       round(sum(pm.monto_final)/1e6, 1) AS millones,
       round(100.0*sum(pm.monto_final)/(SELECT sum(monto_final) FROM pm), 2) AS pct_monto,
       round(median(pm.monto_final), 0) AS mediana_usd,
       round(100.0*avg(CASE WHEN ix.ind_oferente_unico THEN 1.0 ELSE 0.0 END), 2) AS pct_oferente_unico,
       round(100.0*avg(CASE WHEN ix.ind_fraccionamiento_marca THEN 1.0 ELSE 0.0 END), 2) AS pct_antecedente_30d
FROM pm JOIN tip t ON t.ruc = pm.entidad_ruc LEFT JOIN ix ON ix.ocid = pm.ocid
GROUP BY 1 ORDER BY millones DESC
""").df().to_string(index=False))

print('\norigen:', dim.groupby('tipo_entidad_origen').size().to_dict())

print('\n-- patrones nuevos: que entro por cada uno --')
print(con.execute("""
SELECT t.tipo_entidad AS tipo, t.tipo_entidad_regla AS patron,
       count(DISTINCT pm.entidad_ruc) AS entidades, count(*) AS procesos,
       round(sum(pm.monto_final)/1e6, 1) AS millones
FROM pm JOIN tip t ON t.ruc = pm.entidad_ruc
WHERE t.tipo_entidad_regla IN ('ESTACION NAVAL','BASE NAVAL','BRIGADA','BATALLON',
      'CAPITANIA DE PUERTO','FUERZAS ARMADAS','CUERPO DE BOMBEROS','^GOBIERNO\\b',
      'FUNDACION MUSEOS','MALECON 2000','\\bMIDUVI\\b','FEDERACION DEPORTIVA',
      '\\bCENACE\\b','HOLDINGDINE','COSSFA','EXPLOCEN','UCEM','TRANSNAVE',
      'CORPORACION AEROPORTUARIA','PAGO DE ACCIDENTES')
GROUP BY 1, 2 ORDER BY millones DESC
""").df().to_string(index=False))

print('\n' + '='*110)
print('(D) CAMBIOS RESPECTO A LA CORRIDA ANTERIOR (entidades)')
print('='*110)
print(pd.crosstab(dim['tipo_previo'].fillna('-'), dim['tipo_entidad']).to_string())
print(f"\ncambiaron de tipo: {int((dim['tipo_previo'].fillna('-') != dim['tipo_entidad']).sum()):,}")

print('\n' + '='*110)
print('(E) RESIDUO')
print('='*110)
print(con.execute("""
SELECT count(DISTINCT pm.entidad_ruc) AS entidades, count(*) AS procesos,
       round(100.0*count(*)/(SELECT count(*) FROM pm), 2) AS pct_procesos,
       round(sum(pm.monto_final)/1e6, 1) AS millones,
       round(100.0*sum(pm.monto_final)/(SELECT sum(monto_final) FROM pm), 2) AS pct_monto
FROM pm JOIN tip t ON t.ruc = pm.entidad_ruc WHERE t.tipo_entidad = 'Otros'
""").df().to_string(index=False))
print('referencia a batir: 788 entidades, 5,18% de procesos, 1,60% del monto')
print('\n-- las 15 mas grandes que siguen sin clasificar --')
print(con.execute("""
SELECT t.entidad_nombre_canonico AS entidad, pm.entidad_ruc AS ruc,
       count(*) AS procesos, round(sum(pm.monto_final)/1e6, 1) AS millones
FROM pm JOIN tip t ON t.ruc = pm.entidad_ruc WHERE t.tipo_entidad = 'Otros'
GROUP BY 1, 2 ORDER BY millones DESC LIMIT 15
""").df().to_string(index=False))

# ---------------------------------------------------------------------------
# (F) ESCRITURA, FIRMA Y ENMIENDA
# ---------------------------------------------------------------------------
print('\n' + '='*110)
print('(F) ESCRITURA Y FIRMA')
print('='*110)
def sha16(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(1 << 20), b''): h.update(b)
    return h.hexdigest()[:16]

salida = dim.drop(columns=['tipo_previo'])
con.register('dimout', salida)
con.execute(f"COPY dimout TO '{(PA/'planA_dim_entidad.parquet').as_posix()}' "
            f"(FORMAT PARQUET, COMPRESSION ZSTD)")
print(f'  planA_dim_entidad.parquet | {salida.shape[0]:,} filas, {salida.shape[1]} columnas '
      f'| sha256_16 = {sha16(PA / "planA_dim_entidad.parquet")}')

man = []
for p in sorted(PA.glob('planA_*.parquet')):
    f = con.execute(f"SELECT count(*) FROM read_parquet('{p.as_posix()}')").fetchone()[0]
    c = len(con.execute(f"DESCRIBE SELECT * FROM read_parquet('{p.as_posix()}')").df())
    man.append(dict(archivo=p.name, filas=f, columnas=c,
                    MB=round(p.stat().st_size/1e6, 1), sha256_16=sha16(p)))
man = pd.DataFrame(man); print('\n' + man.to_string(index=False))
man.insert(0, 'fecha', datetime.date.today().isoformat()); man.to_csv(DOC/'manifiesto.csv', index=False, encoding='utf-8')

dim[['ruc','entidad_nombre_canonico','n_nombres_distintos','tipo_entidad',
     'tipo_entidad_origen','tipo_entidad_regla']].sort_values(
     ['tipo_entidad','entidad_nombre_canonico']).to_csv(
     DOC/'tipologia_entidades.csv', index=False, encoding='utf-8')
print(f'  auditoria completa en documentacion/tipologia_entidades.csv')

with open(DOC/'enmiendas.md', 'a', encoding='utf-8') as f:
    f.write(f"\n## Enmienda tipologia institucional (definitiva) — {datetime.date.today().isoformat()}\n"
            f"- planA_dim_entidad.parquet gana: tipo_entidad, tipo_entidad_origen\n"
            f"  (manual/patron/residuo), tipo_entidad_regla (el patron exacto que clasifico,\n"
            f"  para auditoria), entidad_nombre_canonico y n_nombres_distintos.\n"
            f"- La tipologia se asigna por RUC y se evalua contra TODOS los nombres historicos\n"
            f"  del RUC, no contra uno elegido: 335 RUC tienen mas de un nombre y en al menos\n"
            f"  un caso (1360070840001) el nombre elegido decidia mal la clasificacion.\n"
            f"- Dos asignaciones verificadas una por una: 1768011230001 DIGMAT = Direccion\n"
            f"  General del Material (Armada) -> Seguridad y defensa; 1360066490001\n"
            f"  Coordinacion Zonal 4 = Ministerio de Educacion -> Educacion.\n"
            f"- Criterio declarado: las entidades se agrupan por su PERFIL DE COMPRA, no por\n"
            f"  su dependencia administrativa. Por eso las empresas publicas municipales de\n"
            f"  agua y electricidad son 'Empresa publica' y no 'Gobierno local', y los cuerpos\n"
            f"  de bomberos son '{TIPO_BOMBEROS}'.\n"
            f"- Se retiro el patron \\bSA\\b: capturaba empresas privadas. Las empresas estatales\n"
            f"  que terminan en S.A. se nombran explicitamente.\n"
            f"- Clasificacion heuristica sobre el nombre salvo las verificadas; se declara como\n"
            f"  tal. Sirve para formar grupos comparables, no es dato oficial.\n"
            f"- Firma planA_dim_entidad.parquet = {sha16(PA/'planA_dim_entidad.parquet')}\n")
print('  enmienda registrada')
print('\nPega aqui toda la salida.')


In [ ]:
# ===========================================================================
# CELDA 50 — TIPOLOGIA: cierre. Las reglas pasan a un JSON versionable y la
#            celda las lee de ahi. Incluye criterio de parada declarado.
# ===========================================================================
import duckdb, pandas as pd, numpy as np, re, unicodedata, hashlib, datetime, pathlib, json

BASE = pathlib.Path(r'C:\SERCOP_FINAL'); PA = BASE / 'plan_A'
DOC  = PA / 'documentacion'; DOC.mkdir(exist_ok=True)
con  = duckdb.connect(); pd.set_option('display.width', 235); pd.set_option('display.max_columns', 60)
for v, f in [('pm','planA_procesos_modelo'), ('de','planA_dim_entidad'), ('ix','planA_indicadores')]:
    con.execute(f"CREATE OR REPLACE VIEW {v} AS SELECT * FROM "
                f"read_parquet('{(PA/(f+'.parquet')).as_posix()}')")

# ---------------------------------------------------------------------------
# (A) LAS REGLAS, COMO INSUMO Y NO COMO CODIGO
# ---------------------------------------------------------------------------
REGLAS_JSON = {
 "version": "1.0",
 "fecha": datetime.date.today().isoformat(),
 "criterio": ("Las entidades se agrupan por su PERFIL DE COMPRA, no por su dependencia "
              "administrativa. Se evalua contra todos los nombres historicos del RUC y gana "
              "la primera regla de la lista que coincida con cualquiera de ellos."),
 "cuerpos_de_bomberos_en": "Gobierno local",
 "overrides_por_ruc": {
   "1768011230001": ["Seguridad y defensa", "DIGMAT = Direccion General del Material, Armada del Ecuador (verificado en SRI y Min. Defensa)"],
   "1360066490001": ["Educacion", "Coordinacion Zonal 4 del Ministerio de Educacion (verificado por el codigo de proceso SIE-MINEDUC-CZ4)"]
 },
 "orden_y_patrones": [
  ["Banca publica", ["BANECUADOR","BANCO CENTRAL","CORPORACION FINANCIERA NACIONAL",
    "\\bCFN\\b","\\bBIESS\\b","BANCO NACIONAL DE FOMENTO","BANCO DE DESARROLLO",
    "BANCO DEL ESTADO","BANCO DEL INSTITUTO","FINANZAS POPULARES","\\bCONAFIPS\\b"]],
  ["Salud", ["HOSPITAL","CENTRO DE SALUD","CENTRO MEDICO","CENTRO DE ESPECIALIDADES",
    "DIALISIS","MINISTERIO DE SALUD","\\bMSP\\b","DISTRITO.*SALUD","DIRECCION.*SALUD",
    "ZONAL.*SALUD","UNIDAD.*SALUD","\\bARCSA\\b","\\bSOLCA\\b","MATERNIDAD","DISPENSARIO",
    "INSTITUTO NACIONAL DE DONACION","ONCOLOG","PSIQUIATR","INSTITUTO NACIONAL DE SALUD"]],
  ["Seguridad y defensa", ["COMANDANCIA","COMANDO","\\bEJERCITO\\b","CUERPO DE INGENIEROS",
    "FUERZA TERRESTRE","FUERZA AEREA","FUERZA NAVAL","\\bFAE\\b","\\bFAEALA","\\bARMADA\\b",
    "SUBZONA","POLICIA","MINISTERIO DE DEFENSA","\\bDIGMAT\\b","\\bFAME\\b",
    "SEGURIDAD CIUDADANA","ECU 911","ECU911","ANTIDROGAS","DIRECCION DE MOVILIZACION",
    "SERVICIO DE DRAGAS","COMISION DE TRANSITO","\\bCTE\\b","\\bISSFA\\b","\\bISSPOL\\b",
    "MINISTERIO DEL INTERIOR","\\bSNAI\\b","REHABILITACION SOCIAL","ESTACION NAVAL",
    "BASE NAVAL","DISTRITO NAVAL","AERONAVAL","BRIGADA","BATALLON","\\bDIVISION\\b",
    "CABALLERIA","ALA DE COMBATE","GRUPO AEREO","CAPITANIA DE PUERTO","ESCUELA MILITAR",
    "ESCUELA NAVAL","ESCUELA SUPERIOR MIL","ESCUELA SUPERIOR DE POLICIA","FUERZAS ARMADAS"]],
  ["Educacion superior", ["UNIVERSIDAD","ESCUELA SUPERIOR POLITECNICA","ESCUELA POLITECNICA",
    "\\bESPOL\\b","\\bESPE\\b","\\bULEAM\\b","\\bUTM\\b","\\bUTE\\b","SENESCYT",
    "EDUCACION SUPERIOR","INSTITUTO SUPERIOR","INSTITUTO TECNOLOGICO SUPERIOR",
    "\\bYACHAY\\b","\\bIKIAM\\b"]],
  ["Educacion", ["MINISTERIO DE EDUCACION","\\bMINEDUC\\b","DISTRITO.*EDUCACION",
    "ZONAL.*EDUCACION","DIRECCION.*EDUCACION","UNIDAD EDUCATIVA","COLEGIO","ESCUELA FISCAL"]],
  ["Empresa publica", ["EMPRESA PUBLICA","EMPRESA MUNICIPAL","\\bEP\\b","\\bEMMOP\\b",
    "\\bEMUVI\\b","\\bEMAPA\\b","\\bETAPA\\b","\\bCELEC\\b","\\bCNEL\\b","\\bCNT\\b",
    "PETROECUADOR","\\bANDEC\\b","AUSTROGAS","AUTORIDAD PORTUARIA","ELECTRO",
    "ECONOMIA MIXTA","TERMINAL TERRESTRE","CORPORACION ELECTRICA",
    "CORPORACION NACIONAL DE TELECOMUNICACIONES","ELECTRICA","AGUA POTABLE","ALCANTARILLADO",
    "MERCADO MAYORISTA","\\bCEM\\b","FERROCARRILES","CORREOS DEL ECUADOR","\\bFLOPEC\\b",
    "\\bTAME\\b","ASTILLEROS","HOLDINGDINE","HDINE","COSSFA","EXPLOCEN","UCEM",
    "UNION CEMENTERA","TRANSNEXA","HIDROVICTORIA","EDITOGRAN","TRANSNAVE","\\bCENACE\\b",
    "OPERADOR NACIONAL DE ELECTRICIDAD","CORPORACION AEROPORTUARIA","FABRICAMOS ECUADOR",
    "\\bDIAF\\b","INDUSTRIA AERONAUTICA"]],
  ["Gobierno local", ["MUNICIP","\\bGADM?\\b","GOBIERNO AUTONOMO","PREFECTURA",
    "CONSEJO PROVINCIAL","GOBIERNO PROVINCIAL","GOB PROV","JUNTA PARROQUIAL",
    "DISTRITO METROPOLITANO","\\bCANTON\\b","MANCOMUNIDAD","REGIMEN ESPECIAL DE GALAPAGOS",
    "CORPOSUCUMBIOS","PATRONATO","^GOBIERNO\\b","FUNDACION MUNICIPAL","FUNDACION MUSEOS",
    "FUNDACION TEATRO","MALECON 2000","ADMINISTRACION ZONAL","DISTRITO LA DELICIA",
    "\\bCONGOPE\\b","CONSORCIO DE GOBIERNOS AUTONOMOS","FIESTA DE LA FRUTA",
    "SUCUMBIOS SOLIDARIO","CUERPO DE BOMBEROS"]],
  ["Gobierno central", ["MINISTERIO","SECRETARIA","SERVICIO DE RENTAS","\\bSRI\\b",
    "SERVICIO DE CONTRATACION","\\bINSTITUTO\\b","\\bAGENCIA\\b","DIRECCION DISTRITAL",
    "COORDINACION ZONAL","PARQUE NACIONAL","\\bSENAE\\b","REGISTRO CIVIL","DIGERCIC",
    "\\bIESS\\b","SEGURIDAD SOCIAL","SUPERINTENDENCIA","DEFENSORIA","FISCALIA",
    "CONSEJO DE LA JUDICATURA","\\bCORTE\\b","ASAMBLEA","PRESIDENCIA","VICEPRESIDENCIA",
    "GOBERNACION","\\bMIES\\b","DESARROLLO HUMANO","CONSEJO NACIONAL","SERVICIO NACIONAL",
    "\\bSENPLADES\\b","CONTRALORIA","AVIACION CIVIL","\\bDGAC\\b","DIRECCION GENERAL",
    "DIRECCION NACIONAL","DIRECCION PROVINCIAL","TRIBUNAL","PROCURADURIA",
    "CONSEJO DE PARTICIPACION","\\bMIDUVI\\b","PAGO DE ACCIDENTES","\\bSPPAT\\b",
    "FEDERACION DEPORTIVA","CONCENTRACION DEPORTIVA","COMITE OLIMPICO","CASA DE LA CULTURA",
    "ESTACION EXPERIMENTAL","\\bINIAP\\b"]],
 ],
 "pendientes_de_verificacion_manual": {
   "1768013010001": "DIRECCION DE LA ADMINISTRACION CENTRAL - revisar en el portal de que institucion depende",
   "1768063460001": "DISTRITO EUGENIO ESPEJO - puede ser el hospital (Salud) o la administracion zonal de Quito (Gobierno local)"
 }
}
RJ = DOC / 'reglas_tipologia.json'
RJ.write_text(json.dumps(REGLAS_JSON, ensure_ascii=False, indent=1), encoding='utf-8')
print(f'reglas escritas en {RJ}')

# ---- de aqui en adelante la celda LEE el json: el json es la fuente unica
R = json.loads(RJ.read_text(encoding='utf-8'))
REGLAS  = [(t, p) for t, p in R['orden_y_patrones']]
MANUAL  = {k: v[0] for k, v in R['overrides_por_ruc'].items()}
print(f'tipos: {len(REGLAS)} | patrones: {sum(len(p) for _, p in REGLAS)} | '
      f'overrides por RUC: {len(MANUAL)} | bomberos en: {R["cuerpos_de_bomberos_en"]}')

def norm(s):
    if s is None or (isinstance(s, float) and pd.isna(s)): return ''
    s = unicodedata.normalize('NFKD', str(s)).encode('ascii','ignore').decode()
    return re.sub(r'\s+', ' ', s.upper().replace('.', '')).strip()

nombres = con.execute("""
WITH n AS (SELECT entidad_ruc, entidad_nombre, max(fecha_inicio_tender) AS ult,
                  count(*) AS proc FROM pm GROUP BY 1, 2)
SELECT entidad_ruc, entidad_nombre, proc,
       row_number() OVER (PARTITION BY entidad_ruc ORDER BY ult DESC, proc DESC) AS rn FROM n
""").df()
nombres['entidad_ruc'] = nombres['entidad_ruc'].astype(str)
POR_RUC = nombres.groupby('entidad_ruc')['entidad_nombre'].apply(list).to_dict()
CANON   = nombres[nombres.rn == 1].set_index('entidad_ruc')['entidad_nombre'].to_dict()
N_NOM   = nombres.groupby('entidad_ruc').size().to_dict()

def clasificar(ruc, fallback):
    if ruc in MANUAL: return MANUAL[ruc], 'manual', 'RUC verificado'
    cands = [norm(x) for x in POR_RUC.get(ruc, [])] or [norm(fallback)]
    for tipo, pats in REGLAS:
        for pat in pats:
            for nm in cands:
                if re.search(pat, nm): return tipo, 'patron', pat
    return 'Otros', 'residuo', '-'

dim = con.execute("SELECT * FROM de").df()
dim['ruc'] = dim['ruc'].astype(str)
dim = dim.rename(columns={'tipo_entidad': 'tipo_previo'})
dim = dim.drop(columns=[c for c in ['tipo_entidad_origen','tipo_entidad_regla',
               'entidad_nombre_canonico','n_nombres_distintos'] if c in dim.columns])
dim['entidad_nombre_canonico'] = [CANON.get(r, n) for r, n in zip(dim['ruc'], dim['entidad_nombre'])]
dim['n_nombres_distintos']     = [N_NOM.get(r, 1) for r in dim['ruc']]
res = [clasificar(r, n) for r, n in zip(dim['ruc'], dim['entidad_nombre_canonico'])]
dim['tipo_entidad'], dim['tipo_entidad_origen'], dim['tipo_entidad_regla'] = \
    [a for a,_,_ in res], [b for _,b,_ in res], [c for _,_,c in res]
con.register('tip', dim[['ruc','tipo_entidad','tipo_entidad_origen','tipo_entidad_regla',
                         'entidad_nombre_canonico']])

print('\n' + '='*110); print('(B) CLASIFICACION FINAL'); print('='*110)
print(con.execute("""
SELECT t.tipo_entidad AS tipo, count(DISTINCT pm.entidad_ruc) AS entidades, count(*) AS procesos,
       round(100.0*count(*)/(SELECT count(*) FROM pm), 2) AS pct_procesos,
       round(sum(pm.monto_final)/1e6, 1) AS millones,
       round(100.0*sum(pm.monto_final)/(SELECT sum(monto_final) FROM pm), 2) AS pct_monto,
       round(median(pm.monto_final), 0) AS mediana_usd,
       round(100.0*avg(CASE WHEN ix.ind_oferente_unico THEN 1.0 ELSE 0.0 END), 2) AS pct_oferente_unico,
       round(100.0*avg(CASE WHEN ix.ind_fraccionamiento_marca THEN 1.0 ELSE 0.0 END), 2) AS pct_antecedente_30d
FROM pm JOIN tip t ON t.ruc = pm.entidad_ruc LEFT JOIN ix ON ix.ocid = pm.ocid
GROUP BY 1 ORDER BY millones DESC
""").df().to_string(index=False))
print('\norigen:', dim.groupby('tipo_entidad_origen').size().to_dict())

print('\n' + '='*110); print('(C) CRITERIO DE PARADA DECLARADO'); print('='*110)
k = con.execute("""
WITH tot AS (SELECT sum(monto_final) AS m FROM pm),
     o AS (SELECT pm.entidad_ruc, t.entidad_nombre_canonico AS nombre,
                  count(*) AS procesos, sum(pm.monto_final) AS monto
           FROM pm JOIN tip t ON t.ruc = pm.entidad_ruc
           WHERE t.tipo_entidad = 'Otros' GROUP BY 1, 2)
SELECT count(*) AS entidades_otros, sum(procesos) AS procesos_otros,
       round(100.0*sum(procesos)/(SELECT count(*) FROM pm), 2) AS pct_procesos,
       round(sum(monto)/1e6, 1) AS millones, round(100.0*sum(monto)/(SELECT m FROM tot), 2) AS pct_monto,
       round(100.0*max(monto)/(SELECT m FROM tot), 3) AS pct_monto_de_la_mayor
FROM o
""").df().iloc[0]
print(k.to_frame().T.to_string(index=False))
c1 = k['pct_monto'] <= 1.0; c2 = k['pct_monto_de_la_mayor'] <= 0.05
print(f"\n  residuo no pasa del 1% del monto ............ {'CUMPLE' if c1 else 'NO CUMPLE'} ({k['pct_monto']}%)")
print(f"  ninguna sin clasificar pasa del 0,05% ...... {'CUMPLE' if c2 else 'NO CUMPLE'} ({k['pct_monto_de_la_mayor']}%)")
print(f"  -> la tipologia {'SE DA POR CERRADA' if c1 and c2 else 'necesita otra ronda'}")

print('\n-- las 12 mas grandes que quedan en Otros --')
print(con.execute("""
SELECT t.entidad_nombre_canonico AS entidad, pm.entidad_ruc AS ruc, count(*) AS procesos,
       round(sum(pm.monto_final)/1e6, 1) AS millones
FROM pm JOIN tip t ON t.ruc = pm.entidad_ruc WHERE t.tipo_entidad = 'Otros'
GROUP BY 1, 2 ORDER BY millones DESC LIMIT 12
""").df().to_string(index=False))

print('\n' + '='*110); print('(D) CAMBIOS RESPECTO A LA CORRIDA ANTERIOR'); print('='*110)
print(pd.crosstab(dim['tipo_previo'].fillna('-'), dim['tipo_entidad']).to_string())
print(f"\ncambiaron de tipo: {int((dim['tipo_previo'].fillna('-') != dim['tipo_entidad']).sum()):,}")

print('\n' + '='*110); print('(E) ESCRITURA Y FIRMA'); print('='*110)
def sha16(p):
    h = hashlib.sha256()
    with open(p,'rb') as f:
        for b in iter(lambda: f.read(1<<20), b''): h.update(b)
    return h.hexdigest()[:16]

con.register('dimout', dim.drop(columns=['tipo_previo']))
con.execute(f"COPY dimout TO '{(PA/'planA_dim_entidad.parquet').as_posix()}' "
            f"(FORMAT PARQUET, COMPRESSION ZSTD)")
dim[['ruc','entidad_nombre_canonico','n_nombres_distintos','tipo_entidad',
     'tipo_entidad_origen','tipo_entidad_regla']].sort_values(
     ['tipo_entidad','entidad_nombre_canonico']).to_csv(
     DOC/'tipologia_entidades.csv', index=False, encoding='utf-8')

man = []
for p in sorted(PA.glob('planA_*.parquet')):
    f = con.execute(f"SELECT count(*) FROM read_parquet('{p.as_posix()}')").fetchone()[0]
    c = len(con.execute(f"DESCRIBE SELECT * FROM read_parquet('{p.as_posix()}')").df())
    man.append(dict(archivo=p.name, filas=f, columnas=c,
                    MB=round(p.stat().st_size/1e6,1), sha256_16=sha16(p)))
man = pd.DataFrame(man); print(man.to_string(index=False))
man.insert(0,'fecha',datetime.date.today().isoformat()); man.to_csv(DOC/'manifiesto.csv', index=False, encoding='utf-8')

with open(DOC/'enmiendas.md','a',encoding='utf-8') as f:
    f.write(f"\n## Tipologia institucional: cierre — {datetime.date.today().isoformat()}\n"
            f"- Las reglas dejan de vivir en codigo y pasan a documentacion/reglas_tipologia.json\n"
            f"  (version {R['version']}), que es la fuente unica: la celda lo escribe y lo lee.\n"
            f"- Criterio: {R['criterio']}\n"
            f"- Criterio de parada declarado: el residuo no pasa del 1% del monto y ninguna\n"
            f"  entidad sin clasificar pasa del 0,05% del monto total.\n"
            f"- Residuo alcanzado: {int(k['entidades_otros'])} entidades, {k['pct_procesos']}% de\n"
            f"  los procesos, {k['pct_monto']}% del monto; la mayor sin clasificar es el\n"
            f"  {k['pct_monto_de_la_mayor']}% del monto total.\n"
            f"- La clasificacion se evalua contra todos los nombres historicos del RUC.\n"
            f"- Hallazgo: 303 de 3774 RUC compradores aparecen tambien como proveedores. La\n"
            f"  revision de los mayores muestra que son empresas publicas universitarias y de\n"
            f"  institutos creadas para VENDER servicios al Estado, no un defecto de ids de\n"
            f"  buyer/parties. Quedan tres casos menores (0,4 MM) donde el campo comprador\n"
            f"  trae algo que no es un comprador.\n"
            f"- Firma planA_dim_entidad.parquet = {sha16(PA/'planA_dim_entidad.parquet')}\n")
print('\n  enmienda registrada | anexo: documentacion/tipologia_entidades.csv')
print('\nPega aqui toda la salida.')

In [ ]:
# ===========================================================================
# CELDA 51 — SOLO MIRA: la actividad economica CIIU del SRI frente a la
#            tipologia por nombre. No clasifica ni escribe nada.
# ===========================================================================
import duckdb, pandas as pd, pathlib, json, datetime

BASE = pathlib.Path(r'C:\SERCOP_FINAL'); PA = BASE / 'plan_A'; DOC = PA / 'documentacion'
con  = duckdb.connect(); pd.set_option('display.width', 250); pd.set_option('display.max_columns', 70)
pd.set_option('display.max_colwidth', 95)
for v, f in [('pm','planA_procesos_modelo'), ('de','planA_dim_entidad')]:
    con.execute(f"CREATE OR REPLACE VIEW {v} AS SELECT * FROM "
                f"read_parquet('{(PA/(f+'.parquet')).as_posix()}')")

# primero: los dos RUC que acabas de verificar entran al json de reglas
RJ = DOC / 'reglas_tipologia.json'
R  = json.loads(RJ.read_text(encoding='utf-8'))
R['overrides_por_ruc']['1768013010001'] = ["Seguridad y defensa",
  "Direccion de la Administracion Central - CIIU SRI: gestion de asuntos y fuerzas de defensa militar"]
R['overrides_por_ruc']['1768063460001'] = ["Seguridad y defensa",
  "Distrito Eugenio Espejo - CIIU SRI: fuerzas especiales de policia (NO es el hospital)"]
R['pendientes_de_verificacion_manual'] = {}
R['version'] = '1.1'; R['fecha'] = datetime.date.today().isoformat()
RJ.write_text(json.dumps(R, ensure_ascii=False, indent=1), encoding='utf-8')
print(f'reglas_tipologia.json -> version {R["version"]}, '
      f'{len(R["overrides_por_ruc"])} overrides verificados a mano\n')

print('='*120)
print('(A) FORMATO REAL DEL CODIGO CIIU Y COBERTURA')
print('='*120)
print('-- 15 valores crudos, para ver el formato antes de interpretarlo --')
print(con.execute("""
SELECT sri_CODIGO_CIIU AS codigo, max(sri_ACTIVIDAD_ECONOMICA) AS actividad, count(*) AS entidades
FROM de WHERE sri_CODIGO_CIIU IS NOT NULL
GROUP BY 1 ORDER BY entidades DESC LIMIT 15
""").df().to_string(index=False))

print('\n-- cobertura: entidades, y el peso de esas entidades en procesos y monto --')
print(con.execute("""
WITH f AS (
  SELECT d.ruc,
         CASE WHEN d.sri_CODIGO_CIIU IS NOT NULL THEN 'con CIIU'
              WHEN d.en_sri THEN 'en SRI pero sin CIIU'
              ELSE 'no esta en el archivo del SRI' END AS estado
  FROM de d
)
SELECT f.estado, count(DISTINCT f.ruc) AS entidades,
       count(pm.ocid) AS procesos,
       round(100.0*count(pm.ocid)/(SELECT count(*) FROM pm), 2) AS pct_procesos,
       round(sum(pm.monto_final)/1e6, 1) AS millones,
       round(100.0*coalesce(sum(pm.monto_final),0)/(SELECT sum(monto_final) FROM pm), 2) AS pct_monto
FROM f LEFT JOIN pm ON pm.entidad_ruc = f.ruc
GROUP BY 1 ORDER BY millones DESC NULLS LAST
""").df().to_string(index=False))

pd.set_option('display.max_colwidth', 72)   # la tabla de (A) salio ilegible de ancha

print('\n' + '='*120)
print('(B) LOS 25 CODIGOS CIIU QUE CONCENTRAN EL MONTO, Y QUE TIPO LES PUSE YO')
print('='*120)
print(con.execute("""
SELECT regexp_replace(d.sri_CODIGO_CIIU, '[^0-9]', '', 'g')[1:4] AS ciiu4,
       max(d.sri_ACTIVIDAD_ECONOMICA)[1:66] AS actividad_segun_el_sri,
       count(DISTINCT d.ruc) AS entidades, count(*) AS procesos,
       round(sum(pm.monto_final)/1e6, 1) AS millones,
       mode(d.tipo_entidad) AS tipo_dominante,
       count(DISTINCT d.tipo_entidad) AS tipos_distintos_que_le_puse
FROM pm JOIN de d ON d.ruc = pm.entidad_ruc
WHERE d.sri_CODIGO_CIIU IS NOT NULL
GROUP BY 1 ORDER BY millones DESC LIMIT 25
""").df().to_string(index=False))
print("""
tipos_distintos_que_le_puse es la columna que importa: si un codigo CIIU recibio un solo
tipo mio, mi heuristica y el SRI coinciden en ese codigo. Si recibio cuatro o cinco, yo
estaba repartiendo entidades de la misma actividad economica en categorias distintas.""")


print('\n' + '='*120)
print('(C) CRUCE: mi tipo por nombre (filas) contra la division CIIU del SRI (columnas)')
print('='*120)
cx = con.execute("""
SELECT d.tipo_entidad AS tipo_por_nombre,
       regexp_replace(d.sri_CODIGO_CIIU, '[^0-9]', '', 'g')[1:2] AS ciiu2,
       count(DISTINCT d.ruc) AS entidades
FROM de d WHERE d.sri_CODIGO_CIIU IS NOT NULL
GROUP BY 1, 2
""").df()
print(cx.pivot_table(index='tipo_por_nombre', columns='ciiu2', values='entidades',
                     aggfunc='sum', fill_value=0).to_string())
print("""
Referencia de las divisiones que importan: 84 administracion publica y defensa,
85 ensenanza, 86 salud, 35 electricidad y gas, 36 agua, 64 servicios financieros,
41-43 construccion, 49-52 transporte y almacenamiento, 94 asociaciones.
Si una fila se concentra en una sola columna, mi heuristica y el SRI coinciden.""")

print('\n-- el detalle dentro de la division 84, que es donde esta el grueso --')
print(con.execute("""
SELECT regexp_replace(d.sri_CODIGO_CIIU, '[^0-9]', '', 'g')[1:4] AS ciiu4,
       max(d.sri_ACTIVIDAD_ECONOMICA)[1:70] AS actividad,
       d.tipo_entidad AS tipo_por_nombre, count(DISTINCT d.ruc) AS entidades
FROM de d
WHERE regexp_replace(d.sri_CODIGO_CIIU, '[^0-9]', '', 'g')[1:2] = '84'
GROUP BY 1, 3 ORDER BY ciiu4, entidades DESC
""").df().to_string(index=False))

print('\n' + '='*120)
print('(D) LOS 627 DE "Otros": ¿la CIIU los podria clasificar?')
print('='*120)
print(con.execute("""
SELECT regexp_replace(d.sri_CODIGO_CIIU, '[^0-9]', '', 'g')[1:4] AS ciiu4,
       max(d.sri_ACTIVIDAD_ECONOMICA)[1:80] AS actividad,
       count(DISTINCT d.ruc) AS entidades, count(*) AS procesos,
       round(sum(pm.monto_final)/1e6, 1) AS millones
FROM pm JOIN de d ON d.ruc = pm.entidad_ruc
WHERE d.tipo_entidad = 'Otros'
GROUP BY 1 ORDER BY millones DESC LIMIT 20
""").df().to_string(index=False))
print(con.execute("""
SELECT count(DISTINCT CASE WHEN d.sri_CODIGO_CIIU IS NOT NULL THEN d.ruc END) AS otros_con_ciiu,
       count(DISTINCT d.ruc) AS otros_total
FROM de d WHERE d.tipo_entidad = 'Otros'
""").df().to_string(index=False))

print('\n' + '='*120)
print('(E) DONDE LA CIIU ME CONTRADICE: los desacuerdos de mayor monto')
print('='*120)
print(con.execute("""
WITH m AS (
  SELECT d.ruc, d.entidad_nombre_canonico AS entidad, d.tipo_entidad AS tipo_por_nombre,
         d.tipo_entidad_regla AS patron_que_uso,
         regexp_replace(d.sri_CODIGO_CIIU, '[^0-9]', '', 'g')[1:4] AS ciiu4,
         d.sri_ACTIVIDAD_ECONOMICA[1:70] AS actividad,
         count(*) AS procesos, sum(pm.monto_final) AS monto
  FROM pm JOIN de d ON d.ruc = pm.entidad_ruc
  WHERE d.sri_CODIGO_CIIU IS NOT NULL
  GROUP BY 1,2,3,4,5,6
)
SELECT entidad, ruc, tipo_por_nombre, patron_que_uso, ciiu4, actividad,
       procesos, round(monto/1e6, 1) AS millones
FROM m
WHERE (ciiu4 LIKE '86%' AND tipo_por_nombre <> 'Salud')
   OR (ciiu4 LIKE '85%' AND tipo_por_nombre NOT IN ('Educacion','Educacion superior'))
   OR (ciiu4 IN ('8422','8423') AND tipo_por_nombre <> 'Seguridad y defensa')
   OR (ciiu4 LIKE '64%' AND tipo_por_nombre <> 'Banca publica')
   OR (tipo_por_nombre = 'Salud' AND ciiu4 NOT LIKE '86%')
   OR (tipo_por_nombre = 'Seguridad y defensa' AND ciiu4 NOT IN ('8422','8423'))
ORDER BY monto DESC LIMIT 30
""").df().to_string(index=False))

print('\nEsta celda no escribio ningun parquet: solo mira. El mapeo CIIU -> tipo se escribe')
print('en la siguiente, con esta evidencia a la vista y no antes.')
print('\nPega aqui toda la salida.')

In [ ]:
# ===========================================================================
# CELDA 52 — la CIIU a 6 digitos como SEGUNDA agrupacion, no como reemplazo.
#            Mide el desacuerdo y reprueba el hallazgo de Salud con ella.
# ===========================================================================
import duckdb, pandas as pd, pathlib, json, datetime, hashlib

BASE = pathlib.Path(r'C:\SERCOP_FINAL'); PA = BASE/'plan_A'; DOC = PA/'documentacion'
con = duckdb.connect(); pd.set_option('display.width', 250)
pd.set_option('display.max_columns', 70); pd.set_option('display.max_colwidth', 60)
for v, f in [('pm','planA_procesos_modelo'), ('de','planA_dim_entidad'), ('ix','planA_indicadores')]:
    con.execute(f"CREATE OR REPLACE VIEW {v} AS SELECT * FROM "
                f"read_parquet('{(PA/(f+'.parquet')).as_posix()}')")

# ---- las tres correcciones que la CIIU destapo, al json de reglas --------
RJ = DOC/'reglas_tipologia.json'; R = json.loads(RJ.read_text(encoding='utf-8'))
R['overrides_por_ruc'].update({
 '1768007390001': ['Educacion superior',
   'Universidad de las Fuerzas Armadas ESPE - CIIU 853002 tercer nivel. Es una universidad, '
   'no una unidad militar: el patron FUERZAS ARMADAS la clasificaba mal.'],
 '1790932966001': ['Empresa publica',
   'FAME S.A. - CIIU 141001 fabricacion de prendas de vestir. Compra insumos textiles, '
   'no material militar.'],
 '1768192000001': ['Seguridad y defensa',
   'SNAI - CIIU 842303 administracion de prisiones y servicios correccionales.'],
})
R['version'] = '1.2'; R['fecha'] = datetime.date.today().isoformat()
R['nota_ciiu'] = ('La actividad economica del RUC se declara al inscribirse y no se mantiene: '
  'el Ministerio de Salud figura en 8411, el Consejo Nacional Electoral en 8423 tribunales, '
  'el Distrito Metropolitano de Guayaquil en 8422 defensa militar y 109 entidades compradoras '
  'en 9420 sindicatos. Por eso la CIIU se conserva como agrupacion alternativa para pruebas '
  'de robustez, no como clasificador funcional.')
RJ.write_text(json.dumps(R, ensure_ascii=False, indent=1), encoding='utf-8')
print(f'reglas_tipologia.json -> v{R["version"]}, {len(R["overrides_por_ruc"])} overrides verificados')

# ---- el grupo funcional CIIU, a partir del codigo de 6 digitos ----------
CIIU6 = "regexp_replace(d.sri_CODIGO_CIIU, '[^0-9]', '', 'g')"
GRUPO = f"""
CASE
 WHEN {CIIU6}[1:4] IN ('8411','8413')                              THEN 'Administracion publica general'
 WHEN {CIIU6}[1:6] = '841211'                                      THEN 'Administra programas de salud'
 WHEN {CIIU6}[1:6] IN ('841212','841217')                          THEN 'Administra programas de educacion'
 WHEN {CIIU6}[1:4] = '8412'                                        THEN 'Administra servicios generales'
 WHEN {CIIU6}[1:4] = '8421'                                        THEN 'Relaciones exteriores'
 WHEN {CIIU6}[1:4] = '8422'                                        THEN 'Defensa militar'
 WHEN {CIIU6}[1:6] = '842302'                                      THEN 'Bomberos y rescate'
 WHEN {CIIU6}[1:4] = '8423'                                        THEN 'Orden publico, policia y prisiones'
 WHEN {CIIU6}[1:4] = '8430'                                        THEN 'Seguridad social'
 WHEN {CIIU6}[1:2] = '85'                                          THEN 'Ensenanza'
 WHEN {CIIU6}[1:2] IN ('86','87','88')                             THEN 'Salud y asistencia social'
 WHEN {CIIU6}[1:2] IN ('35','36','37','38','39')                   THEN 'Electricidad, agua y residuos'
 WHEN {CIIU6}[1:2] IN ('64','65','66')                             THEN 'Financiero'
 WHEN {CIIU6}[1:2] IN ('41','42','43')                             THEN 'Construccion'
 WHEN {CIIU6}[1:2] IN ('49','50','51','52','53')                   THEN 'Transporte y almacenamiento'
 WHEN {CIIU6}[1:2] IN ('68','69','70','71','72','73','74')         THEN 'Servicios profesionales e inmobiliarios'
 WHEN {CIIU6}[1:2] IN ('90','91','93')                             THEN 'Cultura y deporte'
 WHEN {CIIU6}[1:2] = '94'                                          THEN 'Asociaciones y sindicatos'
 WHEN {CIIU6} IS NULL OR {CIIU6} = ''                              THEN 'sin CIIU'
 ELSE 'Otra actividad (CIIU ' || {CIIU6}[1:2] || ')'
END"""
GEN = ("('Administracion publica general','Administra servicios generales')")

print('\n' + '='*120)
print('(A) CUANTO DEL DINERO CAE EN UN CODIGO CIIU QUE NO DISTINGUE NADA')
print('='*120)
print(con.execute(f"""
SELECT {GRUPO} AS grupo_ciiu,
       CASE WHEN {GRUPO} IN {GEN} THEN 'generico' ELSE 'especifico' END AS utilidad,
       count(DISTINCT d.ruc) AS entidades, count(*) AS procesos,
       round(sum(pm.monto_final)/1e6, 1) AS millones,
       round(100.0*sum(pm.monto_final)/(SELECT sum(monto_final) FROM pm), 2) AS pct_monto,
       count(DISTINCT d.tipo_entidad) AS tipos_mios_distintos
FROM pm JOIN de d ON d.ruc = pm.entidad_ruc
GROUP BY 1, 2 ORDER BY millones DESC
""").df().to_string(index=False))

print('\n' + '='*120)
print('(B) ACUERDO ENTRE MI TIPOLOGIA Y LA CIIU, SOLO DONDE LA CIIU ES ESPECIFICA')
print('='*120)
print(con.execute(f"""
WITH e AS (
  SELECT d.tipo_entidad AS mi_tipo, {GRUPO} AS grupo_ciiu, pm.monto_final AS monto
  FROM pm JOIN de d ON d.ruc = pm.entidad_ruc
)
SELECT grupo_ciiu, mi_tipo, count(*) AS procesos, round(sum(monto)/1e6, 1) AS millones
FROM e WHERE grupo_ciiu NOT IN {GEN} AND grupo_ciiu <> 'sin CIIU'
GROUP BY 1, 2 HAVING sum(monto) > 20e6 ORDER BY grupo_ciiu, millones DESC
""").df().to_string(index=False))

print('\n' + '='*120)
print('(C) LAS 109 ENTIDADES QUE EL SRI REGISTRA COMO SINDICATOS: QUE SON EN REALIDAD')
print('='*120)
print(con.execute(f"""
SELECT d.entidad_nombre_canonico AS entidad, {CIIU6}[1:6] AS ciiu6,
       d.tipo_entidad AS mi_tipo, count(*) AS procesos,
       round(sum(pm.monto_final)/1e6, 2) AS millones
FROM pm JOIN de d ON d.ruc = pm.entidad_ruc
WHERE {CIIU6}[1:2] = '94'
GROUP BY 1,2,3 ORDER BY millones DESC LIMIT 20
""").df().to_string(index=False))

print('\n' + '='*120)
print('(D) LA PRUEBA QUE IMPORTA: el hallazgo de Salud, agrupando por CIIU')
print('='*120)
print(con.execute(f"""
SELECT {GRUPO} AS grupo_ciiu, count(DISTINCT d.ruc) AS entidades, count(*) AS procesos,
       round(sum(pm.monto_final)/1e6, 1) AS millones,
       round(median(pm.monto_final), 0) AS mediana_usd,
       round(100.0*avg(CASE WHEN ix.ind_oferente_unico THEN 1.0 ELSE 0.0 END), 2) AS pct_oferente_unico,
       round(100.0*avg(CASE WHEN ix.ind_fraccionamiento_marca THEN 1.0 ELSE 0.0 END), 2) AS pct_antecedente_30d
FROM pm JOIN de d ON d.ruc = pm.entidad_ruc LEFT JOIN ix ON ix.ocid = pm.ocid
GROUP BY 1 HAVING count(*) > 500 ORDER BY pct_antecedente_30d DESC
""").df().to_string(index=False))
print("""
Lo que decide: si los grupos de salud ('Salud y asistencia social' y 'Administra programas
de salud') encabezan esta tabla igual que encabezaban la de mi tipologia, el hallazgo esta
confirmado por una agrupacion que no construi yo. Si no, dependia de como agrupe.""")

# ---- la CIIU viaja en dim_entidad, junto a la tipologia -------------------
print('\n' + '='*120)
print('(E) ESCRITURA')
print('='*120)
nuevo = con.execute(f"""
SELECT d.* EXCLUDE (ciiu6, ciiu_grupo),
       {CIIU6}[1:6] AS ciiu6, {GRUPO} AS ciiu_grupo
FROM de d
""".replace('EXCLUDE (ciiu6, ciiu_grupo)', '') ).df()
nuevo = nuevo.loc[:, ~nuevo.columns.duplicated()]
ov = {k: v[0] for k, v in R['overrides_por_ruc'].items()}
antes = nuevo['tipo_entidad'].copy()
nuevo['tipo_entidad'] = [ov.get(r, t) for r, t in zip(nuevo['ruc'].astype(str), nuevo['tipo_entidad'])]
nuevo.loc[nuevo['ruc'].astype(str).isin(ov), 'tipo_entidad_origen'] = 'manual'
nuevo.loc[nuevo['ruc'].astype(str).isin(ov), 'tipo_entidad_regla']  = 'RUC verificado'
print(f'overrides aplicados: {int((antes != nuevo["tipo_entidad"]).sum())} entidades cambiaron de tipo')

con.register('dimout', nuevo)
con.execute(f"COPY dimout TO '{(PA/'planA_dim_entidad.parquet').as_posix()}' "
            f"(FORMAT PARQUET, COMPRESSION ZSTD)")
def sha16(p):
    h = hashlib.sha256()
    with open(p,'rb') as f:
        for b in iter(lambda: f.read(1<<20), b''): h.update(b)
    return h.hexdigest()[:16]
print(f'planA_dim_entidad.parquet | {nuevo.shape[0]:,} filas, {nuevo.shape[1]} columnas '
      f'| sha256_16 = {sha16(PA/"planA_dim_entidad.parquet")}')
print('\nPega aqui toda la salida.')

In [ ]:
# ===========================================================================
# CELDA 53 — ¿el antecedente a 30 dias en hospitales es fraccionamiento o es
#            abastecimiento recurrente de insumos?
# ===========================================================================
import duckdb, pandas as pd, pathlib, json

BASE = pathlib.Path(r'C:\SERCOP_FINAL'); PA = BASE/'plan_A'; DOC = PA/'documentacion'
con = duckdb.connect(); pd.set_option('display.width', 240)
pd.set_option('display.max_columns', 70); pd.set_option('display.max_colwidth', 58)
for v, f in [('pm','planA_procesos_modelo'), ('de','planA_dim_entidad'), ('ix','planA_indicadores')]:
    con.execute(f"CREATE OR REPLACE VIEW {v} AS SELECT * FROM "
                f"read_parquet('{(PA/(f+'.parquet')).as_posix()}')")

print('='*118)
print('(A) QUE MIDE EXACTAMENTE LA BANDERA, SEGUN LA ESPECIFICACION GUARDADA')
print('='*118)
esp = json.loads((DOC/'especificacion_indicadores.json').read_text(encoding='utf-8'))
def buscar(d, clave, ruta=''):
    if isinstance(d, dict):
        for k, v in d.items():
            if clave in str(k): print(f'  {ruta}{k}:\n    {json.dumps(v, ensure_ascii=False)[:700]}\n')
            else: buscar(v, clave, ruta + str(k) + ' > ')
    elif isinstance(d, list):
        for i, v in enumerate(d): buscar(v, clave, ruta)
for c in ['fraccionamiento_marca', 'fraccionamiento_monto_30d']:
    print(f'-- {c} --'); buscar(esp, c)

print('='*118)
print('(B) LOS ANTECEDENTES DE LOS PRESTADORES DE SALUD, POR SECCION CPC')
print('='*118)
print(con.execute("""
WITH b AS (
  SELECT pm.ocid, pm.cpc5, pm.cpc5[1:1] AS seccion, pm.monto_final, pm.proveedor_ruc,
         pm.entidad_ruc, ix.ind_fraccionamiento_marca AS marca,
         CASE WHEN d.ciiu_grupo = 'Salud y asistencia social' THEN 'prestador de salud'
              WHEN d.ciiu_grupo = 'Administra programas de salud' THEN 'administra salud'
              ELSE 'resto del sistema' END AS grupo
  FROM pm JOIN de d ON d.ruc = pm.entidad_ruc JOIN ix ON ix.ocid = pm.ocid
)
SELECT grupo, seccion, count(*) AS procesos,
       round(100.0*avg(CASE WHEN marca THEN 1.0 ELSE 0.0 END), 2) AS pct_antecedente,
       round(median(monto_final), 0) AS mediana_usd
FROM b WHERE grupo <> 'resto del sistema'
GROUP BY 1, 2 HAVING count(*) > 300 ORDER BY grupo, pct_antecedente DESC
""").df().to_string(index=False))

print('\n-- los 15 CPC5 que concentran los antecedentes en prestadores de salud --')
print(con.execute("""
SELECT pm.cpc5, count(*) AS procesos_con_antecedente,
       round(median(pm.monto_final), 0) AS mediana_usd,
       count(DISTINCT pm.proveedor_ruc) AS proveedores,
       count(DISTINCT pm.entidad_ruc) AS entidades
FROM pm JOIN de d ON d.ruc = pm.entidad_ruc JOIN ix ON ix.ocid = pm.ocid
WHERE d.ciiu_grupo = 'Salud y asistencia social' AND ix.ind_fraccionamiento_marca
GROUP BY 1 ORDER BY procesos_con_antecedente DESC LIMIT 15
""").df().to_string(index=False))

print('\n' + '='*118)
print('(C) LA PRUEBA QUE SEPARA ABASTECIMIENTO DE FRACCIONAMIENTO')
print('='*118)
print(con.execute("""
-- para cada proceso con antecedente, se busca el proceso anterior de la MISMA entidad
-- y el MISMO cpc5 dentro de los 30 dias previos, y se mira si fue al mismo proveedor
WITH b AS (
  SELECT pm.ocid, pm.entidad_ruc, pm.cpc5, pm.proveedor_ruc, pm.fecha_inicio_tender AS f,
         pm.monto_final,
         CASE WHEN d.ciiu_grupo = 'Salud y asistencia social' THEN 'prestador de salud'
              WHEN d.ciiu_grupo = 'Administra programas de salud' THEN 'administra salud'
              ELSE 'resto del sistema' END AS grupo
  FROM pm JOIN de d ON d.ruc = pm.entidad_ruc JOIN ix ON ix.ocid = pm.ocid
  WHERE ix.ind_fraccionamiento_marca
),
par AS (
  SELECT b.grupo, b.ocid, b.monto_final,
         max(CASE WHEN a.proveedor_ruc = b.proveedor_ruc THEN 1 ELSE 0 END) AS mismo_prov,
         count(a.ocid) AS n_antecedentes,
         sum(a.monto_final) AS monto_antecedentes
  FROM b LEFT JOIN pm a
    ON a.entidad_ruc = b.entidad_ruc AND a.cpc5 = b.cpc5
   AND a.fecha_inicio_tender < b.f
   AND a.fecha_inicio_tender >= b.f - INTERVAL 30 DAY
  GROUP BY 1, 2, 3
)
SELECT grupo, count(*) AS procesos_con_antecedente,
       round(100.0*avg(mismo_prov), 2) AS pct_mismo_proveedor,
       round(avg(n_antecedentes), 2) AS antecedentes_promedio,
       round(median(monto_final), 0) AS mediana_del_proceso,
       round(median(monto_final + coalesce(monto_antecedentes,0)), 0) AS mediana_suma_ventana
FROM par GROUP BY 1 ORDER BY procesos_con_antecedente DESC
""").df().to_string(index=False))
print("""
Como se lee: un porcentaje ALTO de mismo proveedor apunta a abastecimiento recurrente
(el hospital recompra el mismo insumo a su proveedor habitual). Un porcentaje BAJO, con
varios antecedentes y una suma de ventana que supera con holgura al proceso individual,
apunta a que una necesidad unica se dividio en varias adjudicaciones. Los dos patrones
coexisten; lo que interesa es cual domina en salud frente al resto del sistema.""")
print('\nPega aqui toda la salida.')


In [ ]:
# ===========================================================================
# CELDA 53b — reproducir la bandera, ubicar la columna CPC, y separar
#             abastecimiento recurrente de posible fraccionamiento
# ===========================================================================
import duckdb, pandas as pd, pathlib

BASE = pathlib.Path(r'C:\SERCOP_FINAL'); PA = BASE/'plan_A'
con = duckdb.connect(); pd.set_option('display.width', 240)
pd.set_option('display.max_columns', 70); pd.set_option('display.max_colwidth', 58)
for v, f in [('pm','planA_procesos_modelo'), ('de','planA_dim_entidad'), ('ix','planA_indicadores')]:
    con.execute(f"CREATE OR REPLACE VIEW {v} AS SELECT * FROM "
                f"read_parquet('{(PA/(f+'.parquet')).as_posix()}')")

cols_pm = list(con.execute("DESCRIBE SELECT * FROM pm").df()['column_name'])
cand = [c for c in cols_pm if 'cpc' in c.lower() or 'clasif' in c.lower()]
print('columnas candidatas para el CPC en procesos_modelo:', cand)
C = ('cpc5' if 'cpc5' in cand else
     next((c for c in cand if 'cpc5' in c.lower()), None) or
     next((c for c in cand if 'cpc'  in c.lower()), None))
print(f'se usa: {C}\n')
print(con.execute(f'SELECT "{C}" AS cpc, count(*) AS n FROM pm GROUP BY 1 '
                  f'ORDER BY n DESC LIMIT 5').df().to_string(index=False))

print('\n' + '='*118)
print('(A) REPRODUCCION DE LA BANDERA: ¿es exactamente monto_30d > 0?')
print('='*118)
print(con.execute("""
SELECT count(*) AS filas,
       sum(CASE WHEN ind_fraccionamiento_marca = (coalesce(ind_fraccionamiento_monto_30d,0) > 0)
                THEN 1 ELSE 0 END) AS coinciden,
       sum(CASE WHEN ind_fraccionamiento_marca AND coalesce(ind_fraccionamiento_monto_30d,0) <= 0
                THEN 1 ELSE 0 END) AS marca_sin_monto,
       sum(CASE WHEN NOT ind_fraccionamiento_marca AND coalesce(ind_fraccionamiento_monto_30d,0) > 0
                THEN 1 ELSE 0 END) AS monto_sin_marca
FROM ix
""").df().to_string(index=False))
print('coinciden tiene que igualar a filas. Si no, la bandera no es monto_30d > 0 y su')
print('definicion real hay que averiguarla antes de usarla en cualquier conclusion.')

print('\n' + '='*118)
print('(B) LOS ANTECEDENTES DE LOS PRESTADORES DE SALUD, POR SECCION CPC')
print('='*118)
print(con.execute(f"""
WITH b AS (
  SELECT pm."{C}" AS cpc, substr(pm."{C}", 1, 1) AS seccion, pm.monto_final,
         ix.ind_fraccionamiento_marca AS marca,
         CASE WHEN d.ciiu_grupo = 'Salud y asistencia social'    THEN '1 prestador de salud'
              WHEN d.ciiu_grupo = 'Administra programas de salud' THEN '2 administra salud'
              ELSE '3 resto del sistema' END AS grupo
  FROM pm JOIN de d ON d.ruc = pm.entidad_ruc JOIN ix ON ix.ocid = pm.ocid
)
SELECT grupo, seccion, count(*) AS procesos,
       round(100.0*avg(CASE WHEN marca THEN 1.0 ELSE 0.0 END), 2) AS pct_recompra_30d,
       round(median(monto_final), 0) AS mediana_usd
FROM b GROUP BY 1, 2 HAVING count(*) > 500 ORDER BY grupo, pct_recompra_30d DESC
""").df().to_string(index=False))

print('\n-- los 15 CPC5 que concentran las recompras en prestadores de salud --')
print(con.execute(f"""
SELECT pm."{C}" AS cpc5, count(*) AS procesos_con_recompra,
       round(median(pm.monto_final), 0) AS mediana_usd,
       count(DISTINCT pm.proveedor_ruc) AS proveedores, count(DISTINCT pm.entidad_ruc) AS entidades
FROM pm JOIN de d ON d.ruc = pm.entidad_ruc JOIN ix ON ix.ocid = pm.ocid
WHERE d.ciiu_grupo = 'Salud y asistencia social' AND ix.ind_fraccionamiento_marca
GROUP BY 1 ORDER BY procesos_con_recompra DESC LIMIT 15
""").df().to_string(index=False))

print('\n' + '='*118)
print('(C) CADENCIA: la prueba que si separa abastecimiento de fraccionamiento')
print('='*118)
print(con.execute(f"""
-- para cada par entidad-proveedor-cpc5 y anio, cuantas compras hubo en el anio.
-- un par con 15 compras al anio es un proveedor habitual de insumos; un par con
-- exactamente 2 o 3 compras muy juntas y nada mas en el anio tiene la forma del
-- fraccionamiento: una necesidad unica partida en pedazos.
WITH p AS (
  SELECT pm.entidad_ruc, pm.proveedor_ruc, pm."{C}" AS cpc, pm.anio_proceso AS anio,
         count(*) AS compras_en_el_anio
  FROM pm GROUP BY 1,2,3,4
),
j AS (
  SELECT ix.ind_fraccionamiento_marca AS marca, p.compras_en_el_anio AS n,
         CASE WHEN d.ciiu_grupo = 'Salud y asistencia social'    THEN '1 prestador de salud'
              WHEN d.ciiu_grupo = 'Administra programas de salud' THEN '2 administra salud'
              ELSE '3 resto del sistema' END AS grupo
  FROM pm JOIN ix ON ix.ocid = pm.ocid JOIN de d ON d.ruc = pm.entidad_ruc
  JOIN p ON p.entidad_ruc = pm.entidad_ruc AND p.proveedor_ruc = pm.proveedor_ruc
        AND p.cpc = pm."{C}" AND p.anio = pm.anio_proceso
)
SELECT grupo,
       CASE WHEN n = 1 THEN 'a) 1 compra en el anio'
            WHEN n <= 3 THEN 'b) 2 a 3 compras'
            WHEN n <= 6 THEN 'c) 4 a 6 compras'
            WHEN n <= 12 THEN 'd) 7 a 12 compras'
            ELSE 'e) mas de 12 compras' END AS cadencia_del_par,
       count(*) AS procesos,
       round(100.0*count(*)/sum(count(*)) OVER (PARTITION BY grupo), 2) AS pct_del_grupo,
       round(100.0*avg(CASE WHEN marca THEN 1.0 ELSE 0.0 END), 2) AS pct_recompra_30d
FROM j GROUP BY 1, 2 ORDER BY grupo, cadencia_del_par
""").df().to_string(index=False))
print("""
Como se lee: si las recompras de los prestadores de salud se concentran en pares con
muchas compras al anio, es abastecimiento habitual y hay que decirlo asi. Si se concentran
en pares con 2 o 3 compras y nada mas, esa es la forma del fraccionamiento y ahi si hay
senal. La banda 'b) 2 a 3 compras' es la que interesa comparar entre grupos.""")
print('\nPega aqui toda la salida.')

In [ ]:
# ===========================================================================
# CELDA 54 — descomposicion del efecto composicion, y el indicador de rafaga
#            que si tiene la forma del fraccionamiento
# ===========================================================================
import duckdb, pandas as pd, numpy as np, pathlib, json, datetime

BASE = pathlib.Path(r'C:\SERCOP_FINAL'); PA = BASE/'plan_A'; DOC = PA/'documentacion'
con = duckdb.connect(); pd.set_option('display.width', 240); pd.set_option('display.max_columns', 70)
for v, f in [('pm','planA_procesos_modelo'), ('de','planA_dim_entidad'), ('ix','planA_indicadores')]:
    con.execute(f"CREATE OR REPLACE VIEW {v} AS SELECT * FROM "
                f"read_parquet('{(PA/(f+'.parquet')).as_posix()}')")
C = 'cpc5_principal'
GRUPO = """CASE WHEN d.ciiu_grupo = 'Salud y asistencia social'     THEN 'prestador de salud'
                WHEN d.ciiu_grupo = 'Administra programas de salud' THEN 'administra salud'
                ELSE 'resto del sistema' END"""

print('='*118)
print('(A) DESCOMPOSICION: cuanto de la brecha es composicion y cuanto es efecto propio')
print('='*118)
t = con.execute(f"""
WITH p AS (SELECT entidad_ruc, proveedor_ruc, "{C}" AS cpc, anio_proceso AS anio,
                  count(*) AS n FROM pm GROUP BY 1,2,3,4),
j AS (SELECT {GRUPO} AS grupo, ix.ind_fraccionamiento_marca AS marca,
             CASE WHEN p.n = 1 THEN 1 WHEN p.n <= 3 THEN 2 WHEN p.n <= 6 THEN 3
                  WHEN p.n <= 12 THEN 4 ELSE 5 END AS banda
      FROM pm JOIN ix ON ix.ocid = pm.ocid JOIN de d ON d.ruc = pm.entidad_ruc
      JOIN p ON p.entidad_ruc = pm.entidad_ruc AND p.proveedor_ruc = pm.proveedor_ruc
            AND p.cpc = pm."{C}" AND p.anio = pm.anio_proceso)
SELECT grupo, banda, count(*) AS n,
       avg(CASE WHEN marca THEN 1.0 ELSE 0.0 END) AS u
FROM j GROUP BY 1,2
""").df()
piv = t.pivot(index='banda', columns='grupo', values=['n','u']).fillna(0)
for g in ['prestador de salud','administra salud']:
    s_g = piv[('n', g)] / piv[('n', g)].sum()
    s_r = piv[('n','resto del sistema')] / piv[('n','resto del sistema')].sum()
    u_g, u_r = piv[('u', g)], piv[('u','resto del sistema')]
    obs_g, obs_r = (s_g*u_g).sum(), (s_r*u_r).sum()
    contraf = (s_r*u_g).sum()                     # su tasa, la cadencia del resto
    print(f'\n{g}')
    print(f'  observado                                  {100*obs_g:6.2f}%')
    print(f'  resto del sistema                          {100*obs_r:6.2f}%')
    print(f'  brecha                                     {100*(obs_g-obs_r):6.2f} puntos')
    print(f'  contrafactual con la cadencia del resto    {100*contraf:6.2f}%')
    print(f'  -> composicion (cadencia de compra)        {100*(obs_g-contraf):6.2f} puntos '
          f'({100*(obs_g-contraf)/(obs_g-obs_r):5.1f}% de la brecha)')
    print(f'  -> efecto propio del grupo                 {100*(contraf-obs_r):6.2f} puntos '
          f'({100*(contraf-obs_r)/(obs_g-obs_r):5.1f}% de la brecha)')

print('\n' + '='*118)
print('(B) EL INDICADOR DE RAFAGA: compras concentradas en pocos dias y nada mas')
print('='*118)
raf = con.execute(f"""
WITH p AS (
  SELECT entidad_ruc, proveedor_ruc, "{C}" AS cpc, anio_proceso AS anio, count(*) AS n,
         min(fecha_inicio_tender) AS primera, max(fecha_inicio_tender) AS ultima,
         date_diff('day', min(fecha_inicio_tender), max(fecha_inicio_tender)) AS span_dias,
         sum(monto_final) AS monto_del_par
  FROM pm GROUP BY 1,2,3,4
)
SELECT {GRUPO} AS grupo, p.n AS compras_del_par,
       count(*) AS procesos,
       round(100.0*avg(CASE WHEN p.span_dias <= 30 THEN 1.0 ELSE 0.0 END), 2) AS pct_rafaga_30d,
       round(100.0*avg(CASE WHEN p.span_dias > 180 THEN 1.0 ELSE 0.0 END), 2) AS pct_repartido_anio,
       round(median(p.span_dias), 0) AS span_mediano_dias,
       round(median(p.monto_del_par), 0) AS monto_mediano_del_par
FROM pm JOIN de d ON d.ruc = pm.entidad_ruc
JOIN p ON p.entidad_ruc = pm.entidad_ruc AND p.proveedor_ruc = pm.proveedor_ruc
      AND p.cpc = pm."{C}" AND p.anio = pm.anio_proceso
WHERE p.n BETWEEN 2 AND 6
GROUP BY 1, 2 ORDER BY compras_del_par, grupo
""").df()
print(raf.to_string(index=False))
print("""
Aqui la comparacion es legitima porque esta condicionada: se compara el patron temporal
de pares con el MISMO numero de compras. Si entre los pares de 2 compras los prestadores
de salud concentran mas que el resto dentro de 30 dias, esa si es senal; si concentran
igual o menos, la recompra en salud es abastecimiento repartido en el anio.""")

print('\n-- y lo mismo por familia de contratacion, que es donde opera el umbral legal --')
print(con.execute(f"""
WITH p AS (
  SELECT entidad_ruc, proveedor_ruc, "{C}" AS cpc, anio_proceso AS anio, count(*) AS n,
         date_diff('day', min(fecha_inicio_tender), max(fecha_inicio_tender)) AS span_dias
  FROM pm GROUP BY 1,2,3,4
)
SELECT pm.familia, count(*) AS procesos_en_pares_de_2_a_3,
       round(100.0*avg(CASE WHEN p.span_dias <= 30 THEN 1.0 ELSE 0.0 END), 2) AS pct_rafaga_30d,
       round(median(pm.monto_final), 0) AS mediana_usd
FROM pm JOIN p ON p.entidad_ruc = pm.entidad_ruc AND p.proveedor_ruc = pm.proveedor_ruc
             AND p.cpc = pm."{C}" AND p.anio = pm.anio_proceso
WHERE p.n BETWEEN 2 AND 3
GROUP BY 1 HAVING count(*) > 500 ORDER BY pct_rafaga_30d DESC
""").df().to_string(index=False))

# ---- cerrar el hueco de la especificacion -------------------------------
EJ = DOC/'especificacion_indicadores.json'
esp = json.loads(EJ.read_text(encoding='utf-8'))
esp.setdefault('definiciones_verificadas', {})['ind_fraccionamiento_marca'] = (
 "bandera binaria, verdadera cuando ind_fraccionamiento_monto_30d > 0; es decir, cuando el "
 "mismo par entidad-proveedor compro el mismo CPC5 en los 30 dias anteriores sin contar el "
 "dia del proceso. Reproducida al 100% sobre las 446.677 filas, sin discrepancias en ninguno "
 "de los dos sentidos. ADVERTENCIA: mide recurrencia de recompra al proveedor habitual, no "
 "fraccionamiento. Su tasa esta determinada por la frecuencia de compra del par: condicionada "
 "a esa frecuencia, la tasa es igual en todos los tipos de entidad, y la brecha aparente de "
 "los prestadores de salud (9,44% frente a 3,34%) es composicion en un 99%."
)
esp.setdefault('renombres_pendientes', {}).update({
 'ind_fraccionamiento_marca': 'ind_recompra_30d',
 'ind_fraccionamiento_monto_30d': 'ind_monto_recompra_30d'})
esp['fecha_actualizacion'] = datetime.date.today().isoformat()
EJ.write_text(json.dumps(esp, ensure_ascii=False, indent=1), encoding='utf-8')
print(f'\nespecificacion_indicadores.json actualizado: definicion de la bandera registrada')
print('y renombres anotados como pendientes (no aplicados todavia).')
print('\nPega aqui toda la salida.')

In [ ]:
# ===========================================================================
# CELDA 54b — rafagas por familia, techo observado de cada familia, y el test
#             del cruce: compras bajo el techo cuya suma lo supera
# ===========================================================================
import duckdb, pandas as pd, pathlib
BASE = pathlib.Path(r'C:\SERCOP_FINAL'); PA = BASE/'plan_A'
con = duckdb.connect(); pd.set_option('display.width', 240); pd.set_option('display.max_columns', 70)
for v, f in [('pm','planA_procesos_modelo'), ('de','planA_dim_entidad'), ('ix','planA_indicadores')]:
    con.execute(f"CREATE OR REPLACE VIEW {v} AS SELECT * FROM "
                f"read_parquet('{(PA/(f+'.parquet')).as_posix()}')")
C = 'cpc5_principal'
print('familia esta en indicadores:',
      'familia' in list(con.execute("DESCRIBE SELECT * FROM ix").df()['column_name']))

# la base de trabajo: cada proceso con su par, su familia y su grupo CIIU
con.execute(f"""
CREATE OR REPLACE VIEW base AS
WITH p AS (
  SELECT entidad_ruc, proveedor_ruc, "{C}" AS cpc, anio_proceso AS anio, count(*) AS n,
         date_diff('day', min(fecha_inicio_tender), max(fecha_inicio_tender)) AS span,
         sum(monto_final) AS monto_par, max(monto_final) AS monto_max_par
  FROM pm GROUP BY 1,2,3,4
)
SELECT pm.ocid, pm.entidad_ruc, pm.proveedor_ruc, pm."{C}" AS cpc, pm.anio_proceso AS anio,
       pm.monto_final, ix.familia,
       CASE WHEN d.ciiu_grupo = 'Salud y asistencia social'     THEN 'prestador de salud'
            WHEN d.ciiu_grupo = 'Administra programas de salud' THEN 'administra salud'
            ELSE 'resto del sistema' END AS grupo,
       p.n, p.span, p.monto_par, p.monto_max_par,
       (p.n >= 2 AND p.span <= 30) AS rafaga
FROM pm JOIN ix ON ix.ocid = pm.ocid JOIN de d ON d.ruc = pm.entidad_ruc
JOIN p ON p.entidad_ruc = pm.entidad_ruc AND p.proveedor_ruc = pm.proveedor_ruc
      AND p.cpc = pm."{C}" AND p.anio = pm.anio_proceso
""")

print('\n' + '='*118)
print('(A) RAFAGAS POR FAMILIA, CONDICIONADO AL NUMERO DE COMPRAS DEL PAR')
print('='*118)
print(con.execute("""
SELECT familia, n AS compras_del_par, count(*) AS procesos,
       round(100.0*avg(CASE WHEN rafaga THEN 1.0 ELSE 0.0 END), 2) AS pct_rafaga_30d,
       round(median(monto_final), 0) AS mediana_proceso,
       round(median(monto_par), 0) AS mediana_suma_del_par
FROM base WHERE n IN (2, 3) GROUP BY 1, 2
HAVING count(*) > 300 ORDER BY compras_del_par, pct_rafaga_30d DESC
""").df().to_string(index=False))

print('\n' + '='*118)
print('(B) EL TECHO OBSERVADO DE CADA FAMILIA, POR ANIO')
print('='*118)
print(con.execute("""
SELECT familia, anio, count(*) AS procesos,
       round(quantile_cont(monto_final, 0.99), 0) AS p99,
       round(max(monto_final), 0) AS techo_observado
FROM base WHERE anio BETWEEN 2018 AND 2024 AND familia IN
     ('Infima cuantia','Menor cuantia','Contratacion directa','Cotizacion','Licitacion')
GROUP BY 1, 2 ORDER BY familia, anio
""").df().to_string(index=False))
print("""Si el techo observado de una familia es practicamente el mismo todos los anios,
ese numero es el umbral legal operando y sirve como referencia sin tener la norma.""")

print('\n' + '='*118)
print('(C) EL TEST DEL CRUCE: cada compra por debajo del techo, la suma por encima')
print('='*118)
print(con.execute("""
WITH techo AS (
  SELECT familia, anio, max(monto_final) AS t FROM base GROUP BY 1, 2
),
x AS (
  SELECT b.*, techo.t,
         (b.monto_max_par <= techo.t AND b.monto_par > techo.t) AS cruza
  FROM base b JOIN techo ON techo.familia = b.familia AND techo.anio = b.anio
  WHERE b.n BETWEEN 2 AND 4
)
SELECT familia, grupo, count(*) AS procesos,
       round(100.0*avg(CASE WHEN rafaga THEN 1.0 ELSE 0.0 END), 2) AS pct_rafaga,
       round(100.0*avg(CASE WHEN cruza THEN 1.0 ELSE 0.0 END), 2) AS pct_cruza_el_techo,
       round(100.0*avg(CASE WHEN rafaga AND cruza THEN 1.0 ELSE 0.0 END), 2) AS pct_rafaga_y_cruza
FROM x GROUP BY 1, 2 HAVING count(*) > 500
ORDER BY pct_rafaga_y_cruza DESC LIMIT 25
""").df().to_string(index=False))
print("""
pct_rafaga_y_cruza es el indicador con la forma correcta del fraccionamiento: varias
compras del mismo producto al mismo proveedor, concentradas en menos de 30 dias, cada una
por debajo del techo de su familia y la suma por encima. Si ese numero es parecido en todas
las familias no significa nada; si se concentra en las familias de umbral bajo, el mecanismo
que postulamos existe y el indicador lo capta.""")
print('\nPega aqui toda la salida.')

In [ ]:
# ===========================================================================
# CELDA 55 — el techo como diagnostico, el test del cruce bien estimado, y
#            casos concretos para verificar en el portal
# ===========================================================================
import duckdb, pandas as pd, pathlib
BASE = pathlib.Path(r'C:\SERCOP_FINAL'); PA = BASE/'plan_A'
con = duckdb.connect(); pd.set_option('display.width', 250); pd.set_option('display.max_columns', 70)
for v, f in [('pm','planA_procesos_modelo'), ('de','planA_dim_entidad'),
             ('ix','planA_indicadores'), ('un','planA_universo')]:
    con.execute(f"CREATE OR REPLACE VIEW {v} AS SELECT * FROM "
                f"read_parquet('{(PA/(f+'.parquet')).as_posix()}')")
C = 'cpc5_principal'

print('='*118)
print('(A) ¿EXISTE LA INFIMA CUANTIA EN LA BASE?')
print('='*118)
print('-- familias en los 446.677 procesos del modelo --')
print(con.execute("SELECT familia, count(*) AS procesos FROM ix GROUP BY 1 "
                  "ORDER BY procesos DESC").df().to_string(index=False))
print('\n-- y en el universo completo de 2,78 millones, buscando el texto --')
cols_un = list(con.execute("DESCRIBE SELECT * FROM un").df()['column_name'])
cf = next((c for c in cols_un if 'familia' in c.lower()), None) or \
     next((c for c in cols_un if 'procedimiento' in c.lower() or 'metodo' in c.lower()), None)
print(f'columna usada en universo: {cf}')
print(con.execute(f'SELECT "{cf}" AS tipo, count(*) AS n FROM un GROUP BY 1 '
                  f'ORDER BY n DESC LIMIT 25').df().to_string(index=False))

print('\n' + '='*118)
print('(B) DIAGNOSTICO DE TECHO: el cociente max/p99 por familia')
print('='*118)
print(con.execute("""
WITH a AS (
  SELECT ix.familia, pm.anio_proceso AS anio, count(*) AS procesos,
         quantile_cont(pm.monto_final, 0.99) AS p99, max(pm.monto_final) AS mx
  FROM pm JOIN ix ON ix.ocid = pm.ocid
  WHERE pm.anio_proceso BETWEEN 2016 AND 2024 GROUP BY 1, 2
)
SELECT familia, count(*) AS anios, round(avg(mx/p99), 2) AS cociente_medio,
       round(min(mx), 0) AS techo_min, round(max(mx), 0) AS techo_max,
       round(100.0*(max(mx)-min(mx))/avg(mx), 1) AS pct_variacion_del_techo,
       CASE WHEN avg(mx/p99) < 1.5 THEN 'TECHO DURO: hay umbral legal'
            WHEN avg(mx/p99) < 5   THEN 'dudoso'
            ELSE 'sin techo' END AS veredicto
FROM a GROUP BY 1 ORDER BY cociente_medio
""").df().to_string(index=False))

print('\n' + '='*118)
print('(C) EL TEST DEL CRUCE, SOLO DONDE EL TECHO ES REAL, SIN FILTRO DE TAMANO')
print('='*118)
con.execute(f"""
CREATE OR REPLACE VIEW cruce AS
WITH p AS (
  SELECT entidad_ruc, proveedor_ruc, "{C}" AS cpc, anio_proceso AS anio, count(*) AS n,
         date_diff('day', min(fecha_inicio_tender), max(fecha_inicio_tender)) AS span,
         sum(monto_final) AS monto_par, max(monto_final) AS monto_max_par
  FROM pm GROUP BY 1,2,3,4
),
techo AS (   -- el techo de la familia-anio, que ya validamos como umbral real
  SELECT ix.familia, pm.anio_proceso AS anio, max(pm.monto_final) AS t
  FROM pm JOIN ix ON ix.ocid = pm.ocid GROUP BY 1, 2
)
SELECT pm.ocid, pm.entidad_ruc, pm.proveedor_ruc, pm."{C}" AS cpc, pm.anio_proceso AS anio,
       pm.monto_final, pm.entidad_nombre, pm.proveedor_nombre, ix.familia, techo.t AS techo,
       p.n, p.span, p.monto_par,
       CASE WHEN d.ciiu_grupo = 'Salud y asistencia social'     THEN 'prestador de salud'
            WHEN d.ciiu_grupo = 'Administra programas de salud' THEN 'administra salud'
            ELSE 'resto del sistema' END AS grupo,
       (p.n >= 2 AND p.span <= 30) AS rafaga,
       (p.monto_max_par <= techo.t AND p.monto_par > techo.t) AS cruza
FROM pm JOIN ix ON ix.ocid = pm.ocid JOIN de d ON d.ruc = pm.entidad_ruc
JOIN p ON p.entidad_ruc = pm.entidad_ruc AND p.proveedor_ruc = pm.proveedor_ruc
      AND p.cpc = pm."{C}" AND p.anio = pm.anio_proceso
JOIN techo ON techo.familia = ix.familia AND techo.anio = pm.anio_proceso
WHERE ix.familia IN ('Menor cuantia', 'Cotizacion') AND p.n BETWEEN 2 AND 4
""")
print(con.execute("""
SELECT familia, grupo, count(*) AS procesos,
       sum(CASE WHEN rafaga AND cruza THEN 1 ELSE 0 END) AS casos,
       round(100.0*avg(CASE WHEN rafaga AND cruza THEN 1.0 ELSE 0.0 END), 2) AS pct,
       round(median(monto_final), 0) AS mediana_proceso,
       round(median(monto_par), 0) AS mediana_suma
FROM cruce GROUP BY 1, 2 ORDER BY familia, pct DESC
""").df().to_string(index=False))

print('\n-- por anio, para ver si el patron es estable o concentrado en alguna epoca --')
print(con.execute("""
SELECT anio, count(*) AS procesos, sum(CASE WHEN rafaga AND cruza THEN 1 ELSE 0 END) AS casos,
       round(100.0*avg(CASE WHEN rafaga AND cruza THEN 1.0 ELSE 0.0 END), 2) AS pct
FROM cruce GROUP BY 1 ORDER BY anio
""").df().to_string(index=False))

print('\n' + '='*118)
print('(D) DOCE CASOS CONCRETOS PARA QUE LOS REVISES EN EL PORTAL')
print('='*118)
print(con.execute("""
SELECT ocid, familia, anio, entidad_nombre[1:40] AS entidad,
       proveedor_nombre[1:34] AS proveedor, cpc, n AS compras, span AS dias,
       round(monto_final, 0) AS este_proceso, round(monto_par, 0) AS suma_del_par,
       round(techo, 0) AS techo_de_la_familia
FROM cruce WHERE rafaga AND cruza ORDER BY monto_par DESC LIMIT 12
""").df().to_string(index=False))
print("""
Lo que habria que confirmar en el portal, en dos o tres de estos: que son objetos de
contratacion realmente equivalentes y no cosas distintas que comparten codigo CPC. Si son
equivalentes, el caso es solido; si el CPC agrupa productos diferentes, el indicador tiene
un falso positivo estructural y hay que decirlo.""")
print('\nPega aqui toda la salida.')

In [ ]:
# ===========================================================================
# CELDA 55b — misma medicion, materializando cada paso y con cronometro.
#             Reemplaza el bloque (C) y (D) de la celda 55.
# ===========================================================================
import time, duckdb, pandas as pd, pathlib
BASE = pathlib.Path(r'C:\SERCOP_FINAL'); PA = BASE/'plan_A'
con = duckdb.connect()           # conexion limpia: la anterior quedo con vistas anidadas
con.execute("PRAGMA memory_limit='4GB'")
con.execute("PRAGMA threads=4")
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 70)
for v, f in [('pm','planA_procesos_modelo'), ('de','planA_dim_entidad'), ('ix','planA_indicadores')]:
    con.execute(f"CREATE OR REPLACE VIEW {v} AS SELECT * FROM "
                f"read_parquet('{(PA/(f+'.parquet')).as_posix()}')")
C = 'cpc5_principal'

t0 = time.time()
def tic(m): print(f'  [{time.time()-t0:6.1f}s] {m}', flush=True)

# 1) una sola lectura de los tres archivos, quedandose con 11 columnas
con.execute(f"""
CREATE OR REPLACE TABLE s AS
SELECT pm.ocid, pm.entidad_ruc, pm.proveedor_ruc, pm."{C}" AS cpc,
       pm.anio_proceso AS anio, pm.monto_final, pm.fecha_inicio_tender AS f,
       pm.entidad_nombre, pm.proveedor_nombre, ix.familia,
       CASE WHEN d.ciiu_grupo = 'Salud y asistencia social'     THEN 'prestador de salud'
            WHEN d.ciiu_grupo = 'Administra programas de salud' THEN 'administra salud'
            ELSE 'resto del sistema' END AS grupo
FROM pm JOIN ix ON ix.ocid = pm.ocid JOIN de d ON d.ruc = pm.entidad_ruc
""")
n_s, n_sin_cpc = con.execute(
  "SELECT count(*), sum(CASE WHEN cpc IS NULL THEN 1 ELSE 0 END) FROM s").fetchone()
tic(f'tabla de trabajo: {n_s:,} filas, 11 columnas  (sin CPC: {n_sin_cpc:,})')

# 2) los pares entidad-proveedor-cpc-anio, excluyendo CPC nulo de forma explicita
con.execute("""
CREATE OR REPLACE TABLE pares AS
SELECT entidad_ruc, proveedor_ruc, cpc, anio, count(*) AS n,
       date_diff('day', min(f), max(f)) AS span,
       sum(monto_final) AS monto_par, max(monto_final) AS monto_max_par
FROM s WHERE cpc IS NOT NULL AND f IS NOT NULL
GROUP BY 1, 2, 3, 4
""")
tic(f"pares: {con.execute('SELECT count(*) FROM pares').fetchone()[0]:,}")

# 3) el techo de cada familia-anio
con.execute("CREATE OR REPLACE TABLE techo AS "
            "SELECT familia, anio, max(monto_final) AS t FROM s GROUP BY 1, 2")
tic('techos')

# 4) la tabla final, materializada una sola vez
con.execute("""
CREATE OR REPLACE TABLE cruce AS
SELECT s.ocid, s.familia, s.anio, s.grupo, s.entidad_nombre, s.proveedor_nombre,
       s.cpc, s.monto_final, p.n, p.span, p.monto_par, t.t AS techo,
       (p.n >= 2 AND p.span <= 30)                           AS rafaga,
       (p.monto_max_par <= t.t AND p.monto_par > t.t)         AS cruza
FROM s
JOIN pares p ON p.entidad_ruc = s.entidad_ruc AND p.proveedor_ruc = s.proveedor_ruc
            AND p.cpc = s.cpc AND p.anio = s.anio
JOIN techo t ON t.familia = s.familia AND t.anio = s.anio
WHERE s.familia IN ('Menor cuantia', 'Cotizacion') AND p.n BETWEEN 2 AND 4
""")
tic(f"cruce: {con.execute('SELECT count(*) FROM cruce').fetchone()[0]:,} filas")

print('\n' + '='*118)
print('(C) EL TEST DEL CRUCE, SOLO DONDE EL TECHO ES REAL, SIN FILTRO DE TAMANO')
print('='*118)
print(con.execute("""
SELECT familia, grupo, count(*) AS procesos,
       sum(CASE WHEN rafaga AND cruza THEN 1 ELSE 0 END) AS casos,
       round(100.0*avg(CASE WHEN rafaga AND cruza THEN 1.0 ELSE 0.0 END), 2) AS pct,
       round(median(monto_final), 0) AS mediana_proceso,
       round(median(monto_par), 0) AS mediana_suma
FROM cruce GROUP BY 1, 2 ORDER BY familia, pct DESC
""").df().to_string(index=False))

print('\n-- por anio --')
print(con.execute("""
SELECT anio, count(*) AS procesos, sum(CASE WHEN rafaga AND cruza THEN 1 ELSE 0 END) AS casos,
       round(100.0*avg(CASE WHEN rafaga AND cruza THEN 1.0 ELSE 0.0 END), 2) AS pct
FROM cruce GROUP BY 1 ORDER BY anio
""").df().to_string(index=False))

print('\n' + '='*118)
print('(D) DOCE CASOS CONCRETOS PARA REVISAR EN EL PORTAL')
print('='*118)
print(con.execute("""
SELECT ocid, familia, anio, entidad_nombre[1:38] AS entidad,
       proveedor_nombre[1:32] AS proveedor, cpc, n AS compras, span AS dias,
       round(monto_final, 0) AS este_proceso, round(monto_par, 0) AS suma_del_par,
       round(techo, 0) AS techo_familia
FROM cruce WHERE rafaga AND cruza ORDER BY monto_par DESC LIMIT 12
""").df().to_string(index=False))
tic('listo')
print('\nPega aqui toda la salida.')

In [ ]:
# ===========================================================================
# CELDA 56 — el indicador sobre el PLIEGO, y cuantos casos son lotes
#            declarados en el propio titulo del proceso
# ===========================================================================
import time, duckdb, pandas as pd, pathlib
BASE = pathlib.Path(r'C:\SERCOP_FINAL'); PA = BASE/'plan_A'
con = duckdb.connect(); con.execute("PRAGMA memory_limit='4GB'"); con.execute("PRAGMA threads=4")
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 70)
pd.set_option('display.max_colwidth', 70)
for v, f in [('pm','planA_procesos_modelo'), ('de','planA_dim_entidad'), ('ix','planA_indicadores')]:
    con.execute(f"CREATE OR REPLACE VIEW {v} AS SELECT * FROM "
                f"read_parquet('{(PA/(f+'.parquet')).as_posix()}')")
cols = list(con.execute("DESCRIBE SELECT * FROM pm").df()['column_name'])
T = next((c for c in cols if c in ('titulo','title','objeto','objeto_contratacion')), None) or \
    next((c for c in cols if 'titul' in c.lower() or 'objeto' in c.lower()
                          or 'descrip' in c.lower()), None)
MT = next((c for c in cols if 'monto_tender' in c), None)
print(f'columna de titulo: {T}   |   columna de pliego: {MT}')
t0 = time.time()

con.execute(f"""
CREATE OR REPLACE TABLE s AS
SELECT pm.ocid, pm.entidad_ruc, pm.proveedor_ruc, pm.cpc5_principal AS cpc,
       pm.anio_proceso AS anio, pm.monto_final, pm."{MT}" AS pliego,
       pm.fecha_inicio_tender AS f, pm.entidad_nombre, pm.proveedor_nombre,
       upper(coalesce(pm."{T}", '')) AS titulo, ix.familia
FROM pm JOIN ix ON ix.ocid = pm.ocid
""")
con.execute("CREATE OR REPLACE TABLE techo AS SELECT familia, anio, "
            "max(monto_final) AS t_adj, max(pliego) AS t_pli FROM s GROUP BY 1, 2")

print('\n' + '='*118)
print('(A) EL PLIEGO PEGADO AL TECHO: cuan cerca presupuesta cada entidad')
print('='*118)
print(con.execute("""
SELECT s.familia, count(*) AS procesos,
       round(100.0*avg(CASE WHEN s.pliego >= 0.95*t.t_pli THEN 1.0 ELSE 0.0 END), 2) AS pct_pliego_sobre_95_del_techo,
       round(100.0*avg(CASE WHEN s.pliego >= 0.99*t.t_pli THEN 1.0 ELSE 0.0 END), 2) AS pct_pliego_sobre_99,
       round(median(s.pliego/t.t_pli), 4) AS mediana_pliego_sobre_techo
FROM s JOIN techo t ON t.familia = s.familia AND t.anio = s.anio
WHERE s.pliego > 0 AND s.familia IN ('Menor cuantia','Cotizacion','Lista corta')
GROUP BY 1 ORDER BY pct_pliego_sobre_95_del_techo DESC
""").df().to_string(index=False))

# agrupacion por entidad + cpc, SIN exigir el mismo proveedor: la division en
# lotes puede repartirse entre varios contratistas y seguir siendo fraccionamiento
con.execute("""
CREATE OR REPLACE TABLE grupos AS
SELECT entidad_ruc, cpc, anio, familia, count(*) AS n,
       count(DISTINCT proveedor_ruc) AS n_proveedores,
       date_diff('day', min(f), max(f)) AS span,
       sum(coalesce(pliego, monto_final)) AS pliego_total,
       max(coalesce(pliego, monto_final)) AS pliego_max
FROM s WHERE cpc IS NOT NULL AND f IS NOT NULL AND familia IN ('Menor cuantia','Cotizacion','Lista corta')
GROUP BY 1,2,3,4
""")
con.execute("""
CREATE OR REPLACE TABLE flag AS
SELECT s.ocid, s.familia, s.anio, s.entidad_nombre, s.proveedor_nombre, s.cpc, s.titulo,
       s.monto_final, s.pliego, g.n, g.n_proveedores, g.span, g.pliego_total, t.t_pli AS techo,
       (g.n >= 2 AND g.span <= 30 AND g.pliego_max <= t.t_pli AND g.pliego_total > t.t_pli) AS marcado,
       regexp_matches(s.titulo, 'ZONA|LOTE|TRAMO| ETAPA|FASE |SECTOR|PRIMERA PARTE|SEGUNDA PARTE') AS dice_lote
FROM s JOIN grupos g ON g.entidad_ruc = s.entidad_ruc AND g.cpc = s.cpc
                    AND g.anio = s.anio AND g.familia = s.familia
       JOIN techo t ON t.familia = s.familia AND t.anio = s.anio
""")
print(f'  [{time.time()-t0:.1f}s] tablas listas')

print('\n' + '='*118)
print('(B) ¿CUANTOS CASOS MARCADOS SON LOTES DECLARADOS EN EL TITULO?')
print('='*118)
print(con.execute("""
SELECT familia, count(*) AS marcados,
       sum(CASE WHEN dice_lote THEN 1 ELSE 0 END) AS dicen_zona_lote_tramo,
       round(100.0*avg(CASE WHEN dice_lote THEN 1.0 ELSE 0.0 END), 2) AS pct_lote_declarado,
       round(100.0*avg(CASE WHEN n_proveedores = 1 THEN 1.0 ELSE 0.0 END), 2) AS pct_un_solo_proveedor,
       round(median(span), 0) AS span_mediano,
       round(median(pliego_total/techo), 2) AS veces_el_techo
FROM flag WHERE marcado GROUP BY 1 ORDER BY marcados DESC
""").df().to_string(index=False))
print('\n-- comparacion: la tasa de lote declarado entre los NO marcados --')
print(con.execute("""
SELECT marcado, count(*) AS procesos,
       round(100.0*avg(CASE WHEN dice_lote THEN 1.0 ELSE 0.0 END), 2) AS pct_lote_declarado
FROM flag GROUP BY 1
""").df().to_string(index=False))
print("""Si entre los marcados la mencion de zona/lote es mucho mas frecuente que entre los
no marcados, el indicador esta capturando sobre todo division en lotes y hay que decirlo
con ese numero. Lo que queda despues de descontarlos es el conjunto que interesa.""")

print('\n' + '='*118)
print('(C) LOS CASOS QUE NO SE DECLARAN COMO LOTES: los candidatos de verdad')
print('='*118)
print(con.execute("""
SELECT ocid, familia, anio, entidad_nombre[1:34] AS entidad,
       proveedor_nombre[1:28] AS proveedor, n AS procesos_del_grupo,
       n_proveedores AS provs, span AS dias, round(pliego, 0) AS pliego,
       round(pliego_total, 0) AS pliego_total, round(techo, 0) AS techo, titulo[1:46] AS titulo
FROM flag WHERE marcado AND NOT dice_lote AND n_proveedores = 1
ORDER BY pliego_total DESC LIMIT 15
""").df().to_string(index=False))
print(f'\n  [{time.time()-t0:.1f}s] listo')
print('\nPega aqui toda la salida.')

In [ ]:
# ===========================================================================
# CELDA 57 — encontrar el campo de descripcion real, y el indicador con
#            ventana movil de 30 dias sobre el pliego
# ===========================================================================
import time, duckdb, pandas as pd, pathlib
BASE = pathlib.Path(r'C:\SERCOP_FINAL'); PA = BASE/'plan_A'
con = duckdb.connect(); con.execute("PRAGMA memory_limit='4GB'"); con.execute("PRAGMA threads=4")
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 70)
pd.set_option('display.max_colwidth', 78)
for v, f in [('pm','planA_procesos_modelo'), ('ix','planA_indicadores'), ('un','planA_universo')]:
    con.execute(f"CREATE OR REPLACE VIEW {v} AS SELECT * FROM "
                f"read_parquet('{(PA/(f+'.parquet')).as_posix()}')")

print('='*118)
print('(A) DONDE ESTA EL TEXTO DESCRIPTIVO DEL PROCESO')
print('='*118)
# se busca en las dos tablas toda columna de texto cuyo contenido sea largo:
# una descripcion real tiene decenas de caracteres, un codigo tiene veinte
for tabla in ['pm', 'un']:
    d = con.execute(f"DESCRIBE SELECT * FROM {tabla}").df()
    txt = d[d.column_type.str.upper().str.startswith('VARCHAR')]['column_name'].tolist()
    filas = []
    for c in txt:
        r = con.execute(f'SELECT round(avg(length("{c}")), 1) AS largo_medio, '
                        f'max(length("{c}")) AS largo_max, count("{c}") AS no_nulos '
                        f'FROM {tabla}').df().iloc[0]
        if r.largo_medio and r.largo_medio > 28:
            filas.append(dict(tabla=tabla, columna=c, largo_medio=r.largo_medio,
                              largo_max=int(r.largo_max), no_nulos=int(r.no_nulos)))
    if filas:
        f2 = pd.DataFrame(filas).sort_values('largo_medio', ascending=False)
        print(f'\n-- {tabla}: columnas de texto largo --')
        print(f2.to_string(index=False))
        for c in f2['columna'].head(4):
            v = con.execute(f'SELECT "{c}" FROM {tabla} WHERE "{c}" IS NOT NULL '
                            f'AND length("{c}") > 40 LIMIT 2').df()
            print(f'\n  {c}:')
            for x in v[c]: print(f'    "{str(x)[:150]}"')

print('\n' + '='*118)
print('(B) EL INDICADOR CON VENTANA MOVIL DE 30 DIAS SOBRE EL PLIEGO')
print('='*118)
t0 = time.time()
con.execute("""
CREATE OR REPLACE TABLE s AS
SELECT pm.ocid, pm.entidad_ruc, pm.proveedor_ruc, pm.cpc5_principal AS cpc,
       pm.anio_proceso AS anio, pm.fecha_inicio_tender AS f,
       coalesce(pm.monto_tender, pm.monto_final) AS pliego, pm.monto_final,
       pm.entidad_nombre, pm.proveedor_nombre, ix.familia
FROM pm JOIN ix ON ix.ocid = pm.ocid
WHERE pm.cpc5_principal IS NOT NULL AND pm.fecha_inicio_tender IS NOT NULL
""")
con.execute("CREATE OR REPLACE TABLE techo AS "
            "SELECT familia, anio, max(pliego) AS t FROM s GROUP BY 1, 2")
con.execute("""
CREATE OR REPLACE TABLE vent AS
SELECT s.*,
       count(*)                    OVER v AS n_prev,
       coalesce(sum(s.pliego)      OVER v, 0) AS pliego_prev,
       coalesce(max(s.pliego)      OVER v, 0) AS pliego_max_prev,
       count(DISTINCT s.proveedor_ruc) OVER v AS provs_prev,
       coalesce(min(date_diff('day', s.f, s.f)) OVER v, 0) AS cero
FROM s
WINDOW v AS (PARTITION BY s.entidad_ruc, s.cpc ORDER BY s.f
             RANGE BETWEEN INTERVAL 30 DAY PRECEDING AND INTERVAL 1 DAY PRECEDING)
""")
con.execute("""
CREATE OR REPLACE TABLE flag AS
SELECT v.*, t.t AS techo,
       (v.pliego + v.pliego_prev) AS suma_ventana,
       (v.n_prev >= 1
        AND v.familia IN ('Menor cuantia','Cotizacion','Lista corta')
        AND v.pliego <= t.t AND v.pliego_max_prev <= t.t
        AND (v.pliego + v.pliego_prev) > t.t) AS marcado
FROM vent v JOIN techo t ON t.familia = v.familia AND t.anio = v.anio
""")
print(f'  [{time.time()-t0:.1f}s] construido')
print(con.execute("""
SELECT familia, count(*) AS procesos,
       sum(CASE WHEN marcado THEN 1 ELSE 0 END) AS marcados,
       round(100.0*avg(CASE WHEN marcado THEN 1.0 ELSE 0.0 END), 2) AS pct,
       round(median(CASE WHEN marcado THEN suma_ventana/techo END), 2) AS veces_el_techo,
       round(100.0*avg(CASE WHEN marcado AND provs_prev = 1
             AND proveedor_ruc IS NOT NULL THEN 1.0 ELSE 0.0 END)
             / nullif(avg(CASE WHEN marcado THEN 1.0 ELSE 0.0 END), 0), 1) AS pct_mismo_proveedor
FROM flag WHERE familia IN ('Menor cuantia','Cotizacion','Lista corta')
GROUP BY 1 ORDER BY marcados DESC
""").df().to_string(index=False))

print('\n-- control: los dos casos que verificaste en el portal tienen que aparecer --')
print(con.execute("""
SELECT ocid, familia, anio, entidad_nombre[1:32] AS entidad, round(pliego,0) AS pliego,
       n_prev, round(pliego_prev,0) AS pliego_previo, round(suma_ventana,0) AS suma,
       round(techo,0) AS techo, marcado
FROM flag
WHERE ocid LIKE '%IMSE-021-2021%' OR ocid LIKE '%IMSE-018-2021%' OR ocid LIKE '%IMSE-031-2021%'
   OR ocid LIKE '%GADMCU-00_-2018%' OR ocid LIKE '%EERSSA-01_-2017%'
ORDER BY entidad_nombre, ocid
""").df().to_string(index=False))

print('\n' + '='*118)
print('(C) LOS 15 CANDIDATOS MAS GRANDES CON ESTA DEFINICION')
print('='*118)
print(con.execute("""
SELECT ocid, familia, anio, entidad_nombre[1:32] AS entidad,
       proveedor_nombre[1:26] AS proveedor, n_prev AS previos, provs_prev AS provs,
       round(pliego,0) AS pliego, round(suma_ventana,0) AS suma_ventana, round(techo,0) AS techo
FROM flag WHERE marcado ORDER BY suma_ventana DESC LIMIT 15
""").df().to_string(index=False))
print(f'\n  [{time.time()-t0:.1f}s] listo')
print('\nPega aqui toda la salida.')

In [ ]:
# ===========================================================================
# CELDA 58 — el indicador corregido: ventana de +-30 dias, el racimo acotado,
#            y los tres casos verificados como control de aceptacion
# ===========================================================================
import time, duckdb, pandas as pd, pathlib
BASE = pathlib.Path(r'C:\SERCOP_FINAL'); PA = BASE/'plan_A'
con = duckdb.connect(); con.execute("PRAGMA memory_limit='4GB'"); con.execute("PRAGMA threads=4")
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 70)
for v, f in [('pm','planA_procesos_modelo'), ('ix','planA_indicadores')]:
    con.execute(f"CREATE OR REPLACE VIEW {v} AS SELECT * FROM "
                f"read_parquet('{(PA/(f+'.parquet')).as_posix()}')")
t0 = time.time()

con.execute("""
CREATE OR REPLACE TABLE s AS
SELECT pm.ocid, pm.entidad_ruc, pm.proveedor_ruc, pm.cpc5_principal AS cpc,
       pm.anio_proceso AS anio, pm.fecha_inicio_tender AS f,
       coalesce(pm.monto_tender, pm.monto_final) AS pliego,
       pm.entidad_nombre, pm.proveedor_nombre, ix.familia
FROM pm JOIN ix ON ix.ocid = pm.ocid
WHERE pm.cpc5_principal IS NOT NULL AND pm.fecha_inicio_tender IS NOT NULL
      AND coalesce(pm.monto_tender, pm.monto_final) > 0
""")
con.execute("CREATE OR REPLACE TABLE techo AS "
            "SELECT familia, anio, max(pliego) AS t FROM s GROUP BY 1, 2")

# ventana descriptiva de +-30 dias, excluyendo el propio registro (se resta a mano).
# dos particiones: el racimo del mismo proveedor y el racimo de la entidad.
con.execute("""
CREATE OR REPLACE TABLE vent AS
SELECT s.*,
       count(*)      OVER wp AS np_prov,  sum(s.pliego) OVER wp AS sp_prov,
       max(s.pliego) OVER wp AS mx_prov,
       count(*)      OVER we AS np_ent,
       count(DISTINCT s.proveedor_ruc) OVER we AS provs_ent
FROM s
WINDOW wp AS (PARTITION BY s.entidad_ruc, s.cpc, s.proveedor_ruc ORDER BY s.f
              RANGE BETWEEN INTERVAL 30 DAY PRECEDING AND INTERVAL 30 DAY FOLLOWING),
       we AS (PARTITION BY s.entidad_ruc, s.cpc ORDER BY s.f
              RANGE BETWEEN INTERVAL 30 DAY PRECEDING AND INTERVAL 30 DAY FOLLOWING)
""")
con.execute("""
CREATE OR REPLACE TABLE flag AS
SELECT v.ocid, v.familia, v.anio, v.entidad_ruc, v.entidad_nombre, v.proveedor_nombre,
       v.cpc, v.pliego, t.t AS techo,
       v.np_prov AS racimo_proveedor,          -- incluye el propio proceso
       v.sp_prov AS suma_racimo_proveedor,
       v.mx_prov AS mayor_del_racimo,
       v.np_ent  AS racimo_entidad,
       v.provs_ent AS proveedores_del_racimo,
       (v.familia IN ('Menor cuantia','Cotizacion','Lista corta')
        AND v.np_prov BETWEEN 2 AND 4          -- una necesidad partida da pocos pedazos
        AND v.np_ent <= 6                      -- y la entidad no esta corriendo un programa
        AND v.mx_prov <= t.t                   -- cada pedazo por debajo del techo
        AND v.sp_prov > t.t)       AS marcado,
       (v.np_ent >= 10)            AS programa_de_obra
FROM vent v JOIN techo t ON t.familia = v.familia AND t.anio = v.anio
""")
print(f'  [{time.time()-t0:.1f}s] construido')

print('\n' + '='*118)
print('(A) CONTROL DE ACEPTACION: los tres casos verificados en el portal')
print('='*118)
print(con.execute("""
SELECT ocid, familia, anio, entidad_nombre[1:30] AS entidad,
       proveedor_nombre[1:24] AS proveedor, round(pliego,0) AS pliego,
       racimo_proveedor AS racimo, round(suma_racimo_proveedor,0) AS suma,
       round(techo,0) AS techo, racimo_entidad AS racimo_ent,
       proveedores_del_racimo AS provs, marcado
FROM flag
WHERE (ocid LIKE '%COTO-IMSE-0%-2021%' OR ocid LIKE '%COTO-GADMCU-00%-2018%'
    OR ocid LIKE '%COTO-EERSSA-01%-2017%')
ORDER BY entidad_nombre, ocid
""").df().to_string(index=False))
print('Los tres tienen que salir marcado = True. Si alguno sale False, la definicion')
print('todavia no sirve y no importa que los agregados se vean razonables.')

print('\n' + '='*118)
print('(B) CUANTO MARCA, Y COMO SE REPARTE EL UNIVERSO')
print('='*118)
print(con.execute("""
SELECT familia, count(*) AS procesos,
       sum(CASE WHEN marcado THEN 1 ELSE 0 END) AS marcados,
       round(100.0*avg(CASE WHEN marcado THEN 1.0 ELSE 0.0 END), 2) AS pct_marcado,
       sum(CASE WHEN programa_de_obra THEN 1 ELSE 0 END) AS en_programa_de_obra,
       round(100.0*avg(CASE WHEN programa_de_obra THEN 1.0 ELSE 0.0 END), 2) AS pct_programa
FROM flag WHERE familia IN ('Menor cuantia','Cotizacion','Lista corta')
GROUP BY 1 ORDER BY marcados DESC
""").df().to_string(index=False))

print('\n-- los marcados, por tamano del racimo y numero de proveedores --')
print(con.execute("""
SELECT racimo_proveedor, proveedores_del_racimo, count(*) AS marcados,
       round(median(suma_racimo_proveedor/techo), 2) AS veces_el_techo,
       round(median(pliego/techo), 3) AS pliego_sobre_techo
FROM flag WHERE marcado GROUP BY 1, 2 ORDER BY marcados DESC LIMIT 12
""").df().to_string(index=False))

print('\n-- por entidad: quienes acumulan mas casos --')
print(con.execute("""
SELECT entidad_nombre[1:48] AS entidad, count(*) AS casos,
       count(DISTINCT cpc) AS cpc_distintos, count(DISTINCT anio) AS anios,
       round(sum(pliego)/1e6, 2) AS millones_marcados
FROM flag WHERE marcado GROUP BY 1 ORDER BY casos DESC LIMIT 15
""").df().to_string(index=False))

print('\n' + '='*118)
print('(C) LOS 15 CANDIDATOS MAS GRANDES')
print('='*118)
print(con.execute("""
SELECT ocid, familia, anio, entidad_nombre[1:30] AS entidad,
       proveedor_nombre[1:24] AS proveedor, cpc, racimo_proveedor AS racimo,
       round(pliego,0) AS pliego, round(suma_racimo_proveedor,0) AS suma,
       round(techo,0) AS techo
FROM flag WHERE marcado ORDER BY suma_racimo_proveedor DESC LIMIT 15
""").df().to_string(index=False))
print(f'\n  [{time.time()-t0:.1f}s] listo')
print('\nPega aqui toda la salida.')

In [ ]:
# ===========================================================================
# CELDA 59 — control corregido con concentracion del racimo, y la prueba de
#            amontonamiento del pliego contra el techo de la familia
# ===========================================================================
import time, duckdb, pandas as pd, numpy as np, pathlib
BASE = pathlib.Path(r'C:\SERCOP_FINAL'); PA = BASE/'plan_A'
con = duckdb.connect(); con.execute("PRAGMA memory_limit='4GB'"); con.execute("PRAGMA threads=4")
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 70)
for v, f in [('pm','planA_procesos_modelo'), ('ix','planA_indicadores')]:
    con.execute(f"CREATE OR REPLACE VIEW {v} AS SELECT * FROM "
                f"read_parquet('{(PA/(f+'.parquet')).as_posix()}')")
FAM = "('Menor cuantia','Cotizacion','Lista corta')"
t0 = time.time()

con.execute(f"""
CREATE OR REPLACE TABLE s AS
SELECT pm.ocid, pm.entidad_ruc, pm.proveedor_ruc, pm.cpc5_principal AS cpc,
       pm.anio_proceso AS anio, pm.fecha_inicio_tender AS f,
       coalesce(pm.monto_tender, pm.monto_final) AS pliego, pm.monto_final,
       pm.entidad_nombre, pm.proveedor_nombre, ix.familia
FROM pm JOIN ix ON ix.ocid = pm.ocid
WHERE pm.fecha_inicio_tender IS NOT NULL AND coalesce(pm.monto_tender, pm.monto_final) > 0
""")
con.execute("CREATE OR REPLACE TABLE techo AS "
            "SELECT familia, anio, max(pliego) AS t FROM s GROUP BY 1, 2")
con.execute("""
CREATE OR REPLACE TABLE vent AS
SELECT s.*, count(*) OVER wp AS np_prov, sum(s.pliego) OVER wp AS sp_prov,
       max(s.pliego) OVER wp AS mx_prov, count(*) OVER we AS np_ent,
       count(DISTINCT s.proveedor_ruc) OVER we AS provs_ent
FROM s WHERE s.cpc IS NOT NULL
WINDOW wp AS (PARTITION BY s.entidad_ruc, s.cpc, s.proveedor_ruc ORDER BY s.f
              RANGE BETWEEN INTERVAL 30 DAY PRECEDING AND INTERVAL 30 DAY FOLLOWING),
       we AS (PARTITION BY s.entidad_ruc, s.cpc ORDER BY s.f
              RANGE BETWEEN INTERVAL 30 DAY PRECEDING AND INTERVAL 30 DAY FOLLOWING)
""")
con.execute(f"""
CREATE OR REPLACE TABLE flag AS
SELECT v.ocid, v.familia, v.anio, v.entidad_ruc, v.entidad_nombre, v.proveedor_nombre, v.cpc,
       v.pliego, t.t AS techo, round(v.pliego/t.t, 4) AS razon_pliego_techo,
       v.np_prov AS racimo_prov, v.sp_prov AS suma_racimo, v.np_ent AS racimo_ent,
       v.provs_ent, round(v.np_prov::DOUBLE/v.np_ent, 3) AS concentracion,
       (v.familia IN {FAM} AND v.np_prov BETWEEN 2 AND 4
        AND v.mx_prov <= t.t AND v.sp_prov > t.t) AS marcado,
       CASE WHEN v.np_prov::DOUBLE/v.np_ent >= 0.99 THEN '1 racimo aislado'
            WHEN v.np_prov::DOUBLE/v.np_ent >= 0.40 THEN '2 racimo concentrado'
            ELSE '3 dentro de un programa amplio' END AS tipo_racimo
FROM vent v JOIN techo t ON t.familia = v.familia AND t.anio = v.anio
""")
print(f'  [{time.time()-t0:.1f}s] construido')

print('\n' + '='*118)
print('(A) CONTROL DE ACEPTACION, AHORA SIN LA GUARDIA QUE SOBRABA')
print('='*118)
print(con.execute("""
SELECT ocid, anio, entidad_nombre[1:28] AS entidad, proveedor_nombre[1:22] AS proveedor,
       round(pliego,0) AS pliego, razon_pliego_techo AS razon, racimo_prov AS racimo,
       round(suma_racimo,0) AS suma, racimo_ent, concentracion, tipo_racimo, marcado
FROM flag
WHERE ocid IN ('ocds-5wno2w-COTO-IMSE-018-2021-4081','ocds-5wno2w-COTO-IMSE-021-2021-4081',
               'ocds-5wno2w-COTO-IMSE-031-2021-4081','ocds-5wno2w-COTO-GADMCU-004-2018-72903',
               'ocds-5wno2w-COTO-GADMCU-005-2018-72903','ocds-5wno2w-COTO-GADMCU-006-2018-72903',
               'ocds-5wno2w-COTO-EERSSA-011-2017-3240','ocds-5wno2w-COTO-EERSSA-012-2017-3240')
ORDER BY entidad, ocid
""").df().to_string(index=False))
print('\nLos ocho tienen que salir marcado = True.')

print('\n' + '='*118)
print('(B) LA PRUEBA DE AMONTONAMIENTO: densidad del pliego contra el techo')
print('='*118)
for fam in ['Cotizacion', 'Menor cuantia', 'Lista corta']:
    h = con.execute(f"""
    SELECT least(floor(razon_pliego_techo*50)/50, 0.98) AS bin, count(*) AS n
    FROM flag WHERE familia = '{fam}' AND razon_pliego_techo BETWEEN 0 AND 1
    GROUP BY 1 ORDER BY 1
    """).df()
    # contrafactual: polinomio de grado 5 ajustado FUERA de la zona del umbral
    fuera = h[(h.bin < 0.80)]
    co = np.polyfit(fuera.bin, fuera.n, 5)
    obs = con.execute(f"""
    SELECT sum(CASE WHEN razon_pliego_techo >= 0.90 THEN 1 ELSE 0 END) AS en_0_90_1_00,
           sum(CASE WHEN razon_pliego_techo >= 0.95 THEN 1 ELSE 0 END) AS en_0_95_1_00,
           sum(CASE WHEN razon_pliego_techo >= 0.99 THEN 1 ELSE 0 END) AS en_0_99_1_00,
           count(*) AS total
    FROM flag WHERE familia = '{fam}' AND razon_pliego_techo BETWEEN 0 AND 1
    """).df().iloc[0]
    bins90 = np.arange(0.90, 1.00, 0.02)
    esp90 = max(np.polyval(co, bins90).sum(), 1)
    print(f'\n{fam}:  {int(obs.total):,} procesos con pliego y techo')
    print(f'  masa observada en 0,90-1,00 del techo .... {int(obs.en_0_90_1_00):>6,}')
    print(f'  masa esperada por la curva suave ......... {esp90:>9.0f}')
    print(f'  exceso ................................... {obs.en_0_90_1_00/esp90:>9.2f} veces')
    print(f'  en 0,95-1,00: {int(obs.en_0_95_1_00):,}   |   en 0,99-1,00: {int(obs.en_0_99_1_00):,}')
    print('  histograma de los ultimos tramos (cada 2% del techo):')
    print(con.execute(f"""
    SELECT round(floor(razon_pliego_techo*50)/50, 2) AS desde_pct_del_techo, count(*) AS procesos
    FROM flag WHERE familia = '{fam}' AND razon_pliego_techo >= 0.80
    GROUP BY 1 ORDER BY 1
    """).df().to_string(index=False))

print('\n' + '='*118)
print('(C) QUIEN PRESUPUESTA PEGADO AL TECHO')
print('='*118)
print(con.execute(f"""
SELECT entidad_nombre[1:46] AS entidad, familia, count(*) AS procesos,
       sum(CASE WHEN razon_pliego_techo >= 0.99 THEN 1 ELSE 0 END) AS sobre_99_del_techo,
       round(100.0*avg(CASE WHEN razon_pliego_techo >= 0.99 THEN 1.0 ELSE 0.0 END), 1) AS pct,
       round(median(razon_pliego_techo), 3) AS razon_mediana
FROM flag WHERE familia IN {FAM}
GROUP BY 1, 2 HAVING count(*) >= 20 AND sum(CASE WHEN razon_pliego_techo >= 0.99 THEN 1 ELSE 0 END) >= 3
ORDER BY pct DESC, sobre_99_del_techo DESC LIMIT 18
""").df().to_string(index=False))

print('\n' + '='*118)
print('(D) CUANTO MARCA EL RACIMO, POR TIPO')
print('='*118)
print(con.execute(f"""
SELECT familia, tipo_racimo, count(*) AS procesos,
       sum(CASE WHEN marcado THEN 1 ELSE 0 END) AS marcados,
       round(100.0*avg(CASE WHEN marcado THEN 1.0 ELSE 0.0 END), 2) AS pct
FROM flag WHERE familia IN {FAM} GROUP BY 1, 2 ORDER BY familia, tipo_racimo
""").df().to_string(index=False))
print(f'\n  [{time.time()-t0:.1f}s] listo')
print('\nPega aqui toda la salida.')

In [ ]:
# ===========================================================================
# CELDA 60 — amontonamiento con estimador local, sensibilidad y placebo.
#            Requiere la tabla flag de la celda 59 (ya esta en memoria).
# ===========================================================================
import pandas as pd, numpy as np
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 70)
FAMS = ['Cotizacion', 'Menor cuantia', 'Lista corta']

def hist(fam, w=0.02, lo=0.0, hi=1.0):
    return con.execute(f"""
    SELECT floor(razon_pliego_techo/{w})*{w} AS bin, count(*) AS n
    FROM flag WHERE familia = '{fam}'
      AND razon_pliego_techo >= {lo} AND razon_pliego_techo < {hi}
    GROUP BY 1 ORDER BY 1
    """).df()

print('='*118)
print('(A) ESTIMADOR LOCAL: el tramo pegado al techo contra sus vecinos')
print('='*118)
res = []
for fam in FAMS:
    h = hist(fam)
    for corte in [0.70, 0.80, 0.85]:
        vec = h[(h.bin >= corte) & (h.bin < 0.98)]['n']
        ult = int(h[(h.bin >= 0.98)]['n'].sum())
        # dos referencias: el promedio plano, y una recta ajustada a los vecinos
        plano = vec.mean()
        bx = h[(h.bin >= corte) & (h.bin < 0.98)]['bin']
        recta = np.polyval(np.polyfit(bx, vec, 1), 0.99)
        res.append(dict(familia=fam, vecinos_desde=corte, n_vecinos=len(vec),
                        ultimo_tramo=ult, ref_promedio=round(plano, 1),
                        exceso_vs_promedio=round(ult/plano, 2),
                        ref_recta=round(recta, 1), exceso_vs_recta=round(ult/max(recta, 1), 2),
                        procesos_que_sobran=int(round(ult - plano))))
print(pd.DataFrame(res).to_string(index=False))
print("""La recta ajustada a los vecinos es un contrafactual local: no extrapola lejos, solo
continua la tendencia inmediata. Si las dos referencias dan excesos parecidos y el resultado
no cambia al mover 'vecinos_desde', el amontonamiento es robusto.""")

print('\n' + '='*118)
print('(B) PLACEBO: el mismo calculo en umbrales falsos, donde no hay norma')
print('='*118)
pl = []
for fam in FAMS:
    for centro in [0.30, 0.40, 0.50, 0.60, 0.70, 0.98]:
        h = hist(fam, w=0.02)
        ult = int(h[(h.bin >= centro) & (h.bin < centro + 0.02)]['n'].sum())
        vec = h[(h.bin >= centro - 0.18) & (h.bin < centro)]['n']
        if len(vec) < 5 or vec.mean() == 0: continue
        pl.append(dict(familia=fam, umbral=('REAL' if centro == 0.98 else 'falso'),
                       posicion=centro, tramo=ult, vecinos=round(vec.mean(), 1),
                       exceso=round(ult/vec.mean(), 2)))
print(pd.DataFrame(pl).to_string(index=False))
print("""Solo el umbral REAL deberia mostrar exceso. Si los falsos tambien lo muestran, el
salto es una propiedad de la forma de la distribucion y no una respuesta al techo legal.""")

print('\n' + '='*118)
print('(C) ZOOM: el ultimo 4% antes del techo, en tramos de 0,2%')
print('='*118)
for fam in FAMS:
    h = hist(fam, w=0.002, lo=0.96, hi=1.0001)
    h['pct_del_techo'] = (100*h.bin).round(1)
    print(f'\n{fam}:')
    print(h[['pct_del_techo', 'n']].to_string(index=False))

print('\n' + '='*118)
print('(D) ESTABILIDAD POR ANIO (cotizacion y menor cuantia)')
print('='*118)
print(con.execute("""
SELECT familia, anio, count(*) AS procesos,
       sum(CASE WHEN razon_pliego_techo >= 0.98 THEN 1 ELSE 0 END) AS pegados_al_techo,
       round(100.0*avg(CASE WHEN razon_pliego_techo >= 0.98 THEN 1.0 ELSE 0.0 END), 2) AS pct
FROM flag WHERE familia IN ('Cotizacion','Menor cuantia') AND anio BETWEEN 2015 AND 2024
GROUP BY 1, 2 ORDER BY familia, anio
""").df().to_string(index=False))
print('\nPega aqui toda la salida.')

In [ ]:
# ===========================================================================
# CELDA 61 — validacion de los techos contra los coeficientes legales y el
#            PIE, y la figura del amontonamiento
# ===========================================================================
import pandas as pd, numpy as np, pathlib
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator

GRAF = pathlib.Path(r'C:\SERCOP_FINAL\plan_A\graficos'); GRAF.mkdir(exist_ok=True)
AZUL, NARANJA, GRIS, TINTA = '#2a78d6', '#eb6834', '#8a8781', '#0b0b0b'

print('='*118)
print('(A) LOS TECHOS CONTRA LOS COEFICIENTES LEGALES Y EL PRESUPUESTO DEL ESTADO')
print('='*118)
t = con.execute("""
SELECT anio,
       max(CASE WHEN familia = 'Menor cuantia' THEN t END) AS menor_cuantia,
       max(CASE WHEN familia = 'Cotizacion'    THEN t END) AS cotizacion,
       max(CASE WHEN familia = 'Lista corta'   THEN t END) AS lista_corta
FROM techo WHERE anio BETWEEN 2015 AND 2024 GROUP BY 1 ORDER BY 1
""").df()
t['coti_sobre_menor'] = (t.cotizacion / t.menor_cuantia).round(4)
t['lista_sobre_menor'] = (t.lista_corta / t.menor_cuantia).round(4)
t['PIE_implicito_millones'] = (t.menor_cuantia / 0.000007 / 1e6).round(0)
print(t.to_string(index=False))
print(f"""
La razon cotizacion/menor cuantia deberia ser 30/7 = {30/7:.4f} si los dos techos son
coeficientes (0,00003 y 0,000007) del mismo Presupuesto Inicial del Estado.
observada: media {t.coti_sobre_menor.mean():.4f}, minimo {t.coti_sobre_menor.min():.4f},
maximo {t.coti_sobre_menor.max():.4f}, desviacion {t.coti_sobre_menor.std():.5f}.
PIE implicito = techo de menor cuantia / 0,000007. Confirmar contra el Ministerio de Finanzas.""")

print('\n' + '='*118)
print('(B) LA FIGURA')
print('='*118)
FAMS = [('Cotizacion', 'Cotización'), ('Menor cuantia', 'Menor cuantía'),
        ('Lista corta', 'Lista corta')]
fig, axes = plt.subplots(3, 1, figsize=(9.2, 9.6), sharex=True)
for ax, (fam, etiq) in zip(axes, FAMS):
    h = con.execute(f"""
    SELECT floor(razon_pliego_techo/0.002)*0.002 AS b, count(*) AS n
    FROM flag WHERE familia = '{fam}' AND razon_pliego_techo >= 0.90
      AND razon_pliego_techo <= 1.0001 GROUP BY 1 ORDER BY 1
    """).df()
    h['x'] = 100*h.b
    base = h[(h.x >= 90) & (h.x < 99)]['n'].mean()
    pico = h[h.x >= 99]
    ax.bar(h.x[h.x < 99], h.n[h.x < 99], width=0.17, color=AZUL, linewidth=0)
    ax.bar(pico.x, pico.n, width=0.17, color=NARANJA, linewidth=0)
    ax.axhline(base, color=GRIS, lw=1.0, ls=(0, (4, 3)))
    ax.text(90.15, base, f' densidad típica fuera del umbral: {base:.0f}',
            va='bottom', ha='left', fontsize=8.5, color=GRIS)
    exceso = pico.n.sum() - base*len(pico)
    ax.set_title(f'{etiq} — el último 1% concentra {pico.n.sum():,.0f} procesos '
                 f'frente a los {base*len(pico):,.0f} que correspondían '
                 f'({pico.n.sum()/(base*len(pico)):.1f} veces; sobran {exceso:,.0f})',
                 fontsize=10.5, loc='left', color=TINTA, pad=7)
    ax.set_ylabel('procesos', fontsize=9)
    ax.grid(axis='y', lw=0.4, color='#dddbd6'); ax.set_axisbelow(True)
    for s in ('top', 'right'): ax.spines[s].set_visible(False)
    ax.spines['left'].set_color('#c9c7c0'); ax.spines['bottom'].set_color('#c9c7c0')
    ax.tick_params(labelsize=9, color='#c9c7c0')
axes[-1].set_xlabel('presupuesto referencial del pliego, como porcentaje del techo legal '
                    'de su familia y año', fontsize=9.5)
axes[-1].xaxis.set_major_locator(MultipleLocator(1))
fig.suptitle('Los presupuestos referenciales se amontonan justo por debajo del techo legal',
             fontsize=13, ha='left', x=0.062, y=0.985, color=TINTA, weight='bold')
fig.text(0.062, 0.955, 'Tramos de 0,2% del techo. En naranja el último 1%. '
         'El techo de cada familia y año se estima como el máximo observado y se valida '
         'contra los coeficientes de la LOSNCP.', fontsize=9, color=GRIS, ha='left')
fig.tight_layout(rect=[0, 0, 1, 0.945])
out = GRAF / 'F13_amontonamiento_pliego_techo.png'
fig.savefig(out, dpi=170, facecolor='white'); plt.close(fig)
print(f'  escrita: {out}')

desc = ('F13 | Amontonamiento del presupuesto referencial contra el techo legal | '
        'Histograma en tramos de 0,2% del techo de la familia y anio, para cotizacion, '
        'menor cuantia y lista corta. Muestra densidad plana a lo largo del recorrido y un '
        'pico de entre 5,6 y 6,1 veces la densidad tipica en el ultimo 0,2% antes del techo. '
        'Pruebas de respaldo: el exceso no cambia al mover la ventana de referencia (0,70 / '
        '0,80 / 0,85) y los umbrales falsos (30%, 40%, 50%, 60%, 70% del techo) no muestran '
        'exceso (0,39 a 1,22 veces). El techo se estima como el maximo observado por familia '
        'y anio; su razon entre familias es constante en 4,2857 = 30/7, la razon de los '
        'coeficientes 0,00003 y 0,000007 del PIE que fija la LOSNCP. Limitacion: la infima '
        'cuantia no se publica como proceso, asi que el fraccionamiento por debajo de menor '
        'cuantia es invisible en esta fuente.')
with open(GRAF/'CATALOGO_DE_GRAFICOS.md', 'a', encoding='utf-8') as f:
    f.write('\n\n## ' + desc.replace(' | ', '\n\n**', 1).replace(' | ', '**\n\n', 1))
print('  catalogo actualizado')
print('\nPega aqui toda la salida y pasame la figura cuando la veas.')

In [ ]:
# ===========================================================================
# CELDA 61b — figura corregida: subtitulo en dos lineas, etiqueta sin choque,
#             aire sobre las barras, y el titular en el ultimo 1%
# ===========================================================================
import pandas as pd, numpy as np, pathlib
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator

GRAF = pathlib.Path(r'C:\SERCOP_FINAL\plan_A\graficos')
AZUL, NARANJA, GRIS, TINTA, REJILLA = '#2a78d6', '#eb6834', '#8a8781', '#0b0b0b', '#dddbd6'
FAMS = [('Cotizacion', 'Cotización'), ('Menor cuantia', 'Menor cuantía'),
        ('Lista corta', 'Lista corta')]

fig, axes = plt.subplots(3, 1, figsize=(9.6, 10.2), sharex=True)
resumen = []
for ax, (fam, etiq) in zip(axes, FAMS):
    h = con.execute(f"""
    SELECT least(floor(razon_pliego_techo/0.002)*0.002, 0.998) AS b, count(*) AS n
    FROM flag WHERE familia = '{fam}' AND razon_pliego_techo >= 0.90
      AND razon_pliego_techo <= 1.0001 GROUP BY 1 ORDER BY 1
    """).df()
    h['x'] = 100*h.b
    base  = h[(h.x >= 90) & (h.x < 99)]['n'].mean()
    pico  = h[h.x >= 99]
    obs, esp = pico.n.sum(), base*len(pico)
    resumen.append(dict(familia=etiq, en_el_ultimo_1pct=int(obs), esperados=round(esp),
                        veces=round(obs/esp, 1), sobran=int(round(obs-esp)),
                        tramo_mas_alto=int(h.n.max())))

    ax.bar(h.x[h.x < 99], h.n[h.x < 99], width=0.17, color=AZUL, linewidth=0)
    ax.bar(pico.x, pico.n, width=0.17, color=NARANJA, linewidth=0)
    ax.axhline(base, color=GRIS, lw=1.0, ls=(0, (4, 3)))
    ax.axvline(98.9, color='#c9c7c0', lw=0.8)
    # la etiqueta va arriba a la izquierda, donde no hay barras
    ax.text(0.012, 0.93, f'densidad típica fuera del umbral: {base:.0f} procesos por tramo',
            transform=ax.transAxes, fontsize=8.5, color=GRIS, va='top', ha='left')
    ax.set_title(f'{etiq} — en el último 1% hay {obs:,.0f} procesos y correspondían '
                 f'{esp:,.0f}: {obs/esp:.1f} veces, sobran {obs-esp:,.0f}',
                 fontsize=10.5, loc='left', color=TINTA, pad=7)
    ax.set_ylim(0, h.n.max()*1.20)
    ax.set_ylabel('procesos', fontsize=9)
    ax.grid(axis='y', lw=0.4, color=REJILLA); ax.set_axisbelow(True)
    for s in ('top', 'right'): ax.spines[s].set_visible(False)
    ax.spines['left'].set_color('#c9c7c0'); ax.spines['bottom'].set_color('#c9c7c0')
    ax.tick_params(labelsize=9, color='#c9c7c0')

axes[-1].set_xlabel('presupuesto referencial del pliego, como porcentaje del techo legal '
                    'de su familia y año', fontsize=9.5)
axes[-1].xaxis.set_major_locator(MultipleLocator(1))
fig.suptitle('Los presupuestos referenciales se amontonan justo por debajo del techo legal',
             fontsize=13.5, ha='left', x=0.058, y=0.988, color=TINTA, weight='bold')
fig.text(0.058, 0.962, 'Tramos de 0,2% del techo; en naranja el último 1%. El techo de cada '
         'familia y año se estima como el máximo observado.', fontsize=9, color=GRIS, ha='left')
fig.text(0.058, 0.944, 'La razón entre los techos de las tres familias es constante en 30/7 '
         'y 15/7: los coeficientes de la LOSNCP sobre el PIE.',
         fontsize=9, color=GRIS, ha='left')
fig.tight_layout(rect=[0, 0, 1, 0.932])
out = GRAF / 'F13_amontonamiento_pliego_techo.png'
fig.savefig(out, dpi=170, facecolor='white'); plt.close(fig)
print(f'escrita: {out}\n')
print(pd.DataFrame(resumen).to_string(index=False))
print('\nPasame la figura otra vez.')

In [ ]:
# ===========================================================================
# CELDA 61c — corregir la ficha de F13 en el catalogo de graficos
# ===========================================================================
import pathlib, shutil, datetime
GRAF = pathlib.Path(r'C:\SERCOP_FINAL\plan_A\graficos')
CAT  = GRAF / 'CATALOGO_DE_GRAFICOS.md'
shutil.copyfile(CAT, CAT.with_suffix('.md.bak'))          # respaldo antes de tocar

txt = CAT.read_text(encoding='utf-8')
i = txt.find('F13')
if i > 0:
    j = txt.rfind('\n\n##', 0, i)                          # inicio de la ficha vieja
    txt = txt[:j if j > 0 else i].rstrip()
    print('ficha anterior de F13 eliminada')

txt += """

## F13 — Amontonamiento del presupuesto referencial contra el techo legal

**Qué muestra.** Histograma del presupuesto referencial del pliego expresado como porcentaje
del techo legal de su familia y año, en tramos de 0,2%, para cotización, menor cuantía y
lista corta. La densidad es plana a lo largo de todo el recorrido y salta en el último 1%:
176 procesos donde correspondían 52 en cotización (3,4 veces, sobran 124), 324 donde
correspondían 113 en menor cuantía (2,9 veces, sobran 211) y 60 donde correspondían 14 en
lista corta (4,2 veces, sobran 46). En total sobran 381 procesos pegados al umbral.

**Cómo se construye.** La referencia es la densidad media de los tramos entre el 90% y el 99%
del techo, que es conservadora porque incluye el hombro previo al umbral. El techo de cada
familia y año se estima como el máximo observado del presupuesto referencial.

**Pruebas de respaldo.** Sensibilidad: el exceso no cambia al mover la ventana de referencia
a 0,70, 0,80 o 0,85 del techo (cotización entre 1,99 y 2,42 veces medido sobre el 2% final;
menor cuantía entre 1,49 y 2,08; lista corta entre 2,17 y 2,89). Placebo: el mismo cálculo en
umbrales falsos situados al 30%, 40%, 50%, 60% y 70% del techo da excesos de 0,39 a 1,22
veces, es decir ninguno; sólo el umbral real produce salto, y lo produce en las tres familias.

**Validación de los techos.** La razón entre los techos de las tres familias es constante
durante diez años: cotización sobre menor cuantía da 4,2857 (media 4,2852, desviación
0,00122) y lista corta sobre menor cuantía da 2,1429. Son exactamente 30/7 y 15/7, las
razones entre los coeficientes 0,00003, 0,000015 y 0,000007 que la LOSNCP aplica sobre el
Presupuesto Inicial del Estado. Dividir el techo de menor cuantía por 0,000007 reproduce el
Presupuesto General del Estado de cada año: 36.305 millones en 2015, 29.835 en 2016, 29.834
en 2017. Los techos se estimaron a partir de la forma de la distribución, sin consultar
ninguna norma ni documento presupuestario.

**Casos verificados en el portal.** Municipio de Urdaneta, tres cotizaciones el mismo día al
mismo proveedor y mismo CPC por 2,46 millones contra un techo de 1,05 millones. EERSSA 2017,
dos cotizaciones el mismo día por 552.673 y 553.576 dólares, suma 1,11 millones contra un
techo de 895.053. Municipalidad de Santa Elena 2021, tres cotizaciones en 26 días al mismo
contratista individual por 2,47 millones, una de ellas con un pliego 134 dólares por debajo
del techo.

**Limitaciones declaradas.** La ínfima cuantía no se publica como proceso en el portal de
datos abiertos, de modo que el fraccionamiento por debajo de menor cuantía es invisible en
esta fuente. Y el indicador no distingue la división en lotes técnicamente justificada del
fraccionamiento: el caso verificado de EPMAPS corresponde a zonas geográficas de un mismo
proyecto, una práctica legítima. Lo que produce este análisis son casos candidatos para
revisión administrativa, no determinaciones de irregularidad.
"""
CAT.write_text(txt, encoding='utf-8')
print(f'catalogo reescrito: {CAT}  (respaldo en {CAT.with_suffix(".md.bak").name})')